# Curso de Machine Learning 2026
# Parte 2 — Modelos de predicción

Segundo de dos cuadernos. Este recorre las clases 5 a 8 del curso: la preparación de los datos
para el modelo, la división del dataset, la validación cruzada, cuatro algoritmos de clasificación
y la comparación entre ellos.

**Qué necesita para arrancar**

El archivo `dataset_limpio.csv` que produjo la **Parte 1**. No hace falta que lo tengas descargado:
por defecto el cuaderno lo lee directamente del repositorio de GitHub del proyecto, así que se abre
y se ejecuta sin bajar nada. El punto 0.2 explica las dos alternativas.

**Qué hace este cuaderno**

1. Carga el dataset limpio y verifica que sea lo que espera.
2. Lo transforma: codifica la etiqueta y prepara el escalado.
3. Explica y demuestra dos problemas que arruinan cualquier evaluación: la **fuga de información**
   y el **sobreajuste**.
4. Evalúa cuatro modelos con validación cruzada y el mismo bloque de métricas para todos.
5. Los compara y arma un ranking.

**Una decisión de diseño importante**

El cuaderno está armado para que agregar un modelo nuevo cueste tres líneas y que ese modelo
aparezca automáticamente en todas las tablas y gráficos comparativos del final. El punto 7 explica
cómo.

**Las librerías se importan por bloques**, justo antes de la sección que las usa, en lugar de todas
juntas al principio. Así se ve qué necesita cada etapa.

---

# Índice

### Parte 0 — Preliminares
- [0.1 Qué necesita este cuaderno](#sec-0-1)
- [0.2 Cargar el dataset limpio](#sec-0-2)
- [0.3 Verificación de entrada](#sec-0-3)
- [0.4 Configuración global](#sec-0-4)
- [0.5 Librerías de gráficos y estilo](#sec-0-5)

### 1. ¿Qué es Machine Learning? Dónde estamos parados
- [1.1 Aprendizaje supervisado y no supervisado](#sec-1-1)
- [1.2 Regresión y clasificación: dos preguntas distintas](#sec-1-2)
- [1.3 Diagnóstico de nuestro problema](#sec-1-3)
- [1.4 Los cuatro modelos que vamos a usar](#sec-1-4)

### 2. Transformación de datos
- [2.1 Feature engineering](#sec-2-1)
- [2.2 Codificación de la variable categórica](#sec-2-2)
- [2.3 Normalización y estandarización](#sec-2-3)
- [2.4 Las matrices X e y](#sec-2-4)

### 3. División del dataset: entrenamiento y prueba
- [3.1 Por qué hay que dividir](#sec-3-1)
- [3.2 La división 80/20 estratificada](#sec-3-2)
- [3.3 La fragilidad de una sola división](#sec-3-3)

### 4. Fuga de información (data leakage)
- [4.1 Qué es](#sec-4-1)
- [4.2 Por qué es tan grave: el síntoma es que no hay síntoma](#sec-4-2)
- [4.3 Las siete formas en que se cuela](#sec-4-3)
- [4.4 Demostración 1 — el escalador que mira de más](#sec-4-4)
- [4.5 Demostración 2 — "predecir" ruido puro muy por encima del azar](#sec-4-5)
- [4.6 El Pipeline como antídoto](#sec-4-6)
- [4.7 Lista de control](#sec-4-7)

### 5. Overfitting y underfitting
- [5.1 Qué es cada uno](#sec-5-1)
- [5.2 El compromiso sesgo-varianza](#sec-5-2)
- [5.3 Las señales: tabla de diagnóstico](#sec-5-3)
- [5.4 Las cuatro formas de detectarlo](#sec-5-4)
- [5.5 Cómo se mide: la brecha y sus umbrales](#sec-5-5)
- [5.6 Qué hacer con cada uno](#sec-5-6)
- [5.7 En código: tres niveles de complejidad](#sec-5-7)
- [5.8 La curva de validación](#sec-5-8)
- [5.9 La curva de aprendizaje](#sec-5-9)

### 6. Validación cruzada K-Fold
- [6.1 Cómo funciona](#sec-6-1)
- [6.2 Por qué acá es necesaria y no opcional](#sec-6-2)
- [6.3 El esquema estratificado](#sec-6-3)
- [6.4 Una división contra validación cruzada](#sec-6-4)

### 7. La arquitectura del cuaderno
- [7.1 Los escaladores y el esquema de validación](#sec-7-1)
- [7.2 El catálogo de modelos](#sec-7-2)
- [7.3 El cálculo de métricas](#sec-7-3)
- [7.4 El bloque de resultados](#sec-7-4)
- [7.5 Cómo agregar un modelo nuevo](#sec-7-5)

### Los modelos
- [8. Modelo 1 — Regresión Logística](#sec-8)
- [9. Modelo 2 — K-Nearest Neighbors](#sec-9)
- [10. Modelo 3 — Árbol de Decisión](#sec-10)
- [11. Modelo 4 — Random Forest](#sec-11)

### 12. Comparativa exhaustiva y ranking
- [12.1 Tabla maestra](#sec-12-1)
- [12.2 La mejor variante de cada modelo](#sec-12-2)
- [12.3 Curvas ROC superpuestas](#sec-12-3)
- [12.4 El efecto de la normalización](#sec-12-4)
- [12.5 Estabilidad entre pliegues](#sec-12-5)
- [12.6 Diagnóstico de ajuste comparado](#sec-12-6)
- [12.7 Ranking final](#sec-12-7)
- [12.8 Veredicto y limitaciones](#sec-12-8)

### Cierre
- [Resumen de lo hecho](#sec-cierre)

---
# Parte 0 — Preliminares

<a id="sec-0-1"></a>
## 0.1 Qué necesita este cuaderno

**La entrada:** el archivo `dataset_limpio.csv` que dejó la Parte 1. Tiene 171 filas y 8 columnas:
`TAG` con las clases `AFI` y `NEG`, y siete columnas numéricas.

**Qué se le hizo en la Parte 1** y qué no:

| Ya está hecho | Todavía no |
|---|---|
| Verificado que no hay valores faltantes | Codificar la etiqueta a 0 y 1 |
| Verificado que no hay duplicados ni contradicciones | Escalar las variables |
| Verificado que los tipos son correctos | Dividir en entrenamiento y prueba |
| Verificado que los rangos son posibles | Entrenar cualquier cosa |
| Detectados los outliers, y conservados a propósito | |

La columna de la derecha no es una tarea pendiente que la Parte 1 se haya olvidado: son decisiones
que **pertenecen al modelo**, no a la limpieza. Codificar y escalar dependen de qué modelo se vaya
a usar, y —como vamos a ver en el punto 4— tienen que aplicarse adentro del pipeline, después de
la división, o arruinan la evaluación.

**No hace falta que tengas el archivo a mano.** El punto 0.2 lo lee del repositorio de GitHub del
proyecto, y la única alternativa que ofrece es subirlo vos mismo si preferís trabajar con tu propia
versión.

<a id="sec-0-2"></a>
## 0.2 Cargar el dataset limpio

Dos formas de traer el archivo. La que quede sin comentar tiene que dejar la ruta en la variable
`RUTA_DATOS`.

| Cómo | Cuándo conviene |
|---|---|
| **A** — Desde GitHub | Siempre que haya conexión. Es la que queda **activa**: no hay que descargar ni subir nada |
| **B** — Subir el archivo a mano | Si querés trabajar con tu propia versión del CSV, o no tenés acceso al repositorio |

**Sobre la opción A.** GitHub tiene dos direcciones para cada archivo: la de la página web, que
devuelve el archivo *mostrado dentro del sitio* —es decir, HTML—, y la de `raw.githubusercontent.com`,
que devuelve el contenido crudo. Pandas necesita la segunda. La celda la arma sola a partir del
usuario, el repositorio y la rama, así que para apuntar a otro repositorio alcanza con cambiar esas
tres variables. El repositorio tiene que ser público.

**Sobre la opción B.** `files.upload()` abre un diálogo del navegador para elegir el archivo desde
tu máquina. Solo funciona en Colab, y el archivo vive en la sesión: cuando el entorno se reinicia
hay que volver a subirlo.

In [ ]:
NOMBRE_ARCHIVO = "dataset_limpio.csv"

# ---------------------------------------------------------------------
# OPCIÓN A — Desde GitHub   (ACTIVA)
# ---------------------------------------------------------------------
USUARIO = "FerCipriani"            # <-- cambiar si usás otro repositorio
REPO    = "CursoML_Proyecto_IVE"   # <-- cambiar si usás otro repositorio
RAMA    = "main"                   # GitHub usa 'main' por defecto desde 2020

RUTA_DATOS = f"https://raw.githubusercontent.com/{USUARIO}/{REPO}/{RAMA}/{NOMBRE_ARCHIVO}"

# ---------------------------------------------------------------------
# OPCIÓN B — Subir el archivo a mano   (solo Colab)
# ---------------------------------------------------------------------
# from google.colab import files
# RUTA_DATOS = list(files.upload().keys())[0]

print(f"Ruta configurada: {RUTA_DATOS}")

Ahora las librerías de datos y la lectura.

La celda detecta sola si la ruta termina en `.csv` o en `.xlsx` y usa la función que corresponde,
así que sirve igual si en vez del CSV limpio le pasás un Excel. Y si la lectura falla —típicamente
porque no hay conexión, o porque se cambió el nombre del repositorio— reintenta con un archivo
local del mismo nombre antes de darse por vencida.

In [ ]:
# --- Librerías de manipulación de datos ---
import numpy as np
import pandas as pd

print(f"NumPy  : {np.__version__}")
print(f"Pandas : {pd.__version__}")
print()


# --- Lectura: elige la función según la extensión ---
def leer_datos(ruta):
    if str(ruta).lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(ruta)
    return pd.read_csv(ruta)


try:
    df = leer_datos(RUTA_DATOS)
except Exception as error:
    print(f"[AVISO] No se pudo leer: {RUTA_DATOS}")
    print(f"        {type(error).__name__}: {str(error)[:110]}")
    print(f"        Reintentando con el archivo local '{NOMBRE_ARCHIVO}'...")
    df = leer_datos(NOMBRE_ARCHIVO)
    RUTA_DATOS = NOMBRE_ARCHIVO

print("=" * 70)
print("  LECTURA")
print("=" * 70)
print(f"  Archivo     : {RUTA_DATOS}")
print(f"  Dimensiones : {df.shape[0]} filas x {df.shape[1]} columnas")
print()
display(df.head())

<a id="sec-0-3"></a>
## 0.3 Verificación de entrada

La Parte 1 dejó el dataset auditado, pero este cuaderno **no da eso por sentado**. Repite los
controles esenciales antes de tocar nada.

No es desconfianza gratuita: en un pipeline real, la etapa que consume un archivo no sabe quién lo
generó ni si se modificó en el camino. Alguien pudo haber abierto el CSV en Excel y guardado con
otro separador decimal; pudo haberse subido una versión vieja; pudo haberse leído la hoja
equivocada. Verificar la entrada cuesta una celda y evita horas de confusión.

Los cinco controles: nulos, duplicados, tipos, rangos, y que la etiqueta tenga exactamente dos
clases.

In [ ]:
print("=" * 78)
print("  VERIFICACIÓN DE ENTRADA")
print("=" * 78)

COLUMNA_OBJETIVO = "TAG"
objetivo = COLUMNA_OBJETIVO

problemas = []

# 1. ¿Está la columna objetivo?
if objetivo not in df.columns:
    problemas.append(f"no existe la columna objetivo '{objetivo}'")
    print(f"  [ERROR] No está la columna '{objetivo}'. Columnas: {list(df.columns)}")
else:
    print(f"  [OK] Columna objetivo '{objetivo}' presente")

# 2. Predictoras numéricas
predictoras = [c for c in df.columns
               if c != objetivo and pd.api.types.is_numeric_dtype(df[c])]
no_numericas = [c for c in df.columns if c != objetivo and c not in predictoras]
print(f"  [OK] Variables predictoras numéricas: {len(predictoras)}")
if no_numericas:
    problemas.append(f"columnas no numéricas fuera del objetivo: {no_numericas}")
    print(f"  [AVISO] Columnas no numéricas que quedan fuera: {no_numericas}")

# 3. Nulos
nulos = int(df.isna().sum().sum())
if nulos:
    problemas.append(f"{nulos} valores faltantes")
    print(f"  [ERROR] Hay {nulos} valores faltantes. Volvé a la Parte 1.")
else:
    print("  [OK] Sin valores faltantes")

# 4. Duplicados
duplicados = int(df.duplicated().sum())
if duplicados:
    problemas.append(f"{duplicados} filas duplicadas")
    print(f"  [AVISO] Hay {duplicados} filas duplicadas.")
else:
    print("  [OK] Sin filas duplicadas")

# 5. Clases
clases = sorted(df[objetivo].astype(str).unique())
if len(clases) != 2:
    problemas.append(f"la etiqueta tiene {len(clases)} clases, no 2")
    print(f"  [ERROR] La etiqueta tiene {len(clases)} clases: {clases}")
    print("          Este cuaderno está armado para clasificación BINARIA.")
else:
    print(f"  [OK] Clasificación binaria: {clases}")

# 6. Rangos
if predictoras:
    minimo = df[predictoras].to_numpy().min()
    maximo = df[predictoras].to_numpy().max()
    print(f"  [OK] Rango de las predictoras: {minimo:.6f} a {maximo:.6f}")
    sumas = df[predictoras].sum(axis=1)
    if np.allclose(sumas, 1.0, atol=1e-6):
        print("  [OK] Las filas suman 1: las variables son proporciones de un total fijo")

print()
if problemas:
    print(f"  RESULTADO: {len(problemas)} problema(s) -> {problemas}")
    print("  Revisá la Parte 1 antes de seguir.")
else:
    print("  RESULTADO: el dataset pasó todos los controles. Se puede modelar.")

print()
print(f"  Reparto de clases:")
conteo = df[objetivo].value_counts()
proporcion = df[objetivo].value_counts(normalize=True)
for clase in conteo.index:
    print(f"    {clase:>5} : {conteo[clase]:>4} casos ({proporcion[clase]:.1%})")
linea_base = proporcion.max()
clase_mayoritaria = proporcion.idxmax()
print(f"  LÍNEA DE BASE: {linea_base:.2%}  (responder siempre '{clase_mayoritaria}')")

<a id="sec-0-4"></a>
## 0.4 Configuración global

Todos los valores que se podrían querer cambiar, juntos y con nombre propio.

Dos merecen explicación:

- **`SEMILLA`** — varios procedimientos que vamos a usar tienen un componente aleatorio: cómo se
  mezclan las filas antes de partirlas, qué variables mira un árbol en cada nodo, qué muestra le
  toca a cada árbol del bosque. Fijar la semilla hace que esa aleatoriedad sea siempre la misma y
  que el cuaderno dé el mismo resultado cada vez que se corre. Sin semilla fija, los números
  cambian en cada ejecución y no se puede comparar nada.

- **`CLASE_POSITIVA`** — en clasificación binaria hay que decidir cuál de las dos clases es "la
  positiva". Esa elección **no** afecta el accuracy, pero **sí** afecta la precisión, el recall y
  el F1-score, porque esas métricas están definidas desde el punto de vista de la clase positiva.
  La elección es arbitraria; lo que no es arbitrario es dejarla escrita.

In [ ]:
# --- Reproducibilidad ---
SEMILLA = 42

# --- El problema ---
CLASE_POSITIVA = "AFI"    # la otra pasa a ser la negativa

# --- Validación cruzada ---
N_PLIEGUES = 5

# --- Umbrales del diagnóstico de ajuste (punto 5) ---
BRECHA_SOBREAJUSTE = 0.10   # (score entrenamiento - score validación) por encima de esto
DESVIO_INESTABLE   = 0.08   # desvío entre pliegues por encima de esto
SCORE_SUBAJUSTE    = 0.70   # score de validación por debajo de esto

# --- Formato de la salida ---
import warnings
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
warnings.filterwarnings("ignore")

clase_negativa = [c for c in clases if c != CLASE_POSITIVA][0]

print("Configuración cargada.")
print(f"  Semilla           : {SEMILLA}")
print(f"  Clase positiva    : {CLASE_POSITIVA}  (la negativa es {clase_negativa})")
print(f"  Pliegues (K-Fold) : {N_PLIEGUES}")
print(f"  Umbral sobreajuste: {BRECHA_SOBREAJUSTE}")
print(f"  Umbral desvío     : {DESVIO_INESTABLE}")
print(f"  Umbral subajuste  : {SCORE_SUBAJUSTE}")

<a id="sec-0-5"></a>
## 0.5 Librerías de gráficos y estilo

Este cuaderno grafica en casi todas sus secciones, así que las librerías de visualización y el
estilo se configuran una vez, acá.

- **Matplotlib** — la base. Controla cada detalle: ejes, colores, leyendas.
- **Seaborn** — construida encima. Menos código para gráficos estadísticos.

Fijamos también un color por clase y uno por modelo, para que los gráficos de todo el cuaderno se
puedan comparar de un vistazo. Que la Regresión Logística sea siempre azul y el Random Forest
siempre violeta no es cosmético: es lo que permite leer la comparativa final sin consultar la
leyenda cada vez.

Plotly no se usa en esta Parte; queda en la Parte 1.

In [ ]:
# --- Librerías de gráficos ---
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

# --- Estilo global ---
sns.set_theme(style="whitegrid", palette="deep")
mpl.rcParams["figure.figsize"] = (10, 5)
mpl.rcParams["figure.dpi"] = 110
mpl.rcParams["axes.titlesize"] = 12
mpl.rcParams["axes.titleweight"] = "bold"
mpl.rcParams["font.size"] = 10

# --- Colores fijos ---
COLOR_CLASE = {"AFI": "#1f77b4", "NEG": "#d62728"}

print(f"Matplotlib : {mpl.__version__}")
print(f"Seaborn    : {sns.__version__}")
print()
print("Colores por clase:", COLOR_CLASE)

---
# 1. ¿Qué es Machine Learning? Dónde estamos parados

<a id="sec-1-1"></a>
## 1.1 Aprendizaje supervisado y no supervisado

El **aprendizaje automático** (*Machine Learning*) es la rama de la inteligencia artificial que
permite a un programa aprender de los datos en lugar de seguir reglas escritas a mano.

La diferencia con la programación tradicional es de fondo. En el enfoque clásico, una persona
define las reglas: *si el asunto contiene tal palabra y el remitente está en tal lista, entonces es
spam*. En Machine Learning el sistema **descubre las reglas solo**, a partir de ejemplos ya
resueltos. Y lo que se busca no es que memorice esos ejemplos, sino que **generalice**: que
funcione con casos nuevos que nunca vio.

**Los dos tipos principales de aprendizaje**

- **Supervisado.** Los datos vienen con la respuesta conocida (una "etiqueta"). El modelo busca la
  relación entre las variables de entrada y esa etiqueta. Se divide a su vez en **regresión**
  (etiqueta numérica continua) y **clasificación** (etiqueta categórica).
- **No supervisado.** No hay etiqueta. El objetivo es descubrir estructura oculta en los datos. El
  caso típico es el *clustering*: agrupar casos parecidos entre sí.

La diferencia está en si el dataset tiene o no etiquetas que guíen al modelo. El nuestro las tiene:
la columna `TAG`. Es un problema **supervisado**.

**El ciclo de vida de un proyecto**

| Etapa | Dónde está |
|---|---|
| 1. Recolección y comprensión de los datos | Parte 1, puntos 0.2 y 1 |
| 2. Análisis exploratorio (EDA) y limpieza | Parte 1, puntos 2 y 3 |
| 3. Preprocesamiento y preparación | **Parte 2, punto 2** |
| 4. Selección y entrenamiento del modelo | **Parte 2, puntos 8 a 11** |
| 5. Evaluación del modelo | **Parte 2, puntos 6 a 12** |
| 6. Implementación y monitoreo | fuera del alcance de estos cuadernos |
| 7. Mantenimiento y mejora continua | fuera del alcance de estos cuadernos |

Entre las dos partes cubrimos las etapas 1 a 5. Las dos últimas empiezan cuando el modelo sale a
producción.

<a id="sec-1-2"></a>
## 1.2 Regresión y clasificación: dos preguntas distintas

Los dos son problemas de aprendizaje supervisado —los dos parten de datos con la respuesta
conocida— pero responden preguntas de naturaleza distinta.

### Regresión: la pregunta es "¿cuánto?"

La variable objetivo es un **número continuo**: un precio, una cantidad de días, una temperatura,
un monto. La salida del modelo puede ser cualquier valor real.

El modelo básico es la **regresión lineal**, que ajusta la recta `y = a·x + b` que mejor pasa entre
los puntos —o su versión con varias variables,
`y = a₁·x₁ + a₂·x₂ + … + aₙ·xₙ + b`—. Se evalúa con métricas que miden *cuán lejos* quedó de la
realidad:

| Métrica | Qué mide | Cuándo conviene |
|---|---|---|
| **MAE** (error absoluto medio) | El promedio de los errores, en las unidades originales | Cuando se quiere una idea clara del margen de error: "se equivoca en promedio por $8.500" |
| **MSE** (error cuadrático medio) | Eleva cada error al cuadrado antes de promediar | Cuando los errores grandes son mucho peores que los chicos. Pierde las unidades originales |
| **RMSE** (raíz del MSE) | La raíz cuadrada del anterior | El punto medio: castiga los errores grandes pero recupera las unidades |
| **R²** (coeficiente de determinación) | Qué proporción de la variabilidad explica el modelo | Para saber cuánto aporta el modelo respecto de responder siempre el promedio |

**Ejemplos de regresión:** predecir el precio de una casa según su superficie y ubicación; estimar
cuántos días va a durar una internación; anticipar el monto de los reclamos de un cliente en un
año; calcular el tiempo de entrega de un envío.

### Clasificación: la pregunta es "¿cuál?"

La variable objetivo es una **categoría**. Cuando hay solo dos categorías posibles se llama
**clasificación binaria**, que es nuestro caso. Con tres o más, **multiclase**.

El modelo básico es la **regresión logística** que —a pesar del nombre— no predice un número sino
la **probabilidad** de pertenecer a una clase: calcula una combinación lineal de las variables y la
pasa por una función **sigmoide**, que aplasta cualquier número real al intervalo entre 0 y 1.
Después se aplica un **umbral de corte**, habitualmente 0,5, para convertir esa probabilidad en una
decisión.

Las métricas son completamente distintas, porque ya no medimos distancias sino aciertos y errores:

| Métrica | Qué mide | Cuándo conviene |
|---|---|---|
| **Accuracy** (precisión general) | Qué proporción del total acertó | Cuando las clases están balanceadas |
| **Recall** (sensibilidad) | De los casos que realmente eran positivos, cuántos detectó | Cuando dejar pasar un positivo es caro: diagnóstico médico, detección de fraude |
| **Precisión** | De los que predije positivos, cuántos lo eran | Cuando una falsa alarma es cara |
| **F1-score** | La media armónica entre precisión y recall | Cuando no se quiere privilegiar ninguna de las dos, o las clases están desbalanceadas |
| **Matriz de confusión** | El desglose completo de aciertos y errores | Siempre. Es la que muestra *cómo* se equivoca, no solo cuánto |

**Ejemplos de clasificación:** decidir si un correo es spam; predecir si un cliente va a cancelar
la suscripción; determinar si un paciente tiene o no una enfermedad; predecir si un estudiante
aprueba o no el curso.

### Por qué no se puede usar una en lugar de la otra

Se podría forzar una regresión lineal a predecir valores cercanos a 0 y a 1, pero el resultado no
tendría interpretación probabilística válida: nada impide que devuelva -0,3 o 1,4, valores que no
significan nada como probabilidad. Y el modelo comete errores graves justamente en los extremos, que
es donde más importa. Al revés tampoco funciona: un clasificador no puede estimar un monto.

**En una línea:** si la pregunta es "¿cuánto?", regresión. Si es "¿cuál de estas dos?",
clasificación.

<a id="sec-1-3"></a>
## 1.3 Diagnóstico de nuestro problema

Lo verificamos con código en lugar de afirmarlo: miramos el tipo de la variable objetivo y su
cantidad de valores distintos.

In [ ]:
print("=" * 78)
print("  DIAGNÓSTICO DEL PROBLEMA")
print("=" * 78)

es_numerica = pd.api.types.is_numeric_dtype(df[objetivo])
n_valores = df[objetivo].nunique()

print(f"  Variable objetivo      : {objetivo}")
print(f"  Tipo de dato           : {df[objetivo].dtype}")
print(f"  ¿Es numérica continua? : {'sí' if es_numerica and n_valores > 20 else 'no'}")
print(f"  Valores distintos      : {n_valores}  ->  {clases}")
print(f"  Variables predictoras  : {len(predictoras)}, todas numéricas")
print(f"  Observaciones          : {len(df)}")
print()

if es_numerica and n_valores > 20:
    tipo_problema = "REGRESIÓN"
elif n_valores == 2:
    tipo_problema = "CLASIFICACIÓN BINARIA"
else:
    tipo_problema = f"CLASIFICACIÓN MULTICLASE ({n_valores} clases)"

print(f"  TIPO DE PROBLEMA : {tipo_problema}")
print()
print("  La variable objetivo es una etiqueta con dos valores posibles, no un número.")
print("  No tiene sentido preguntarse 'cuánto vale TAG': la pregunta es a cuál de las")
print("  dos clases pertenece cada caso.")
print()
print("  En consecuencia:")
print("    - NO se usa regresión lineal, ni MAE / MSE / RMSE / R².")
print("    - SÍ se usan modelos de clasificación, evaluados con accuracy, precisión,")
print("      recall, F1-score, matriz de confusión y curva ROC.")

<a id="sec-1-4"></a>
## 1.4 Los cuatro modelos que vamos a usar

La idea no es amontonar modelos sino elegir un conjunto que cubra **formas distintas de razonar**.
Si los cuatro funcionaran igual, tres estarían de más; que rindan distinto es lo que nos enseña
algo sobre los datos.

| Modelo | Cómo decide | Qué frontera dibuja | ¿Necesita escalado? | Por qué está en la lista |
|---|---|---|---|---|
| **Regresión Logística** | Calcula una combinación lineal de las variables y la pasa por una sigmoide para obtener una probabilidad | Un plano | **Sí** | Es el modelo de referencia obligado en clasificación binaria: rápido, estable e interpretable. Sus coeficientes dicen cuánto pesa cada variable y en qué dirección. Cualquier modelo más complejo tiene que justificar su complejidad superándolo |
| **K-Nearest Neighbors** | Mira los *k* casos más parecidos y vota por mayoría | Cualquiera, dictada por los datos | **Sí, imprescindible** | No ajusta ninguna fórmula: se apoya directamente en la proximidad entre casos. Es el contraste perfecto con la logística, y como decide por distancias, es el modelo donde el efecto del escalado se ve con más claridad |
| **Árbol de Decisión** | Una cadena de preguntas del tipo "¿esta variable es mayor que tanto?" | Escalones perpendiculares a los ejes | **No** | Captura relaciones no lineales e interacciones entre variables, y se puede dibujar y leer entero. Además, al ser indiferente a la escala, sirve para verificar que nuestra comparación de escalados hace lo que creemos |
| **Random Forest** | Muchos árboles distintos votando | Muchos escalones promediados | **No** | Entra en el punto 11 con doble propósito: suele ser el que mejor rinde de los cuatro, y sirve de demostración de que agregar un modelo al cuaderno cuesta tres líneas |

Los tres primeros están en el programa del curso —clases 7, 8 y 9—. El cuarto es una extensión
natural del tercero.

**Lo que dejamos afuera, y por qué:** SVM con kernel, redes neuronales y gradient boosting pueden
rendir muy bien, pero con 171 filas el margen de mejora sobre estos cuatro es difícil de demostrar,
y ninguno aporta una forma de razonar que no esté ya representada. Si querés probarlos igual, el
punto 7.5 explica cómo agregarlos.

---
# 2. Transformación de datos

Los datos están limpios y verificados. Ahora hay que **prepararlos para el modelo**. Esta etapa no
se ve en el resultado final, pero decide buena parte de él.

Tres tareas:

1. **Feature engineering** — ¿conviene fabricar variables nuevas a partir de las que tenemos?
2. **Codificación** — los algoritmos no leen texto: hay que convertir las categorías en números.
3. **Escalado** — poner las variables numéricas en escalas comparables.

Una advertencia que va a atravesar todo este punto y explota en el punto 4: la codificación se
puede hacer ahora, sobre todo el dataset, porque no aprende nada de los datos. El escalado **no**:
aprende parámetros (mínimos, máximos, medias) y tiene que aplicarse después de la división, adentro
del pipeline. Acá lo vamos a explicar y a mostrar, pero no a aplicar.

<a id="sec-2-1"></a>
## 2.1 Feature engineering

**Feature engineering** (ingeniería de características) es crear, seleccionar, transformar o
eliminar variables para que el modelo tenga una mejor representación del problema. Los ejemplos
clásicos: extraer el mes y el día de la semana de una fecha, o calcular el margen restando el costo
del precio de venta.

Se suele decir que una buena ingeniería de características mejora más el resultado que cambiar de
algoritmo. Es cierto **cuando hay información latente que el modelo no puede deducir solo**.

En nuestro caso no hay fechas ni campos compuestos, pero sí podemos construir variables que resuman
la **forma del reparto** de cada fila. Vamos a fabricar tres y ver si aportan algo:

- **`maximo`** — el valor de la variable dominante en ese caso. Mide cuán concentrado está.
- **`brecha`** — la diferencia entre la variable más alta y la segunda. Mide cuán clara es esa
  dominancia.
- **`entropia`** — mide cuán repartido está el total entre las siete variables. Es baja cuando casi
  todo está en una sola y alta cuando está parejo.

Las construimos, medimos su poder de separación con la **d de Cohen** —la diferencia entre las
medias de las dos clases dividida por el desvío combinado, la misma vara que usamos en la Parte 1—
y recién ahí decidimos si entran al modelo.

In [ ]:
def d_de_cohen(serie, etiquetas, clase_a, clase_b):
    a = serie[etiquetas == clase_a].to_numpy()
    b = serie[etiquetas == clase_b].to_numpy()
    na, nb = len(a), len(b)
    s = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    return 0.0 if s == 0 else (a.mean() - b.mean()) / s


def etiqueta_efecto(m):
    if m < 0.2:
        return "despreciable"
    if m < 0.5:
        return "chico"
    if m < 0.8:
        return "mediano"
    return "grande"


clase_a, clase_b = clases[0], clases[1]

# --- Poder de separación de las variables ORIGINALES, como referencia ---
separacion = pd.DataFrame({
    "variable": predictoras,
    "d_cohen": [d_de_cohen(df[c], df[objetivo], clase_a, clase_b) for c in predictoras],
})
separacion["magnitud"] = separacion["d_cohen"].abs()
separacion["efecto"] = separacion["magnitud"].map(etiqueta_efecto)
separacion = separacion.sort_values("magnitud", ascending=False).reset_index(drop=True)
orden = separacion["variable"].tolist()

print("=" * 70)
print("  VARIABLES ORIGINALES, ordenadas por poder de separación")
print("=" * 70)
display(separacion)

# --- Las tres variables derivadas ---
proporciones = df[predictoras].to_numpy()
ordenadas = np.sort(proporciones, axis=1)[:, ::-1]      # de mayor a menor

nuevas = pd.DataFrame(index=df.index)
nuevas["maximo"] = ordenadas[:, 0]
nuevas["brecha"] = ordenadas[:, 0] - ordenadas[:, 1]
seguro = np.clip(proporciones, 1e-12, None)
nuevas["entropia"] = -(seguro * np.log(seguro)).sum(axis=1)

evaluacion = pd.DataFrame({
    "variable": nuevas.columns,
    "d_cohen": [d_de_cohen(nuevas[c], df[objetivo], clase_a, clase_b) for c in nuevas.columns],
})
evaluacion["magnitud"] = evaluacion["d_cohen"].abs()
evaluacion["efecto"] = evaluacion["magnitud"].map(etiqueta_efecto)

print("=" * 70)
print("  VARIABLES DERIVADAS: ¿aportan separación?")
print("=" * 70)
display(evaluacion)
print(f"  Mejor variable ORIGINAL : |d| = {separacion['magnitud'].max():.3f}")
print(f"  Mejor variable DERIVADA : |d| = {evaluacion['magnitud'].max():.3f}")

fig, ejes = plt.subplots(1, 3, figsize=(14, 3.6))
for i, col in enumerate(nuevas.columns):
    for clase in clases:
        sns.kdeplot(nuevas.loc[df[objetivo] == clase, col], ax=ejes[i],
                    fill=True, alpha=0.35, linewidth=1.8,
                    color=COLOR_CLASE.get(clase), label=str(clase))
    d = evaluacion.loc[evaluacion["variable"] == col, "d_cohen"].iloc[0]
    ejes[i].set_title(f"{col}  (d = {d:+.2f})")
    ejes[i].set_xlabel(""); ejes[i].set_ylabel("")
    ejes[i].legend(fontsize=8, title=objetivo)
plt.tight_layout()
plt.show()

**Decisión: no las incorporamos.**

Tres razones:

1. **No traen información nueva.** Las tres se calculan enteramente a partir de las siete columnas
   originales. Un modelo no lineal como el árbol o el bosque puede reconstruir esas relaciones solo
   si le sirven. Agregarlas es repetirle algo que ya tiene.
2. **Agregan redundancia.** Cada variable derivada está fuertemente correlacionada con las
   originales de las que sale, y eso desestabiliza los coeficientes de la regresión logística sin
   mejorar sus predicciones.
3. **Ensucian la comparación.** El objetivo de este cuaderno es comparar cuatro modelos y dos
   escalados sobre las mismas entradas. Si además cambiáramos el conjunto de variables, no
   podríamos saber a qué atribuir una diferencia.

Vale la pena decir que esto **no** es una regla general: en la mayoría de los problemas reales el
feature engineering es de lo que más rinde. Acá no aplica porque las siete variables ya son una
descripción completa de cada caso.

Y una advertencia que se conecta con el punto 4: fabricar variables mirando la etiqueta —por
ejemplo, quedarse solo con las que mejor separan según la tabla de arriba— **es fuga de
información** si se hace sobre el dataset completo. La tabla de arriba sirve para entender los
datos, no para seleccionar variables.

Si más adelante quisieras probarlas, alcanza con descomentar las dos líneas de abajo: el resto del
cuaderno las tomaría solo, porque todo trabaja sobre la lista `predictoras`.

In [ ]:
# --- Para incorporar las variables derivadas, descomentá estas dos líneas ---
# df = pd.concat([df, nuevas], axis=1)
# predictoras = predictoras + list(nuevas.columns)

print(f"Variables predictoras en uso ({len(predictoras)}):")
for c in predictoras:
    print(f"    - {c}")

<a id="sec-2-2"></a>
## 2.2 Codificación de la variable categórica

Los algoritmos de scikit-learn trabajan con números. Una columna de texto tiene que convertirse
antes. Hay dos técnicas principales:

**Label Encoding** — asigna un entero a cada categoría: `AFI → 1`, `NEG → 0`. Es directo, no agrega
columnas. El riesgo: introduce un **orden** donde puede no haberlo. Si codificamos
`rojo=0, azul=1, verde=2`, el modelo puede interpretar que verde es "más" que rojo, lo cual es un
sinsentido. Por eso solo conviene en variables **ordinales** (bajo / medio / alto).

**One-Hot Encoding** — crea una columna binaria por categoría. Con `rojo`, `azul` y `verde` se
generan tres columnas de ceros y unos. No introduce ningún orden, así que es la opción segura para
variables **nominales**. El costo: si la variable tiene muchas categorías distintas, genera
muchísimas columnas. En ese caso conviene agrupar las poco frecuentes en un "Otros".

**Cuál corresponde acá.** Nuestra única columna de texto es la **variable objetivo**, y la regla es
distinta para el objetivo que para las predictoras: en clasificación binaria el objetivo se
codifica **siempre** como 0 y 1 con Label Encoding. No hay problema de orden falso porque con dos
clases solo hay dos valores y ningún modelo interpreta una jerarquía entre ellos. One-Hot sobre un
objetivo binario generaría dos columnas perfectamente redundantes.

Las siete predictoras ya son numéricas: no hay nada que codificar ahí.

**Por qué esto sí se puede hacer sobre todo el dataset.** Codificar no aprende nada de los datos:
es un reemplazo fijo, `AFI` por 1 y `NEG` por 0. No hay parámetro estimado, así que no hay
información que pueda filtrarse. Con el escalado, en la sección siguiente, la historia es otra.

In [ ]:
# --- Librería de preprocesamiento ---
from sklearn.preprocessing import LabelEncoder

# --- LABEL ENCODING de la variable objetivo ---
# Fijamos explícitamente qué clase es la positiva (1) en lugar de dejarlo al azar
# del orden alfabético, porque precisión, recall y F1 se calculan respecto de ella.

mapa_clases = {clase_negativa: 0, CLASE_POSITIVA: 1}
mapa_inverso = {v: k for k, v in mapa_clases.items()}

y = df[objetivo].map(mapa_clases).astype(int)

print("=" * 70)
print("  CODIFICACIÓN DE LA VARIABLE OBJETIVO")
print("=" * 70)
for etiqueta, codigo in mapa_clases.items():
    rol = "POSITIVA" if codigo == 1 else "negativa"
    print(f"  {etiqueta:>5}  ->  {codigo}   (clase {rol})   {int((y == codigo).sum())} casos")
print()
print(f"  Las métricas orientadas a una clase (precisión, recall, F1) se reportan")
print(f"  respecto de la clase positiva: {CLASE_POSITIVA}")

# El mismo resultado con LabelEncoder de scikit-learn, para mostrar la herramienta.
# Ojo: LabelEncoder asigna los números por orden alfabético, sin preguntar.
codificador = LabelEncoder()
y_sklearn = codificador.fit_transform(df[objetivo])
print()
print(f"  Con LabelEncoder el mapeo sería: "
      f"{dict(zip(codificador.classes_, range(len(codificador.classes_))))}")
print(f"  Coincide con el nuestro: {bool((y_sklearn == y.to_numpy()).all())}")

Y así se vería el One-Hot Encoding, solo para tener presente la diferencia. Se aplica a **variables
predictoras nominales**, que en este dataset no existen.

In [ ]:
demo_onehot = pd.get_dummies(df[[objetivo]], columns=[objetivo], prefix="clase").astype(int)

print("=" * 70)
print("  ONE-HOT ENCODING (demostración)")
print("=" * 70)
display(pd.concat([df[[objetivo]].head(6), demo_onehot.head(6)], axis=1))
print(f"  Una columna de texto se convirtió en {demo_onehot.shape[1]} columnas binarias.")
print()
print("  Notar que las dos columnas son complementarias: una es 1 - la otra.")
print("  Para un objetivo binario eso es información duplicada, y por eso usamos")
print("  Label Encoding. One-Hot es para predictoras nominales de 3 o más categorías.")
print()
print("  La forma habitual en la práctica sería:")
print('    df = pd.get_dummies(df, columns=["color", "ciudad"], drop_first=True)')
print("  drop_first=True elimina una columna por variable para evitar la redundancia.")

<a id="sec-2-3"></a>
## 2.3 Normalización y estandarización

La **escala** de una variable es el rango de valores que puede tomar. Cuando unas variables van de
0 a 1 y otras de 0 a 100.000, algunos modelos le dan más peso a las segundas solo porque sus
números son más grandes, no porque sean más importantes.

Las dos técnicas que vamos a usar son las del curso:

**Normalización (Min-Max Scaling)**

$$X_{norm} = \frac{X - X_{min}}{X_{max} - X_{min}}$$

Lleva todos los valores al rango 0–1: el mínimo pasa a ser 0, el máximo 1, y el resto queda
proporcionalmente en el medio. Conserva la forma exacta de la distribución. Es la opción natural
cuando el rango está acotado y para modelos basados en distancias como KNN. Su punto débil: un solo
valor extremo define el máximo y comprime todo lo demás contra el cero.

**Estandarización (Z-score Scaling)**

$$X_{std} = \frac{X - \mu}{\sigma}$$

Centra los datos en media 0 y los lleva a desvío 1. La mayoría de los valores queda entre -2 y 2.
No acota el rango. Es la opción habitual cuando no hay un rango fijo y para modelos que suponen
distribuciones centradas, como la regresión logística.

**Y hay muchas más.** Estas son las otras opciones que ofrece scikit-learn, con sus ventajas y sus
desventajas:

| Escalador | Qué hace | Ventaja | Desventaja |
|---|---|---|---|
| `RobustScaler` | Resta la mediana y divide por el IQR | Los outliers casi no lo afectan | No acota el rango; menos intuitivo |
| `MaxAbsScaler` | Divide por el máximo valor absoluto | Preserva los ceros; ideal para datos ralos | Muy sensible a un único valor extremo |
| `Normalizer` | Escala cada **fila** a longitud 1 | Útil cuando importa la dirección y no la magnitud | Trabaja por fila, no por columna: es otra cosa |
| `PowerTransformer` | Yeo-Johnson o Box-Cox | Corrige asimetría; acerca los datos a una normal | Deforma la escala original; difícil de interpretar |
| `QuantileTransformer` | Fuerza una distribución uniforme o normal | Neutraliza outliers por completo | Muy agresivo; necesita bastantes datos |
| `np.log1p` | Logaritmo de (1 + x) | Simple y eficaz contra colas largas | Solo para valores no negativos |

**Qué modelos lo necesitan y cuáles no**

| Modelo | ¿Necesita escalado? | Por qué |
|---|---|---|
| Regresión Logística | **Sí** | Usa regularización, que penaliza los coeficientes por igual; si las escalas difieren, la penalización es injusta con las variables de números chicos |
| KNN | **Sí, imprescindible** | Todas sus decisiones son distancias |
| SVM | **Sí** | También trabaja con distancias y márgenes |
| Redes neuronales | **Sí** | El descenso de gradiente converge mucho mejor |
| Árbol de Decisión | **No** | Corta por umbrales; una transformación monótona no cambia dónde corta |
| Random Forest | **No** | Es un conjunto de árboles |
| Gradient Boosting | **No** | Ídem |

Escalar no agrega información. Cambia cómo el modelo la interpreta.

**Un aviso antes del código.** Lo que sigue es una **demostración visual** del efecto de cada
escalador, no la aplicación real. El escalado de verdad se aplica adentro del pipeline, después de
la división, por las razones que explica el punto 4. Fijate que la celda trabaja sobre una copia y
**no modifica `df`**.

In [ ]:
# --- Los escaladores ---
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler

# OJO: esto es una DEMOSTRACIÓN sobre una copia. No modifica df.
ejemplo = df[predictoras].copy()

mostrar = pd.DataFrame({
    "original_min": ejemplo.min(),
    "original_max": ejemplo.max(),
    "original_media": ejemplo.mean(),
    "original_desvio": ejemplo.std(),
})

minmax_demo = pd.DataFrame(MinMaxScaler().fit_transform(ejemplo),
                           columns=predictoras, index=ejemplo.index)
zscore_demo = pd.DataFrame(StandardScaler().fit_transform(ejemplo),
                           columns=predictoras, index=ejemplo.index)
robusto_demo = pd.DataFrame(RobustScaler().fit_transform(ejemplo),
                            columns=predictoras, index=ejemplo.index)

mostrar["minmax_min"] = minmax_demo.min()
mostrar["minmax_max"] = minmax_demo.max()
mostrar["zscore_media"] = zscore_demo.mean()
mostrar["zscore_desvio"] = zscore_demo.std(ddof=0)

print("=" * 88)
print("  EFECTO DE CADA ESCALADOR")
print("=" * 88)
display(mostrar)
print("  Min-Max deja todo entre 0 y 1.  Z-score deja media 0 y desvío 1.")
print("  Ninguno de los dos cambia la FORMA de la distribución: solo el eje.")

fig, ejes = plt.subplots(1, 4, figsize=(17, 4))
for eje, datos, titulo in zip(
        ejes,
        [ejemplo, minmax_demo, zscore_demo, robusto_demo],
        ["Original", "Min-Max (0 a 1)", "Z-score (media 0, desvío 1)", "Robusto (mediana e IQR)"]):
    datos_largos = datos.melt(var_name="variable", value_name="valor")
    sns.boxplot(data=datos_largos, x="variable", y="valor", ax=eje,
                color="#4c72b0", fliersize=2.5)
    eje.set_title(titulo, fontsize=10)
    eje.tick_params(axis="x", rotation=90)
    eje.set_xlabel("")
plt.tight_layout()
plt.show()

print(f"  df NO se modificó: sigue con {df.shape[0]} filas y {df.shape[1]} columnas,")
print(f"  y sus valores originales (máximo global = {df[predictoras].to_numpy().max():.4f}).")

<a id="sec-2-4"></a>
## 2.4 Las matrices X e y

Armamos las dos estructuras que va a consumir el resto del cuaderno:

- **`X`** — la matriz de variables predictoras: una fila por caso, una columna por variable.
- **`y`** — el vector objetivo, ya codificado como 0 y 1.

`X` queda **sin escalar**, a propósito. El escalado se aplica dentro de cada pipeline. El punto
siguiente explica por qué eso no es una preferencia de estilo sino una necesidad.

In [ ]:
X = df[predictoras].copy()
# y quedó definido en 2.2

print("=" * 70)
print("  DATOS LISTOS PARA MODELAR")
print("=" * 70)
print(f"  X : {X.shape[0]} filas x {X.shape[1]} columnas  (SIN escalar)")
print(f"  y : {y.shape[0]} etiquetas  ->  "
      f"{int((y == 1).sum())} de clase 1 ({CLASE_POSITIVA}), "
      f"{int((y == 0).sum())} de clase 0 ({clase_negativa})")
print()
print("  Verificaciones finales:")
print(f"    Valores faltantes en X  : {int(X.isna().sum().sum())}")
print(f"    Valores faltantes en y  : {int(y.isna().sum())}")
print(f"    Columnas no numéricas   : {list(X.select_dtypes(exclude=[np.number]).columns)}")
print(f"    Filas alineadas X <-> y : {len(X) == len(y)}")
print()
display(X.head())

---
# 3. División del dataset: entrenamiento y prueba

<a id="sec-3-1"></a>
## 3.1 Por qué hay que dividir

Entrenar un modelo es enseñarle a reconocer patrones. Para saber si **realmente** aprendió, hay que
evaluarlo con datos que no vio durante el entrenamiento.

Si usáramos los mismos datos para las dos cosas, el modelo podría parecer excelente simplemente
porque ya conocía las respuestas. Es como tomarle a alguien un examen con exactamente las mismas
preguntas que estudió: un 10 no prueba que haya entendido el tema, solo que tiene buena memoria.

Por eso la práctica estándar es partir el dataset en dos:

- **Conjunto de entrenamiento** (*training set*) — con el que el modelo ajusta sus parámetros
  internos. Suele ser el 70–80%. Cuantos más datos tenga, mejor puede identificar patrones
  complejos.
- **Conjunto de prueba** (*test set*) — reservado, se usa **una sola vez**, al final, para estimar
  cómo se comportaría el modelo con datos nuevos. Simula el "mundo real": los datos que el modelo
  va a encontrar cuando esté en producción.

Las proporciones habituales son 80/20, 70/30 o 90/10, según el tamaño del dataset. Con pocos datos
conviene dejar más para entrenar; con muchos, se puede ser más generoso con la prueba.

La imagen que lo explica bien: no hacer esta división es como construir un puente y ensayarlo solo
con las condiciones del plano original. Puede parecer perfecto, hasta que le toca el primer viento
fuerte.

<a id="sec-3-2"></a>
## 3.2 La división 80/20 estratificada

Dos detalles de implementación que importan:

- **`stratify=y`** mantiene la misma proporción de clases en las dos partes. Sin esto, el azar
  podría dejar el conjunto de prueba con muchos más casos de una clase que de la otra, y la
  evaluación quedaría distorsionada. Con clases desbalanceadas es prácticamente obligatorio.
- **`random_state`** fija la aleatoriedad para que la división sea siempre la misma y los
  resultados se puedan reproducir.

In [ ]:
# --- Herramienta de división ---
from sklearn.model_selection import train_test_split

X_entrena, X_prueba, y_entrena, y_prueba = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEMILLA
)

print("=" * 70)
print("  DIVISIÓN 80 / 20 ESTRATIFICADA")
print("=" * 70)
print(f"  Entrenamiento : {len(X_entrena):>4} casos  ({len(X_entrena)/len(X):.1%})")
print(f"  Prueba        : {len(X_prueba):>4} casos  ({len(X_prueba)/len(X):.1%})")
print()
print("  Reparto de clases (la estratificación lo mantiene parejo):")
print(f"    {'':>16} {CLASE_POSITIVA:>8} {clase_negativa:>8}   % positiva")
for nombre, yy in [("Dataset completo", y), ("Entrenamiento", y_entrena), ("Prueba", y_prueba)]:
    print(f"    {nombre:>16} {int((yy==1).sum()):>8} {int((yy==0).sum()):>8} "
          f"{(yy==1).mean():>11.1%}")
print()
print("  Sin stratify, el conjunto de prueba de 34 casos podría quedar con 25 de")
print("  una clase y 9 de la otra solo por azar, y las métricas serían irreales.")

<a id="sec-3-3"></a>
## 3.3 La fragilidad de una sola división

Con este dataset, esa división tiene un problema serio que conviene ver antes de seguir.

**34 casos de prueba son muy pocos.** Cada acierto o error mueve el accuracy en casi 3 puntos
porcentuales. Y el resultado depende enteramente de **qué** 34 casos hayan caído del lado de la
prueba, que es cosa del azar.

Lo comprobamos: entrenamos el mismo modelo, con los mismos hiperparámetros, cambiando solamente el
`random_state` de la división. Si la estimación fuera estable, los resultados deberían ser
parecidos.

(Usamos una regresión logística solo como vehículo para la demostración. El modelo en serio,
explicado y evaluado como corresponde, viene en el punto 8.)

In [ ]:
# --- Solo para esta demostración ---
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

modelo_demo = Pipeline([
    ("escalador", StandardScaler()),
    ("modelo", LogisticRegression(max_iter=5000, random_state=SEMILLA)),
])

resultados_particion = []
for semilla in range(12):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.20, stratify=y,
                                          random_state=semilla)
    modelo_demo.fit(Xtr, ytr)
    predicho = modelo_demo.predict(Xte)
    resultados_particion.append({
        "random_state": semilla,
        "accuracy": accuracy_score(yte, predicho),
        "f1_macro": f1_score(yte, predicho, average="macro"),
    })

resultados_particion = pd.DataFrame(resultados_particion)

print("=" * 70)
print("  EL MISMO MODELO, 12 DIVISIONES DISTINTAS")
print("=" * 70)
display(resultados_particion)

acc = resultados_particion["accuracy"]
print(f"  Accuracy mínimo : {acc.min():.4f}")
print(f"  Accuracy máximo : {acc.max():.4f}")
print(f"  Rango           : {acc.max() - acc.min():.4f}  "
      f"({(acc.max()-acc.min())*100:.1f} puntos porcentuales)")
print(f"  Media           : {acc.mean():.4f}")
print(f"  Desvío          : {acc.std():.4f}")

fig, eje = plt.subplots(figsize=(10, 4))
eje.plot(resultados_particion["random_state"], acc, "o-", color="#4c72b0",
         linewidth=1.8, markersize=8, label="accuracy de cada división")
eje.axhline(acc.mean(), color="#d62728", linestyle="--", linewidth=1.6,
            label=f"media = {acc.mean():.3f}")
eje.fill_between(resultados_particion["random_state"],
                 acc.mean() - acc.std(), acc.mean() + acc.std(),
                 color="#d62728", alpha=0.12, label="± 1 desvío")
eje.set_xlabel("random_state de la división")
eje.set_ylabel("accuracy sobre el conjunto de prueba")
eje.set_title("El mismo modelo evaluado sobre 12 particiones distintas")
eje.legend()
plt.tight_layout()
plt.show()

print()
print("  CONCLUSIÓN: el mismo modelo, con los mismos datos y los mismos")
print("  hiperparámetros, arroja resultados muy distintos según cómo se")
print("  haya partido el dataset. Reportar el número de UNA división sería")
print("  reportar en buena medida el azar.")
print("  Esto es lo que la validación cruzada viene a resolver (punto 6).")

---
# 4. Fuga de información (data leakage)

<a id="sec-4-1"></a>
## 4.1 Qué es

**Fuga de información** es cualquier situación en la que el modelo tuvo acceso, directa o
indirectamente, a información del conjunto de prueba durante el entrenamiento.

La palabra clave es **indirectamente**. Nadie copia el conjunto de prueba dentro del de
entrenamiento: eso se vería enseguida. La fuga se da por caminos mucho más sutiles, y el más común
de todos es este:

> Calcular algo usando **todos** los datos, y después dividir.

Un ejemplo concreto. Querés estandarizar las variables. Calculás la media y el desvío de cada
columna sobre el dataset completo, aplicás la transformación, y recién entonces dividís en
entrenamiento y prueba. Parece razonable —todavía no entrenaste nada—, pero acabás de filtrar
información: esa media y ese desvío contienen los valores del conjunto de prueba. El modelo, al
entrenarse sobre datos transformados con esos parámetros, recibe una pista sobre cómo son los datos
que después lo van a evaluar.

La regla, en general: **cualquier cosa que se aprenda de los datos tiene que aprenderse solamente
del conjunto de entrenamiento.** Medias, desvíos, mínimos, máximos, medianas, listas de variables
seleccionadas, categorías conocidas, hiperparámetros. Todo.

<a id="sec-4-2"></a>
## 4.2 Por qué es tan grave: el síntoma es que no hay síntoma

Un error de programación tira una excepción. Un modelo mal configurado da métricas malas. La fuga
de información no hace ninguna de las dos cosas: **hace que todo se vea mejor**.

Ese es exactamente el problema. La secuencia típica es así:

1. Entrenás un modelo. Las métricas dan sorprendentemente bien: 96% de accuracy.
2. Nadie sospecha, porque un resultado bueno no se cuestiona con la misma severidad que uno malo.
3. El modelo sale a producción.
4. En producción rinde 78%.
5. Nadie entiende por qué, porque el código "es el mismo".

Y el diagnóstico es difícil justamente porque el código **es** el mismo. La diferencia está en algo
que pasó antes: un `fit_transform` aplicado dos líneas más arriba de donde correspondía.

Tres consecuencias prácticas:

- **La evaluación miente, y miente hacia arriba.** Todas las decisiones que tomes a partir de esas
  métricas —qué modelo elegir, qué variables conservar, si el proyecto sirve— están basadas en
  números inflados.
- **La comparación entre modelos se rompe.** Si un modelo tiene fuga y otro no, el que gana es el
  que tiene fuga, no el mejor.
- **No hay alarma.** No hay error, no hay aviso, no hay nada que mirar. Solo un número lindo.

Por eso este punto tiene sección propia, y por eso la prevención tiene que ser **estructural**
—usar un `Pipeline`— y no depender de acordarse en cada caso.

<a id="sec-4-3"></a>
## 4.3 Las siete formas en que se cuela

Estas son las vías más frecuentes, ordenadas de la más común a la más sutil.

### 1. Escalar antes de dividir

La clásica. `StandardScaler().fit_transform(X)` sobre el dataset completo y después
`train_test_split`. La media y el desvío que usa la transformación contienen los datos de prueba.

**Cómo se evita:** `fit_transform` sobre el entrenamiento, `transform` sobre la prueba. O, mejor,
poner el escalador adentro de un `Pipeline`.

### 2. Imputar valores faltantes antes de dividir

Idéntico al anterior. Si imputás con la mediana de todo el dataset, esa mediana incluye las filas
de prueba.

**Cómo se evita:** `SimpleImputer` dentro del `Pipeline`.

### 3. Seleccionar variables mirando todo el dataset

Calculás la correlación de cada variable con la etiqueta usando las 171 filas, te quedás con las 10
mejores, y recién después dividís. Esa selección ya vio las etiquetas de prueba. Es la fuga **más
peligrosa de todas**, porque puede inventar señal donde no hay ninguna —lo demostramos en 4.5—.

**Cómo se evita:** `SelectKBest` dentro del `Pipeline`, o cualquier selección hecha solo con el
conjunto de entrenamiento.

### 4. Ajustar hiperparámetros contra el conjunto de prueba

Probás `max_depth` de 1 a 15, mirás cuál da mejor accuracy **en el conjunto de prueba** y te
quedás con ese. El conjunto de prueba dejó de ser independiente: lo usaste para decidir. El número
que reportás ya no es una estimación honesta.

**Cómo se evita:** tres conjuntos (entrenamiento, validación, prueba), o validación cruzada
anidada: un bucle interno que elige los hiperparámetros y uno externo que evalúa.

### 5. Duplicados repartidos entre las dos partes

Si la misma fila aparece dos veces y una copia cae en entrenamiento y la otra en prueba, el modelo
"acierta" ese caso porque lo memorizó. Con muchos duplicados, la evaluación se infla mucho. Esta es
la razón de fondo del control de duplicados de la Parte 1.

**Cómo se evita:** eliminar duplicados **antes** de dividir. Y si hay grupos naturales —varias
filas del mismo cliente, del mismo paciente, del mismo documento— hay que dividir por grupo, no
por fila: `GroupKFold`.

### 6. Variables que no existirían en el momento de predecir

Una columna que se llena *después* de que ocurre lo que querés predecir. El ejemplo clásico:
predecir si un paciente tiene una enfermedad usando como variable "medicación recetada". La
medicación se receta *porque* ya se diagnosticó. En producción esa columna estaría vacía.

Es el tipo de fuga que ningún `Pipeline` previene: hace falta entender de dónde sale cada dato y
cuándo se registra.

**Cómo se evita:** preguntar, para cada variable, *¿esto lo voy a tener en el momento en que
necesite predecir?*

### 7. Datos con orden temporal partidos al azar

Si las filas tienen fecha y partís al azar, el modelo entrena con datos de marzo y se evalúa con
datos de enero: está prediciendo el pasado con información del futuro. En producción eso nunca
pasa.

**Cómo se evita:** división temporal —entrenar con lo viejo, evaluar con lo nuevo— y
`TimeSeriesSplit` en lugar de `KFold`.

---

**En nuestro dataset**, las formas 5, 6 y 7 no aplican: no hay duplicados (lo verificamos en la
Parte 1), las siete variables se calculan todas juntas en el mismo momento, y no hay orden
temporal. Las que sí nos afectan son la **1** (el escalado), la **3** (la selección de variables) y
la **4** (el ajuste de `k` y de la poda del árbol). Las dos primeras las prevenimos con el
`Pipeline`; sobre la tercera vamos a ser honestos en las limitaciones del punto 12.8.

<a id="sec-4-4"></a>
## 4.4 Demostración 1 — el escalador que mira de más

Empecemos por la más simple: cuánto cambian los parámetros del escalador según se los calcule con
todo el dataset o solo con el entrenamiento.

Esta demostración no muestra un desastre: muestra el **mecanismo**. Las diferencias en los
parámetros son chicas, pero son datos del conjunto de prueba que se están filtrando, y el efecto
crece cuanto más chico es el dataset y más extremos tiene.

In [ ]:
# --- MAL: el escalador ve todo el dataset, incluido el conjunto de prueba ---
escalador_mal = StandardScaler().fit(X)

# --- BIEN: el escalador solo ve el conjunto de entrenamiento ---
escalador_bien = StandardScaler().fit(X_entrena)

comparacion = pd.DataFrame({
    "media_con_todo_MAL": escalador_mal.mean_,
    "media_solo_entrena_BIEN": escalador_bien.mean_,
    "dif_medias": escalador_mal.mean_ - escalador_bien.mean_,
    "desvio_con_todo_MAL": np.sqrt(escalador_mal.var_),
    "desvio_solo_entrena_BIEN": np.sqrt(escalador_bien.var_),
}, index=predictoras)
comparacion["dif_desvios"] = (comparacion["desvio_con_todo_MAL"] -
                              comparacion["desvio_solo_entrena_BIEN"])

print("=" * 88)
print("  LOS PARÁMETROS DEL ESCALADOR CAMBIAN SEGÚN QUÉ DATOS VIO")
print("=" * 88)
display(comparacion)

print("  Esas diferencias son información del conjunto de prueba filtrándose")
print("  hacia la transformación que se aplica al entrenamiento.")
print()
print(f"  Diferencia relativa máxima en las medias : "
      f"{(comparacion['dif_medias'].abs() / comparacion['media_solo_entrena_BIEN'].abs()).max():.2%}")
print(f"  Diferencia relativa máxima en los desvíos: "
      f"{(comparacion['dif_desvios'].abs() / comparacion['desvio_solo_entrena_BIEN']).max():.2%}")
print()
print("  Con Min-Max el efecto es más visible todavía, porque el mínimo y el máximo")
print("  dependen de UN SOLO valor cada uno: si el valor más alto del dataset está")
print("  en el conjunto de prueba, el escalador 'sabe' hasta dónde llegan los datos")
print("  que todavía no vio.")

minmax_todo = MinMaxScaler().fit(X)
minmax_entrena = MinMaxScaler().fit(X_entrena)
comparacion_mm = pd.DataFrame({
    "max_con_todo_MAL": minmax_todo.data_max_,
    "max_solo_entrena_BIEN": minmax_entrena.data_max_,
    "diferencia": minmax_todo.data_max_ - minmax_entrena.data_max_,
}, index=predictoras)
print()
display(comparacion_mm)
cuantas = int((comparacion_mm["diferencia"].abs() > 1e-12).sum())
print(f"  En {cuantas} de {len(predictoras)} variables, el valor máximo del dataset")
print("  está en el conjunto de PRUEBA. Escalando con todo, el modelo lo 'sabe'.")

<a id="sec-4-5"></a>
## 4.5 Demostración 2 — "predecir" ruido puro muy por encima del azar

Esta es la demostración que hay que ver. El escalado filtra un poco; la **selección de variables**
mirando todo el dataset puede inventar señal donde no hay absolutamente nada.

**El experimento.** Generamos 1.000 variables de **ruido aleatorio puro**. Números al azar, sin
ninguna relación con la etiqueta: por construcción, un modelo honesto no debería superar el
azar.

Después hacemos dos cosas:

- **La forma incorrecta.** Calculamos qué 20 de esas 1.000 variables se relacionan mejor con la
  etiqueta, usando **todo** el dataset. Nos quedamos con esas 20 y recién entonces hacemos
  validación cruzada.
- **La forma correcta.** Metemos la selección adentro del `Pipeline`, de modo que se rehaga en cada
  pliegue usando solo los datos de entrenamiento de ese pliegue.

**Por qué la forma incorrecta "funciona".** Con 1.000 variables al azar y 171 filas, algunas van a
parecerse a la etiqueta **por pura casualidad**. Al elegirlas mirando todo el dataset, estamos
eligiendo justamente las que coinciden con las etiquetas de prueba. El modelo después "predice"
esas etiquetas porque las variables fueron seleccionadas para coincidir con ellas.

Es exactamente lo que pasa cuando alguien reporta un resultado espectacular con un dataset de
pocas filas y muchas columnas.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import cross_val_score, StratifiedKFold

generador = np.random.RandomState(SEMILLA)
N_VARIABLES_RUIDO = 1000
K_SELECCIONADAS = 20

# 1.000 variables de ruido puro: ninguna relación con y
X_ruido = pd.DataFrame(
    generador.normal(size=(len(y), N_VARIABLES_RUIDO)),
    columns=[f"ruido_{i}" for i in range(N_VARIABLES_RUIDO)],
    index=y.index,
)

cv_demo = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)

print("=" * 88)
print("  EXPERIMENTO: 1.000 VARIABLES DE RUIDO PURO, CERO SEÑAL REAL")
print("=" * 88)
print(f"  Filas                : {len(y)}")
print(f"  Variables de ruido   : {N_VARIABLES_RUIDO}")
print(f"  Variables a elegir   : {K_SELECCIONADAS}")
print(f"  Accuracy esperable   : ~{max(y.mean(), 1-y.mean()):.2%} (la línea de base)")
print()

# ---------- FORMA INCORRECTA: seleccionar mirando TODO el dataset ----------
selector_mal = SelectKBest(f_classif, k=K_SELECCIONADAS).fit(X_ruido, y)
X_ruido_elegido = X_ruido.loc[:, selector_mal.get_support()]

modelo_simple = LogisticRegression(max_iter=5000, random_state=SEMILLA)
score_mal = cross_val_score(modelo_simple, X_ruido_elegido, y,
                            cv=cv_demo, scoring="accuracy")

# ---------- FORMA CORRECTA: la selección adentro del Pipeline ----------
pipeline_bien = Pipeline([
    ("seleccion", SelectKBest(f_classif, k=K_SELECCIONADAS)),
    ("modelo", LogisticRegression(max_iter=5000, random_state=SEMILLA)),
])
score_bien = cross_val_score(pipeline_bien, X_ruido, y, cv=cv_demo, scoring="accuracy")

print("  RESULTADOS")
print("  " + "-" * 84)
print(f"  MAL  (selección sobre todo el dataset, después validación cruzada)")
print(f"       accuracy por pliegue : {np.round(score_mal, 4)}")
print(f"       MEDIA                : {score_mal.mean():.4f}   <-- con RUIDO PURO")
print()
print(f"  BIEN (selección adentro del Pipeline, rehecha en cada pliegue)")
print(f"       accuracy por pliegue : {np.round(score_bien, 4)}")
print(f"       MEDIA                : {score_bien.mean():.4f}   <-- lo honesto")
print()
print(f"  DIFERENCIA: {score_mal.mean() - score_bien.mean():+.4f} de accuracy")
print(f"              inventados de la nada.")

fig, eje = plt.subplots(figsize=(9, 4.4))
barras = eje.bar(["MAL\n(selección sobre todo)", "BIEN\n(selección en el Pipeline)"],
                 [score_mal.mean(), score_bien.mean()],
                 color=["#d62728", "#2ca02c"], edgecolor="black", linewidth=0.8,
                 yerr=[score_mal.std(), score_bien.std()], capsize=6)
eje.axhline(max(y.mean(), 1 - y.mean()), color="black", linestyle="--", linewidth=1.5,
            label=f"línea de base = {max(y.mean(), 1-y.mean()):.3f}")
for barra, valor in zip(barras, [score_mal.mean(), score_bien.mean()]):
    eje.text(barra.get_x() + barra.get_width()/2, valor + 0.03, f"{valor:.4f}",
             ha="center", fontweight="bold", fontsize=12)
eje.set_ylabel("accuracy con validación cruzada")
eje.set_title("Clasificar ruido puro: el efecto de la fuga de información")
eje.set_ylim(0, 1.05)
eje.legend()
plt.tight_layout()
plt.show()

print()
print("  LO QUE HAY QUE ENTENDER:")
print("  No hay ninguna señal en esos datos. Cero. Son números al azar.")
print("  El primer resultado es enteramente artificial, y sin embargo no hay")
print("  ningún error, ningún aviso y ninguna forma de notarlo mirando el código")
print("  del modelo. La fuga ocurrió TRES LÍNEAS ANTES de entrenar.")

<a id="sec-4-6"></a>
## 4.6 El Pipeline como antídoto

La conclusión práctica de todo esto: **no hay que confiar en acordarse**. La prevención tiene que
ser estructural.

Un **`Pipeline`** de scikit-learn encadena una serie de pasos —transformaciones y, al final, un
modelo— y los trata como un único objeto. Lo importante es cómo se comporta:

- Cuando se le pide `fit`, aplica `fit_transform` a cada transformación **con los datos que
  recibe**.
- Cuando se le pide `predict`, aplica solamente `transform`, con los parámetros que aprendió en el
  `fit`.

Y acá está la clave: cuando un `Pipeline` se pasa a una función de validación cruzada, **ese ciclo
se repite dentro de cada pliegue**. El escalador se reajusta cinco veces, cada una con el 80% de
entrenamiento de ese pliegue, y se aplica al 20% restante sin haberlo visto nunca.

La fuga deja de ser un error posible y pasa a ser imposible.

```python
# En lugar de esto, que es frágil:
X_escalado = StandardScaler().fit_transform(X)      # <-- vio todo
modelo.fit(X_escalado, y)

# Esto, que es a prueba de descuidos:
pipeline = Pipeline([
    ("escalador", StandardScaler()),
    ("modelo",    LogisticRegression()),
])
cross_val_score(pipeline, X, y, cv=cv)              # <-- reajusta en cada pliegue
```

Todo el resto de este cuaderno usa pipelines. La función `construir_pipeline()` del punto 7.2 es
la que los arma, y ningún modelo se evalúa fuera de uno.

La celda de abajo muestra el mecanismo: los parámetros que aprende el escalador cambian en cada
pliegue, y son siempre los del 80% de entrenamiento.

In [ ]:
pipeline_ejemplo = Pipeline([
    ("escalador", StandardScaler()),
    ("modelo", LogisticRegression(max_iter=5000, random_state=SEMILLA)),
])

print("=" * 88)
print("  EL ESCALADOR SE REAJUSTA EN CADA PLIEGUE")
print("=" * 88)
print(f"  {'Pliegue':>8} {'Casos de entrenamiento':>24} "
      f"{'media de ' + predictoras[0]:>26} {'desvío':>12}")

for i, (idx_tr, idx_te) in enumerate(cv_demo.split(X, y), start=1):
    pipeline_ejemplo.fit(X.iloc[idx_tr], y.iloc[idx_tr])
    escalador_pliegue = pipeline_ejemplo.named_steps["escalador"]
    print(f"  {i:>8} {len(idx_tr):>24} {escalador_pliegue.mean_[0]:>26.6f} "
          f"{np.sqrt(escalador_pliegue.var_[0]):>12.6f}")

escalador_todo = StandardScaler().fit(X)
print(f"  {'(todo)':>8} {len(X):>24} {escalador_todo.mean_[0]:>26.6f} "
      f"{np.sqrt(escalador_todo.var_[0]):>12.6f}   <-- lo que NO hay que usar")
print()
print("  Cada fila usa parámetros distintos, calculados solo con su 80% de")
print("  entrenamiento. El 20% de prueba de cada pliegue nunca participó del ajuste.")
print("  Eso es lo que hace que la estimación sea honesta.")

<a id="sec-4-7"></a>
## 4.7 Lista de control

Para aplicar antes de dar por buena cualquier evaluación:

| # | Pregunta | Si la respuesta es "no" |
|---|---|---|
| 1 | ¿El escalado está adentro de un `Pipeline`? | Moverlo adentro |
| 2 | ¿La imputación está adentro del `Pipeline`? | Moverla adentro |
| 3 | ¿La selección de variables está adentro del `Pipeline`? | Moverla adentro |
| 4 | ¿Los duplicados se eliminaron **antes** de dividir? | Eliminarlos y volver a dividir |
| 5 | ¿Hay grupos naturales (cliente, paciente, documento) repartidos entre las partes? | Usar `GroupKFold` |
| 6 | ¿Cada variable va a existir en el momento de predecir? | Sacar las que no |
| 7 | ¿Los datos tienen orden temporal? | Dividir por tiempo, `TimeSeriesSplit` |
| 8 | ¿Los hiperparámetros se eligieron mirando el conjunto de prueba? | Validación cruzada anidada |
| 9 | ¿El resultado es sospechosamente bueno? | Buscar la fuga antes de festejar |

La número 9 no es una broma. Un resultado muy por encima de lo esperable es, estadísticamente, más
veces una fuga que un hallazgo.

---
# 5. Overfitting y underfitting

Este es, junto con la fuga de información, el concepto que más decide si un modelo sirve o no.
La diferencia entre los dos es importante: la fuga es un **error de procedimiento** —se comete o
no se comete, y se evita con un `Pipeline`—, mientras que el ajuste es un **equilibrio** que hay
que buscar de nuevo en cada problema. No existe el modelo que esté bien ajustado por defecto.

Este punto es todo conceptual hasta el 5.7. Recién ahí aparece el código, y lo que hace es mostrar
en nuestros datos lo que acá se explica en general.

<a id="sec-5-1"></a>
## 5.1 Qué es cada uno

Todo modelo tiene que encontrar un equilibrio entre dos errores opuestos: aprender de menos y
aprender de más.

Conviene tener presente cuál es el objetivo real. No queremos un modelo que acierte los casos que
ya tenemos —esos ya vienen con la respuesta escrita al lado—. Queremos uno que acierte **casos que
todavía no existen**. A eso se le llama **generalizar**, y es la única medida de éxito que
importa.

### Overfitting (sobreajuste)

El modelo aprende **demasiado bien** los datos de entrenamiento. En lugar de captar los patrones
generales, memoriza los detalles: el ruido, los casos particulares, los errores de carga, las
casualidades de esa muestra concreta.

A simple vista parece un éxito: las predicciones sobre los datos conocidos son casi perfectas. El
problema aparece con datos nuevos, donde el modelo falla, porque los detalles que memorizó no se
repiten.

**La analogía:** un estudiante que se aprendió de memoria cada pregunta del simulacro, con su
respuesta, sin entender el tema. Saca 10 en el simulacro. En el examen real, donde cambia el
enunciado, no sabe qué hacer.

**Un ejemplo concreto.** Supongamos que queremos predecir si un alumno aprueba a partir de las
horas que estudió. La relación real es simple: más horas, más probabilidad de aprobar. Pero en
nuestros 30 casos hay dos alumnos que estudiaron 12 horas y desaprobaron —uno estaba enfermo, el
otro se confundió de tema—. Un modelo suficientemente flexible va a aprender la regla *"12 horas
exactas ⇒ desaprueba"*, porque eso le hace acertar esos dos casos.

Con datos nuevos, esa regla es puro daño: no describe nada del mundo, describe un accidente de la
muestra. **Eso es memorizar ruido**, y es la definición operativa del sobreajuste: el modelo trata
como señal algo que era casualidad.

**Cuándo aparece:** cuando el modelo es demasiado complejo para la cantidad de datos disponibles.
Un árbol sin límite de profundidad, un KNN con *k*=1, un bosque enorme sobre 50 filas, una red
neuronal con más parámetros que ejemplos. La palabra clave es **para la cantidad de datos**: el
mismo árbol que sobreajusta con 171 filas puede estar perfectamente ajustado con 100.000.

**Por qué es peligroso:** genera una falsa sensación de éxito. Es el error que lleva a poner en
producción modelos que funcionaban perfecto en las pruebas. Y como el número que uno mira es
bueno, nadie lo investiga —es el mismo problema que la fuga de información del punto 4—.

### Underfitting (subajuste)

El caso opuesto. El modelo **no logra captar los patrones** ni siquiera en los datos de
entrenamiento. Se equivoca en todos lados por igual.

**La analogía:** un estudiante que solo leyó el título del tema. No resuelve los ejercicios del
examen, pero tampoco los de la clase.

**El mismo ejemplo de antes.** Si en lugar de un modelo flexible usamos uno que solo puede
responder siempre lo mismo —*"todos aprueban"*—, acertamos el 60 % y nunca aprendemos que las
horas de estudio importan. No memorizamos ruido, pero tampoco capturamos la señal. Es el error
simétrico.

**Cuándo aparece:** cuando el modelo es demasiado simple para el problema —una recta para separar
datos que forman círculos concéntricos—, cuando las variables no representan bien el fenómeno, o
cuando se entrenó con muy pocas iteraciones.

**Por qué es menos peligroso:** porque se nota. Un modelo que rinde mal en todos lados no engaña a
nadie: el primer número que uno mira ya avisa. El sobreajuste, en cambio, se disfraza de éxito.

Que sea menos peligroso no quiere decir que sea aceptable. Un modelo subajustado es un modelo que
no sirve, solo que además lo dice de entrada.

### El equilibrio

Entre los dos extremos hay una zona donde el modelo aprende lo suficiente para generalizar sin
memorizar. Encontrarla es uno de los objetivos centrales de todo el proceso.

```
       error
         |
         |\                                    /
         | \                                  /
         |  \        error de validación     /
         |   \                              /
         |    \___                      ___/
         |        \___              ___/
         |            \____________/          <-- la zona buena
         |             ________
         |        ____/                       error de entrenamiento
         |   ____/
         |__/________________________________________
              SUBAJUSTE        justo        SOBREAJUSTE
                        complejidad del modelo
```

Hay tres cosas para leer en ese dibujo:

1. **El error de entrenamiento baja siempre** que se agrega complejidad. Nunca sube. Por eso, por
   sí solo, no informa nada: un modelo con error de entrenamiento cero puede ser excelente o puede
   ser una tabla de búsqueda.
2. **El error de validación baja y después sube.** Ese cambio de dirección es el único indicador
   confiable, y el punto donde ocurre es el objetivo.
3. **La distancia vertical entre las dos curvas** —la brecha— se ensancha hacia la derecha. Esa
   distancia es lo que vamos a medir.

<a id="sec-5-2"></a>
## 5.2 El compromiso sesgo–varianza

Sobreajuste y subajuste son la cara visible de algo más general, que conviene conocer porque
explica *por qué* no se pueden evitar los dos a la vez.

El error que un modelo comete con datos nuevos se descompone en tres partes:

```
error esperado  =  sesgo²  +  varianza  +  error irreducible
```

**Sesgo (*bias*)** — cuánto se equivoca el modelo **sistemáticamente**, por ser demasiado simple
para la forma real del problema. Un modelo con mucho sesgo se equivoca siempre para el mismo lado,
sin importar con qué datos lo entrenemos. Es el subajuste visto desde adentro.

**Varianza** — cuánto **cambia el modelo** si lo entrenamos con otra muestra del mismo fenómeno.
Un modelo con mucha varianza es inestable: entrenado con estas 171 filas da una cosa, entrenado
con otras 171 igual de válidas da algo bastante distinto. Es el sobreajuste visto desde adentro.

**Error irreducible** — el ruido propio del fenómeno. Dos casos con exactamente las mismas
variables de entrada pueden tener etiquetas distintas, sencillamente porque las variables no
contienen toda la información que determina el resultado. Ningún modelo puede bajar de ahí. Si el
error irreducible de un problema es del 15 %, un modelo que reporta 98 % de accuracy no es
brillante: es sospechoso.

### La analogía del tiro al blanco

Es la que mejor fija la idea. Cada entrenamiento con una muestra distinta es un tiro:

|  | **Poca varianza** | **Mucha varianza** |
|---|---|---|
| **Poco sesgo** | Los tiros agrupados en el centro. Es el objetivo | Los tiros repartidos alrededor del centro: en promedio dan bien, pero cualquiera de ellos por separado puede estar lejos |
| **Mucho sesgo** | Los tiros agrupados, pero lejos del centro. Consistentemente equivocado | Los tiros repartidos **y** corridos. El peor de los casos |

Notá que "en promedio da bien" no consuela: en la práctica entrenamos el modelo **una sola vez**,
con los datos que tenemos. Un modelo de alta varianza es una apuesta.

### Por qué es un compromiso y no un problema con solución

Bajar el sesgo —darle más capacidad al modelo— **sube** la varianza. Bajar la varianza
—limitarlo— **sube** el sesgo. Moviendo la complejidad no se pueden minimizar los dos a la vez:
por eso se habla de un *compromiso* (*trade-off*).

Lo que sí se puede hacer es bajar los dos cambiando **otra cosa**:

| Qué cambiar | Efecto | Por qué |
|---|---|---|
| **Más datos** | Baja la varianza, no toca el sesgo | Con más ejemplos, las casualidades pesan menos y el modelo entrenado con una muestra se parece más al entrenado con otra |
| **Mejores variables** | Baja el sesgo, no toca la varianza | Si la información está mejor representada, un modelo simple alcanza. Es lo que hace valioso al feature engineering |
| **Conjuntos (*ensembles*)** | Bajan la varianza | Promediar muchos modelos de alta varianza cancela buena parte de sus errores individuales. Es exactamente la razón de ser del Random Forest del punto 11 |
| **Regularización** | Baja la varianza a cambio de algo de sesgo | Es un movimiento sobre el compromiso, no una salida de él, pero suele ser un buen negocio |

### Dónde se para cada uno de nuestros cuatro modelos

| Modelo | Sesgo | Varianza | Comentario |
|---|---|---|---|
| Regresión Logística | Alto | Baja | Solo puede dibujar un plano; la regularización L2 le baja todavía más la varianza |
| KNN con *k* grande | Alto | Baja | Promedia sobre muchos vecinos: se vuelve casi una constante |
| KNN con *k* = 1 | Muy bajo | Muy alta | Cada predicción depende de un único caso |
| Árbol sin podar | Muy bajo | Muy alta | Puede separar cada caso en su propia hoja |
| Árbol podado | Medio | Media | La poda es exactamente el mando de este compromiso |
| Random Forest | Bajo | Baja | Conserva el bajo sesgo del árbol y le cancela la varianza promediando |

Esa última fila explica por qué el Random Forest suele ganar: es el único de los cuatro que
consigue mejorar en las dos columnas a la vez, y lo logra porque no cambia la complejidad del
modelo individual sino la cantidad de modelos. El precio que paga está en otro lado: deja de ser
legible.

<a id="sec-5-3"></a>
## 5.3 Las señales: tabla de diagnóstico

La forma básica de diagnosticar es comparar el rendimiento del modelo **en los datos con los que
entrenó** contra el de **datos que no vio**. Los cuatro escenarios posibles:

| Rendimiento en entrenamiento | Rendimiento en validación | Diagnóstico | Qué está pasando | Qué hacer |
|---|---|---|---|---|
| Alto (0,99) | Bajo (0,72) | **Sobreajuste** | Memorizó | Simplificar el modelo o conseguir más datos |
| Bajo (0,65) | Bajo (0,63) | **Subajuste** | No aprendió | Dar más capacidad o mejores variables |
| Alto (0,92) | Alto (0,89) | **Correcto** | Generaliza | Nada: es el objetivo |
| Bajo (0,70) | Alto (0,85) | **Raro** | Casi siempre indica un error en el código, o una regularización que solo se aplica al entrenar | Revisar el código antes que el modelo |

Sobre la cuarta fila, que desconcierta cuando aparece: rendir **mejor** en datos que no se vieron
que en los de entrenamiento no es una buena noticia, es un síntoma. Las causas habituales son que
el conjunto de validación quedó más fácil por azar —muy común con pocos datos—, que hay filas
repetidas entre las dos partes, o que el modelo aplica una penalización o un *dropout* durante el
entrenamiento que después desactiva.

Un ejemplo ilustrativo, con tres árboles de complejidad distinta (los números son inventados, para
mostrar el patrón; los reales de nuestros datos salen en 5.7):

| Modelo | Accuracy entrenamiento | Accuracy prueba | Brecha | Diagnóstico |
|---|---|---|---|---|
| Árbol sin límite de profundidad | 1,000 | 0,800 | **0,200** | Sobreajuste claro |
| Árbol de profundidad 4 | 0,920 | 0,880 | 0,040 | Razonable |
| Árbol de profundidad 1 | 0,760 | 0,740 | 0,020 | Brecha chica, pero rendimiento bajo: subajuste |

Fijate en las dos últimas filas: **una brecha chica no alcanza**. El árbol de profundidad 1 tiene
la brecha más chica de los tres y es el peor modelo. Hay que mirar las dos cosas: la brecha *y* el
nivel.

### Otras señales, menos obvias

**El modelo tiene muchos más parámetros que datos.** Un árbol con 150 hojas sobre 171 filas está
prácticamente guardando una fila por hoja. La regla de bolsillo: si la cantidad de decisiones que
el modelo puede tomar se acerca a la cantidad de ejemplos que vio, no está aprendiendo, está
indexando.

**Los resultados cambian mucho al cambiar la semilla.** Un modelo que generaliza da resultados
parecidos con cualquier partición. Uno que memoriza depende de qué le haya tocado memorizar. Es
exactamente lo que medimos en 3.3.

**El desvío entre pliegues es grande.** Si el F1 va de 0,70 a 0,95 según el pliegue, el modelo no
está capturando algo estable: está reaccionando a los datos puntuales de cada partición.

**El modelo usa variables que no tienen sentido.** Si al mirar la importancia de las variables
aparece arriba una que no debería importar, puede estar explotando una casualidad de la muestra.
Este es el único indicio que exige conocer el dominio, y por eso es tan valioso: ninguna métrica
lo detecta.

**Las probabilidades son extremas.** Un modelo que predice 0,999 y 0,001 y casi nada en el medio
suele estar demasiado seguro de sí mismo. Las métricas de calibración —*log loss* y *Brier*, que
calculamos en el punto 7.3— son sensibles a esto, mientras que el accuracy no lo nota.

### Tres cosas que parecen sobreajuste y no lo son

| Lo que se ve | Por qué no es sobreajuste |
|---|---|
| El modelo acierta el 100 % en entrenamiento | Por sí solo no dice nada. Si también acierta en validación, el problema simplemente era fácil |
| Hay una brecha de 0,04 | Con un conjunto de validación chico, una brecha de ese tamaño puede ser enteramente ruido de medición. Hay que compararla contra el desvío entre pliegues |
| El rendimiento cae al pasar a producción | Puede ser sobreajuste, pero también *data drift*: los datos nuevos ya no se parecen a los viejos. Son problemas distintos con soluciones distintas |

<a id="sec-5-4"></a>
## 5.4 Las cuatro formas de detectarlo

### Forma 1 — La brecha entre entrenamiento y validación

La más directa. Se entrena el modelo, se lo evalúa **sobre los mismos datos de entrenamiento** y
sobre datos que no vio, y se comparan los dos números.

*Cómo se lee:* la resta. Cuanto más grande, más memoriza.

*Qué la hace útil:* es inmediata y no requiere nada especial. Debería ser el primer número que uno
mira de cualquier modelo.

*Su límite:* con una sola partición, la medición es ruidosa. Como vimos en 3.3, el resultado sobre
34 casos de prueba se mueve varios puntos según el azar. Una brecha de 0,05 puede ser sobreajuste
o puede ser la partición que le tocó.

### Forma 2 — La curva de validación

Se repite el entrenamiento variando **un hiperparámetro que controla la complejidad** —la
profundidad del árbol, el *k* de KNN, la fuerza de la regularización— y se grafican las dos
curvas: rendimiento en entrenamiento y en validación, contra complejidad.

*Cómo se lee:*

- La curva de entrenamiento **sube siempre**: más complejidad, más capacidad de memorizar.
- La de validación sube hasta un punto y después **baja**.
- El punto donde las dos se separan marca el comienzo del sobreajuste.
- El máximo de la curva de validación marca la complejidad justa.
- Si las dos curvas están bajas y juntas en toda la izquierda del gráfico, eso es subajuste.

*Qué la hace útil:* no dice solo "hay sobreajuste", dice **dónde empieza** y **cuánta complejidad
conviene**. Es una herramienta de diagnóstico y de ajuste a la vez.

*Su límite:* mira un hiperparámetro por vez. Si la complejidad está repartida entre varios —la
profundidad *y* el mínimo de casos por hoja, por ejemplo—, la curva de uno solo puede engañar. Ahí
conviene una búsqueda en grilla, que es lo que usamos con el árbol en el punto 10.

### Forma 3 — El desvío entre pliegues de la validación cruzada

En lugar de una partición, se usan cinco o diez, y se mira no solo la media sino la **dispersión**
de los resultados.

*Cómo se lee:* un desvío chico significa que el modelo rinde parecido sin importar qué datos le
toquen —está capturando algo estable—. Un desvío grande significa que el rendimiento depende de la
partición, que es otra forma de decir que el modelo está reaccionando a particularidades de los
datos y no a un patrón general.

*Qué la hace útil:* es la única de las cuatro que da una **medida del error de la medición**. Sin
ella no se puede saber si la diferencia entre dos modelos es real. Es la que vamos a usar en toda
la comparativa del punto 12.

*Su límite:* no distingue entre un modelo inestable y un dataset chico. Con 171 filas, cierto
desvío es inevitable venga el modelo que venga.

### Forma 4 — La curva de aprendizaje

Grafica el rendimiento contra la **cantidad de datos** en lugar de contra la complejidad. Se
entrena el mismo modelo con el 20 %, el 30 %, el 40 %… de los datos disponibles, y en cada tamaño
se miden las dos curvas.

Responde una pregunta distinta de las anteriores, y muy práctica: ***¿conviene conseguir más
datos?***

*Cómo se lee:*

- Si las dos curvas **convergen y se aplanan** en un valor bajo, más datos no van a ayudar: el
  problema es el modelo. Hay **sesgo alto** y lo que hace falta es un modelo más expresivo o
  mejores variables.
- Si las dos curvas **siguen separadas** y la de validación todavía viene subiendo en el extremo
  derecho, más datos sí ayudarían. Hay **varianza alta**.
- Si la curva de entrenamiento arranca en 1,0 y se mantiene ahí, el modelo memoriza desde el
  primer momento.

*Qué la hace útil:* es la única que informa sobre una decisión que está **fuera del modelo** —ir a
buscar más datos, que suele ser lo más caro del proyecto— y la única que distingue con claridad un
problema de sesgo de uno de varianza.

*Su límite:* es la más cara de calcular, porque entrena el modelo muchas veces sobre muchos
tamaños.

### Las cuatro, resumidas

| Forma | Contra qué grafica | Qué pregunta responde | Costo |
|---|---|---|---|
| Brecha | nada, es un número | ¿Hay sobreajuste? | Nulo |
| Curva de validación | complejidad | ¿Cuánta complejidad conviene? | Medio |
| Desvío entre pliegues | nada, es un número | ¿Cuánto vale esta medición? | Bajo |
| Curva de aprendizaje | cantidad de datos | ¿Conviene conseguir más datos? | Alto |

Las cuatro se usan en este cuaderno: la 1 y la 2 en los puntos 5.7 y 5.8, la 4 en el 5.9, y la 3
en todo el resto, desde el punto 6 hasta el final.

<a id="sec-5-5"></a>
## 5.5 Cómo se mide: la brecha y sus umbrales

La **brecha** es la medida más usada:

```
brecha = rendimiento en entrenamiento - rendimiento en validación
```

Cuanto más grande, más memoriza el modelo. Pero *cuánto* es mucho no tiene una respuesta
universal: depende del tamaño del dataset, del problema y de qué métrica se use.

Algunas referencias prácticas, que son convenciones y no leyes:

| Brecha (en F1 o accuracy) | Lectura habitual |
|---|---|
| Menor a 0,02 | Sin señales de sobreajuste |
| 0,02 a 0,05 | Normal, sobre todo en datasets chicos |
| 0,05 a 0,10 | Aviso: conviene no aumentar más la complejidad |
| Mayor a 0,10 | Sobreajuste que hay que atender |
| Mayor a 0,20 | El modelo está memorizando |

Y tres cosas que la brecha sola no dice:

**El nivel.** Una brecha de 0,01 con un F1 de 0,60 en validación no es un buen modelo: es un
modelo que aprende poco y de manera consistente. Por eso hay que mirar **brecha y nivel juntos**.

**El error de medición.** Una brecha de 0,04 medida sobre 34 casos no es distinguible de una de
0,00. Con pocos datos, la brecha hay que estimarla con validación cruzada y compararla contra el
desvío entre pliegues. Si la brecha es menor que el desvío, no está demostrada.

**Con qué métrica se calculó.** La brecha en accuracy y la brecha en F1 macro del mismo modelo
pueden ser bastante distintas, sobre todo si las clases están desbalanceadas. Hay que decir
siempre con cuál se midió. En este cuaderno usamos **F1 macro**, porque trata a las dos clases por
igual.

En este cuaderno los umbrales están declarados en la configuración del punto 0.4:

```python
BRECHA_SOBREAJUSTE = 0.10   # por encima: se avisa sobreajuste
DESVIO_INESTABLE   = 0.08   # por encima: se avisa inestabilidad
SCORE_SUBAJUSTE    = 0.70   # por debajo: se avisa subajuste
```

Y la función `diagnosticar_ajuste()` del punto 7.3 los aplica automáticamente a cada modelo. Están
arriba de todo, con nombre, justamente para que se puedan discutir y cambiar en lugar de quedar
escondidos en el medio del código. Si te parecen mal calibrados para este problema, cambiarlos es
una línea y todo el cuaderno se recalcula.

<a id="sec-5-6"></a>
## 5.6 Qué hacer con cada uno

Los dos problemas tienen remedios **opuestos**. Por eso diagnosticar bien es la mitad del trabajo:
aplicar el remedio equivocado empeora las cosas. Bajarle la profundidad a un árbol que ya estaba
subajustado lo empeora dos veces.

### Contra el subajuste: darle más capacidad al modelo

| Remedio | En concreto | Cuándo es el correcto |
|---|---|---|
| Usar un modelo más expresivo | Pasar de una regresión logística a un árbol o un bosque | Cuando la frontera real no es lineal |
| Aumentar la complejidad | Subir `max_depth`, bajar `k` en KNN, subir `C` (menos regularización) | Cuando el modelo elegido está frenado por sus hiperparámetros |
| Mejorar las variables | Feature engineering: crear variables que representen mejor el fenómeno | Cuando hay información latente que el modelo no puede deducir solo |
| Entrenar más | Más iteraciones, más árboles | Cuando el entrenamiento se cortó antes de converger |
| Revisar la limpieza | A veces el problema es que se eliminó información valiosa al "limpiar" | Cuando el rendimiento empeoró justo después de limpiar |

### Contra el sobreajuste: limitar al modelo o darle más datos

| Remedio | En concreto | Cuándo es el correcto |
|---|---|---|
| Simplificar el modelo | Bajar `max_depth`, subir `min_samples_leaf`, subir `k` en KNN | Casi siempre, y es lo primero que conviene probar |
| Regularizar | Penalizar los coeficientes grandes: bajar `C` en la regresión logística | En modelos lineales y en redes |
| Conseguir más datos | El remedio más eficaz de todos, y casi siempre el menos disponible | Cuando la curva de aprendizaje (5.4, forma 4) muestra que todavía sube |
| Reducir variables | Menos columnas, menos oportunidades de encontrar casualidades | Cuando hay muchas más variables que filas |
| Usar conjuntos (*ensembles*) | Un bosque promedia muchos árboles y cancela buena parte del sobreajuste individual | Cuando el modelo base es bueno pero inestable |
| Parada temprana | En modelos iterativos, cortar cuando el error de validación deja de bajar | En *boosting* y redes neuronales |
| Validar bien | La validación cruzada no evita el sobreajuste, pero lo hace visible | Siempre |

### Dónde se sobreajusta cada uno de nuestros cuatro modelos

Esta tabla es el puente entre este punto y los puntos 8 a 11: dice, para cada modelo, cuál es el
mando que controla su complejidad y hacia qué lado hay que moverlo.

| Modelo | El mando | Sobreajusta cuando | Subajusta cuando | Valor por defecto en scikit-learn |
|---|---|---|---|---|
| **Regresión Logística** | `C` (inverso de la regularización) | `C` muy alto, o hay más variables que filas | `C` muy bajo | `C=1.0`, regularización L2 moderada. Sobreajusta poco de fábrica |
| **KNN** | `n_neighbors` (*k*) | *k* = 1: cada predicción depende de un solo vecino | *k* comparable al tamaño del dataset | `k=5`. Es el modelo donde el mando se ve más claro |
| **Árbol de Decisión** | `max_depth` y `min_samples_leaf` | Sin límite de profundidad: puede aislar cada caso en su hoja | Profundidad 1 o 2 | **Sin límites**: sobreajusta por defecto. Hay que podarlo a mano |
| **Random Forest** | `n_estimators`, `max_features`, `max_depth` | Rara vez, y nunca por agregar árboles | Profundidad muy limitada | Sin límite de profundidad, pero el promedio lo compensa |

Dos observaciones que vale la pena retener:

- **El árbol es el único que sobreajusta por defecto.** Un `DecisionTreeClassifier()` recién
  sacado de la caja crece hasta separar cada caso. Por eso en el punto 10 buscamos la poda con una
  grilla, mientras que a la logística no le tocamos nada.
- **En el Random Forest, agregar árboles nunca empeora.** Más árboles es siempre mejor o igual, y
  solo cuesta tiempo. Es la excepción a la regla de que más complejidad trae sobreajuste, y la
  razón es que los árboles no se suman: se promedian.

### Un caso particular: la regularización

Vale detenerse porque es el remedio más usado y el menos intuitivo. Regularizar es agregarle a la
función que el modelo minimiza un término que **penaliza la complejidad**. El modelo deja de
buscar solamente el mejor ajuste y pasa a buscar un equilibrio entre ajustar bien y mantenerse
simple.

En términos del punto 5.2: la regularización **compra varianza barata a cambio de sesgo caro**.
Empeora un poco el ajuste sobre los datos que ya tenemos, a cambio de que el modelo no cambie
tanto con datos nuevos. En datasets chicos ese intercambio casi siempre conviene.

Las dos formas habituales:

| Penalización | Qué penaliza | Efecto característico |
|---|---|---|
| **L2** (*Ridge*) | La suma de los coeficientes **al cuadrado** | Achica todos los coeficientes de manera pareja; ninguno llega a cero |
| **L1** (*Lasso*) | La suma de los **valores absolutos** | Lleva algunos coeficientes exactamente a cero: selecciona variables sola |

En la regresión logística se controla con el parámetro `C`, que es el **inverso** de la fuerza de
la penalización: valores chicos regularizan **más** —coeficientes más chicos, modelo más simple— y
valores grandes regularizan menos. Es una convención que confunde a todo el mundo la primera vez.

Por defecto scikit-learn usa `C=1.0` con penalización L2, que es una regularización moderada y
**siempre activa**. Es, entre otras cosas, lo que hace que la regresión logística funcione bien en
datasets chicos sin necesidad de ajustar nada.

<a id="sec-5-7"></a>
## 5.7 En código: tres niveles de complejidad

Con los conceptos claros, vamos a verlos en nuestros datos.

Tomamos el mismo modelo —un árbol de decisión— en tres niveles de complejidad: uno **sin límite de
profundidad**, que puede memorizar cada caso; uno de **profundidad 3**; y uno de **profundidad 1**,
que solo puede hacer una pregunta. Los tres se entrenan con los mismos datos y se evalúan sobre el
entrenamiento y sobre la prueba.

Esta es la **forma 1** de 5.4: la brecha. Lo que esperamos ver, si todo lo anterior es cierto:

- El árbol sin límite debería acertar **todo** en entrenamiento y bastante menos en prueba.
- El de profundidad 3 debería quedar en el medio, con una brecha chica.
- El de profundidad 1 debería tener una brecha mínima pero un nivel bajo —subajuste—.

La columna `hojas` de la tabla es la que hace visible el mecanismo: dice en cuántos grupos terminó
partiendo el árbol los datos de entrenamiento.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

ejemplos = {
    "Árbol sin límite (complejo)":     DecisionTreeClassifier(random_state=SEMILLA),
    "Árbol de profundidad 3 (medio)":  DecisionTreeClassifier(max_depth=3, random_state=SEMILLA),
    "Árbol de profundidad 1 (simple)": DecisionTreeClassifier(max_depth=1, random_state=SEMILLA),
}

comparacion_ajuste = []
for nombre, estimador in ejemplos.items():
    estimador.fit(X_entrena, y_entrena)
    acc_tr = accuracy_score(y_entrena, estimador.predict(X_entrena))
    acc_te = accuracy_score(y_prueba, estimador.predict(X_prueba))
    comparacion_ajuste.append({
        "modelo": nombre,
        "hojas": estimador.get_n_leaves(),
        "profundidad": estimador.get_depth(),
        "accuracy_entrenamiento": acc_tr,
        "accuracy_prueba": acc_te,
        "brecha": acc_tr - acc_te,
    })

comparacion_ajuste = pd.DataFrame(comparacion_ajuste)

print("=" * 96)
print("  TRES NIVELES DE COMPLEJIDAD")
print("=" * 96)
display(comparacion_ajuste)

for _, fila in comparacion_ajuste.iterrows():
    partes = []
    if fila["brecha"] > BRECHA_SOBREAJUSTE:
        partes.append(f"brecha {fila['brecha']:.3f}: SOBREAJUSTE")
    elif fila["brecha"] > BRECHA_SOBREAJUSTE / 2:
        partes.append(f"brecha {fila['brecha']:.3f}: sobreajuste leve")
    else:
        partes.append(f"brecha {fila['brecha']:.3f}: aceptable")
    if fila["accuracy_prueba"] < SCORE_SUBAJUSTE:
        partes.append(f"nivel {fila['accuracy_prueba']:.3f}: bajo, señal de SUBAJUSTE")
    else:
        partes.append(f"nivel {fila['accuracy_prueba']:.3f}: aceptable")
    print(f"  {fila['modelo']:<34} -> {'  |  '.join(partes)}")

print()
casos_por_hoja = len(X_entrena) / comparacion_ajuste.loc[0, "hojas"]
print(f"  Notá la columna 'hojas': el árbol sin límite llegó a "
      f"{comparacion_ajuste.loc[0, 'hojas']} hojas")
print(f"  para {len(X_entrena)} casos de entrenamiento, es decir unos "
      f"{casos_por_hoja:.1f} casos por hoja,")
print("  y con eso alcanza para acertarlos TODOS (accuracy de entrenamiento = 1,000).")
print("  Eso es memorizar, no aprender.")
print()
print("  ADVERTENCIA IMPORTANTE sobre estos números:")
print(f"  el accuracy de prueba está medido sobre {len(X_prueba)} casos. Cada acierto")
print(f"  o error vale {100/len(X_prueba):.1f} puntos, así que las tres brechas de esta tabla")
print("  tienen un margen de error grande. Sirven para ver el PATRÓN —más")
print("  complejidad, más brecha— pero no para decidir nada con precisión.")
print("  En el punto 6 vamos a volver a medirlo como corresponde.")

fig, eje = plt.subplots(figsize=(10, 4.4))
posiciones = np.arange(len(comparacion_ajuste))
ancho = 0.38
eje.bar(posiciones - ancho/2, comparacion_ajuste["accuracy_entrenamiento"], ancho,
        label="entrenamiento", color="#d62728", alpha=0.85, edgecolor="black", linewidth=0.6)
eje.bar(posiciones + ancho/2, comparacion_ajuste["accuracy_prueba"], ancho,
        label="prueba", color="#1f77b4", alpha=0.9, edgecolor="black", linewidth=0.6)
eje.set_xticks(posiciones)
eje.set_xticklabels([m.replace(" (", "\n(") for m in comparacion_ajuste["modelo"]], fontsize=9)
eje.set_ylabel("accuracy")
eje.set_ylim(0, 1.12)
eje.set_title("La brecha entre entrenamiento y prueba")
eje.legend()
for i, fila in comparacion_ajuste.iterrows():
    eje.text(i, 1.04, f"brecha {fila['brecha']:.3f}", ha="center", fontsize=9,
             fontweight="bold")
plt.tight_layout()
plt.show()

<a id="sec-5-8"></a>
## 5.8 La curva de validación

Ahora la **forma 2** de 5.4. Repetimos el experimento anterior variando la profundidad del árbol
de 1 a 15 y graficamos las dos curvas, para ver el recorrido completo en lugar de tres puntos
sueltos.

Acá la hacemos con la división simple 80/20, que es lo único que tenemos hasta este punto del
cuaderno. Fijate en lo temblorosa que sale la curva de prueba: sube y baja de manera errática. Eso
**no** es el comportamiento del modelo, es el ruido de estar midiendo sobre 34 casos.

En el punto 6.4 vamos a repetir exactamente este gráfico con validación cruzada, y la diferencia
va a ser evidente. Es la mejor justificación de por qué hace falta.

In [ ]:
profundidades = list(range(1, 16))
curva_simple = []

for prof in profundidades:
    arbol = DecisionTreeClassifier(max_depth=prof, random_state=SEMILLA)
    arbol.fit(X_entrena, y_entrena)
    curva_simple.append({
        "max_depth": prof,
        "entrenamiento": f1_score(y_entrena, arbol.predict(X_entrena), average="macro"),
        "prueba": f1_score(y_prueba, arbol.predict(X_prueba), average="macro"),
    })

curva_simple = pd.DataFrame(curva_simple)
curva_simple["brecha"] = curva_simple["entrenamiento"] - curva_simple["prueba"]

print("=" * 78)
print("  CURVA DE VALIDACIÓN con una sola división 80/20")
print("=" * 78)
display(curva_simple)

fig, eje = plt.subplots(figsize=(10, 5))
eje.plot(curva_simple["max_depth"], curva_simple["entrenamiento"], "o-",
         color="#d62728", linewidth=2, label="entrenamiento")
eje.plot(curva_simple["max_depth"], curva_simple["prueba"], "s-",
         color="#1f77b4", linewidth=2, label="prueba (una sola división)")
eje.fill_between(curva_simple["max_depth"], curva_simple["prueba"],
                 curva_simple["entrenamiento"], color="grey", alpha=0.12,
                 label="la brecha")
mejor_simple = int(curva_simple.loc[curva_simple["prueba"].idxmax(), "max_depth"])
eje.axvline(mejor_simple, color="green", linestyle="--", linewidth=1.6,
            label=f"máximo de la curva de prueba = {mejor_simple}")
eje.set_xlabel("max_depth  (complejidad del modelo)")
eje.set_ylabel("F1 macro")
eje.set_title("Curva de validación con UNA división — mirá lo ruidosa que es")
eje.set_xticks(profundidades)
eje.legend(fontsize=9)
plt.tight_layout()
plt.show()

print("  Qué se ve:")
print("    - La curva roja (entrenamiento) sube siempre hasta llegar a 1,0 y quedarse.")
print("      Más profundidad = más capacidad de memorizar.")
print("    - La curva azul (prueba) sube, baja, vuelve a subir. Ese zigzag NO es el")
print("      modelo: es el ruido de medir sobre 34 casos.")
print("    - El área gris es la brecha, que se ensancha hacia la derecha.")
print()
print(f"  Con esta curva, ¿cuál es la profundidad óptima? El máximo está en {mejor_simple},")
print("  pero con tanto ruido es difícil confiar en ese número. Lo repetimos en 6.4")
print("  con validación cruzada.")

<a id="sec-5-9"></a>
## 5.9 La curva de aprendizaje

La última de las cuatro formas, y la que responde una pregunta distinta: **¿conviene conseguir más
datos?**

En lugar de variar la complejidad del modelo, variamos la **cantidad de datos**: entrenamos el
mismo modelo con el 20 %, el 30 %, el 40 %… hasta el 100 % de las filas disponibles, y en cada
tamaño medimos el rendimiento en entrenamiento y en validación cruzada.

Ponemos uno al lado del otro los dos diagnósticos opuestos del punto 5.2:

| Panel | Modelo | Qué buscamos ver |
|---|---|---|
| Izquierda | Árbol **sin límite** de profundidad | **Varianza alta**: la curva de entrenamiento clavada en 1,0 desde el primer tramo y la de validación bastante más abajo. La distancia entre las dos es el síntoma |
| Derecha | Regresión logística con `C=0.0001` | **Sesgo alto**: las dos curvas juntas y planas en un valor bajo. No hay distancia entre ellas porque el modelo no aprende ni siquiera los datos que ve |

**Por qué hay que arruinar un modelo a propósito.** El de la derecha está deliberadamente
estropeado: `C=0.0001` es una regularización brutal que aplasta todos los coeficientes contra cero
y deja al modelo respondiendo prácticamente siempre lo mismo. Hizo falta hacerlo porque **en este
dataset es difícil conseguir subajuste de manera natural**: la señal es tan clara que hasta un
árbol de una sola pregunta rinde bien, como se vio en 5.7. Eso ya es un dato sobre el problema, y
es una buena noticia; pero para ilustrar el sesgo alto hay que fabricarlo.

De paso, es la demostración práctica de lo que dijimos en 5.6 sobre la regularización: `C` chico
compra estabilidad a cambio de sesgo, y llevado al extremo el negocio deja de convenir.

**Qué mirar en cada curva:**

- **La distancia vertical entre las dos líneas** al final del recorrido. Grande ⇒ varianza alta.
- **La altura a la que terminan.** Baja y juntas ⇒ sesgo alto.
- **La pendiente de la línea azul en el tramo final.** Si todavía sube, más datos ayudarían; si ya
  se aplanó, el modelo llegó a su techo con los datos que tiene.

Una advertencia sobre lo que sigue: con 171 filas, los tramos de la izquierda se calculan sobre muy
pocos casos y salen ruidosos. La banda sombreada muestra ese ruido. Hay que leer la **forma** de
las curvas, no cada punto por separado.

In [ ]:
from sklearn.model_selection import learning_curve

modelos_curva = [
    ("Árbol SIN límite de profundidad",
     DecisionTreeClassifier(random_state=SEMILLA)),
    ("Regresión logística con C=0.0001",
     Pipeline([("escalador", StandardScaler()),
               ("modelo", LogisticRegression(C=0.0001, max_iter=5000,
                                             random_state=SEMILLA))])),
]

proporciones = np.linspace(0.2, 1.0, 8)

fig, ejes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
lectura = []

for eje, (titulo, estimador) in zip(ejes, modelos_curva):
    n_casos, score_tr, score_val = learning_curve(
        estimador, X, y,
        cv=cv_demo,
        scoring="f1_macro",
        train_sizes=proporciones,
        shuffle=True,
        random_state=SEMILLA,
        n_jobs=1,
    )
    media_tr, desvio_tr = score_tr.mean(axis=1), score_tr.std(axis=1)
    media_val, desvio_val = score_val.mean(axis=1), score_val.std(axis=1)

    eje.plot(n_casos, media_tr, "o-", color="#d62728", linewidth=2,
             label="entrenamiento")
    eje.fill_between(n_casos, media_tr - desvio_tr, media_tr + desvio_tr,
                     color="#d62728", alpha=0.15)
    eje.plot(n_casos, media_val, "s-", color="#1f77b4", linewidth=2,
             label="validación cruzada")
    eje.fill_between(n_casos, media_val - desvio_val, media_val + desvio_val,
                     color="#1f77b4", alpha=0.15, label="± 1 desvío entre pliegues")
    eje.set_title(titulo, fontsize=11)
    eje.set_xlabel("casos de entrenamiento usados")
    eje.legend(fontsize=9, loc="lower right")

    # Tendencia del tramo final: pendiente por mínimos cuadrados sobre los
    # últimos cuatro puntos, expresada como cuánto subiría a lo largo de ese tramo.
    pendiente = np.polyfit(n_casos[-4:], media_val[-4:], 1)[0]
    tendencia = pendiente * (n_casos[-1] - n_casos[-4])

    lectura.append({
        "modelo": titulo,
        "f1_entrena_final": media_tr[-1],
        "f1_validacion_final": media_val[-1],
        "brecha_final": media_tr[-1] - media_val[-1],
        "tendencia_tramo_final": tendencia,
        "ruido_tipico": desvio_val.mean(),
    })

ejes[0].set_ylabel("F1 macro")
ejes[0].set_ylim(0, 1.05)
fig.suptitle("Curvas de aprendizaje: ¿conviene conseguir más datos?",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

lectura = pd.DataFrame(lectura)
print("=" * 92)
print("  LECTURA DE LAS CURVAS DE APRENDIZAJE")
print("=" * 92)
display(lectura)

for _, fila in lectura.iterrows():
    sube = fila["tendencia_tramo_final"] > fila["ruido_tipico"] / 2
    print(f"  {fila['modelo']}")
    print(f"      Con todos los datos : entrenamiento {fila['f1_entrena_final']:.4f}, "
          f"validación {fila['f1_validacion_final']:.4f}")
    print(f"      Brecha final        : {fila['brecha_final']:.4f}")
    print(f"      Tendencia del tramo final : {fila['tendencia_tramo_final']:+.4f} "
          f"(ruido típico {fila['ruido_tipico']:.4f})")
    print(f"      La curva de validación {'TODAVÍA SUBE' if sube else 'YA SE APLANÓ'}")

    if fila["f1_validacion_final"] < SCORE_SUBAJUSTE:
        print("      >> SESGO ALTO. Las dos curvas juntas y bajas: el modelo no aprende")
        print("         ni siquiera los datos que ve. Más datos NO ayudarían; hace falta")
        print("         un modelo más expresivo, menos regularización o mejores variables.")
    elif fila["brecha_final"] > BRECHA_SOBREAJUSTE and sube:
        print("      >> VARIANZA ALTA y la curva todavía sube: MÁS DATOS AYUDARÍAN.")
    elif fila["brecha_final"] > BRECHA_SOBREAJUSTE:
        print("      >> VARIANZA ALTA pero la curva ya se aplanó. Más datos moverían poco:")
        print("         lo primero que conviene probar es simplificar el modelo (podarlo).")
    else:
        print("      >> Las dos curvas convergen en un valor razonable: el modelo está")
        print("         aprovechando bien los datos que tiene.")
    print()

print("  DOS COSAS PARA NOTAR")
print("  " + "-" * 88)
print("  1. El árbol sin podar alcanza su techo de validación con muy pocos casos y")
print("     después se queda ahí, mientras su curva de entrenamiento está clavada en")
print("     1,0 desde el principio. Esa distancia que no se cierra es la firma de la")
print("     varianza alta: el modelo no necesita más datos para aprender el patrón,")
print("     necesita que lo frenen para que deje de aprender también el ruido.")
print()
print("  2. El modelo sobre-regularizado tiene las dos curvas pegadas, y eso que a")
print("     veces se lee como 'no hay sobreajuste' acá no vale nada: están pegadas")
print("     abajo. Es el recordatorio del punto 5.5 de que la brecha sin el nivel no")
print("     alcanza para diagnosticar.")
print()
print("  CÓMO SE USA ESTO EN LA PRÁCTICA")
print("  " + "-" * 88)
print("  Conseguir más datos suele ser lo más caro de un proyecto: hay que recolectarlos,")
print("  etiquetarlos y validarlos. Esta es la única de las cuatro formas de diagnóstico")
print("  que informa sobre esa decisión ANTES de tomarla, y la única que distingue con")
print("  claridad un problema de sesgo de uno de varianza.")

---
# 6. Validación cruzada K-Fold

<a id="sec-6-1"></a>
## 6.1 Cómo funciona

El punto 3.3 mostró el problema: la evaluación depende de cómo se partió el dataset. El punto 5
mostró qué buscamos detectar. La **validación cruzada** resuelve las dos cosas.

En lugar de partir el dataset una sola vez, se lo divide en *K* partes llamadas **pliegues**
(*folds*). Después se entrena *K* veces: en cada vuelta, un pliegue distinto hace de conjunto de
prueba y los otros *K-1* de entrenamiento.

```
   Pliegue 1:  [PRUEBA][entrena][entrena][entrena][entrena]
   Pliegue 2:  [entrena][PRUEBA][entrena][entrena][entrena]
   Pliegue 3:  [entrena][entrena][PRUEBA][entrena][entrena]
   Pliegue 4:  [entrena][entrena][entrena][PRUEBA][entrena]
   Pliegue 5:  [entrena][entrena][entrena][entrena][PRUEBA]
```

Así **cada caso del dataset es evaluado exactamente una vez**, por un modelo que no lo vio durante
su entrenamiento. Al final se tienen *K* mediciones: su **media** es la estimación del rendimiento,
y su **desvío** dice cuán confiable es esa estimación.

<a id="sec-6-2"></a>
## 6.2 Por qué acá es necesaria y no opcional

Tres razones concretas, no genéricas:

**1. El dataset es chico.** Con 171 filas, un conjunto de prueba del 20% son 34 casos. Ya vimos en
3.3 que la estimación se mueve varios puntos de un `random_state` a otro, y en 5.8 que la curva de
validación sale ilegible de tan ruidosa. La validación cruzada usa las 171 filas para evaluar, no
34.

**2. Vamos a comparar cuatro modelos y tres escalados.** Doce combinaciones. Si las diferencias
entre ellas son de unos pocos puntos y la medición tiene un ruido del mismo orden, la comparación
no significa nada. La validación cruzada da, además de la media, el **desvío entre pliegues**, que
es lo que permite decir si una diferencia observada es real o cabe dentro del ruido. Sin esa
segunda cifra, un ranking de modelos es una lista de casualidades ordenadas.

**3. No se desperdicia ningún dato.** Cada caso participa del entrenamiento en *K-1* vueltas y de
la evaluación en una. Con 171 filas, reservar 34 y no usarlas nunca para entrenar es un lujo que no
nos podemos dar.

**Estratificada, además.** Usamos `StratifiedKFold`, que arma los pliegues manteniendo en cada uno
la misma proporción de clases que el dataset completo. Sin estratificar, algún pliegue podría
quedar con un reparto muy distinto y su métrica sería poco representativa.

**Elegimos K = 5.** Es el valor habitual y el que usa el material del curso. Con K=5 cada pliegue
tiene unos 34 casos de prueba y unos 137 de entrenamiento, lo que mantiene el entrenamiento
razonablemente grande.

¿Por qué no K=10? Daría una estimación levemente menos sesgada —cada modelo entrena con el 90% de
los datos en lugar del 80%— pero con más varianza entre pliegues, porque cada uno evalúa sobre
solo 17 casos. Y cuesta el doble de tiempo. Con este tamaño de dataset, no compensa.

¿Y el extremo, K = número de filas (*leave-one-out*)? Cada pliegue evalúa sobre un solo caso. Es
insesgado pero extremadamente ruidoso y 171 veces más caro.

<a id="sec-6-3"></a>
## 6.3 El esquema estratificado

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=N_PLIEGUES, shuffle=True, random_state=SEMILLA)

print("=" * 78)
print(f"  ESQUEMA DE VALIDACIÓN: StratifiedKFold con {N_PLIEGUES} pliegues")
print("=" * 78)
print(f"  {'Pliegue':>8} {'Entrena':>9} {'Prueba':>8} {'% pos. entrena':>16} {'% pos. prueba':>15}")
for i, (idx_tr, idx_te) in enumerate(cv.split(X, y), start=1):
    print(f"  {i:>8} {len(idx_tr):>9} {len(idx_te):>8} "
          f"{y.iloc[idx_tr].mean():>15.1%} {y.iloc[idx_te].mean():>15.1%}")
print()
print(f"  Proporción de la clase positiva en el dataset completo: {y.mean():.1%}")
print("  La estratificación la mantiene casi idéntica en todos los pliegues.")

fig, eje = plt.subplots(figsize=(11, 3.2))
for i, (idx_tr, idx_te) in enumerate(cv.split(X, y)):
    marcas = np.zeros(len(X))
    marcas[idx_te] = 1
    eje.scatter(range(len(X)), [i] * len(X), c=marcas, marker="_", lw=9,
                cmap=mpl.colors.ListedColormap(["#c7d9ec", "#d62728"]), vmin=0, vmax=1)
eje.set_yticks(range(N_PLIEGUES))
eje.set_yticklabels([f"pliegue {i+1}" for i in range(N_PLIEGUES)])
eje.set_xlabel("índice del caso")
eje.set_title("Cómo se reparten los casos  (rojo = prueba, azul = entrenamiento)")
eje.invert_yaxis()
plt.tight_layout()
plt.show()

<a id="sec-6-4"></a>
## 6.4 Una división contra validación cruzada

Dos comparaciones que cierran el argumento.

**La primera:** el mismo modelo evaluado de las dos formas.

In [ ]:
modelo_demo.fit(X_entrena, y_entrena)
una_division = f1_score(y_prueba, modelo_demo.predict(X_prueba), average="macro")

resultado_cv = cross_validate(modelo_demo, X, y, cv=cv, scoring="f1_macro",
                              return_train_score=True)
por_pliegue = resultado_cv["test_score"]

print("=" * 78)
print("  UNA DIVISIÓN vs. VALIDACIÓN CRUZADA   (F1 macro)")
print("=" * 78)
print(f"  Una sola división 80/20 : {una_division:.4f}   <- un número solo, sin idea del error")
print()
print("  Validación cruzada, pliegue por pliegue:")
for i, s in enumerate(por_pliegue, start=1):
    print(f"      pliegue {i} : {s:.4f}")
print()
print(f"  Media                   : {por_pliegue.mean():.4f}")
print(f"  Desvío entre pliegues   : {por_pliegue.std():.4f}")
print(f"  Rango                   : {por_pliegue.min():.4f} a {por_pliegue.max():.4f}")
print()
print(f"  ESTIMACIÓN FINAL: {por_pliegue.mean():.4f} ± {por_pliegue.std():.4f}")
print()
print("  Esa segunda cifra es la que permite decir si una diferencia entre dos")
print("  modelos es real o si entra dentro del ruido de la medición.")
print("  De acá en adelante, TODA evaluación del cuaderno se hace así.")

**La segunda:** la curva de validación de 5.8, rehecha con validación cruzada.

Es el mismo gráfico, el mismo modelo y el mismo rango de profundidades. Lo único que cambia es cómo
se mide cada punto: antes, con una sola división de 34 casos; ahora, con el promedio de cinco
pliegues, y con una banda que muestra el desvío entre ellos.

Compará las dos curvas azules. La de 5.8 zigzaguea y su máximo es prácticamente arbitrario. Esta
tiene una forma clara: sube, llega a un máximo, y baja. Y la banda sombreada dice, en cada punto,
cuánta confianza merece ese valor.

In [ ]:
from sklearn.model_selection import validation_curve

scores_tr, scores_val = validation_curve(
    DecisionTreeClassifier(random_state=SEMILLA),
    X, y,
    param_name="max_depth",
    param_range=profundidades,
    cv=cv,
    scoring="f1_macro",
    n_jobs=1,
)

media_tr, desvio_tr = scores_tr.mean(axis=1), scores_tr.std(axis=1)
media_val, desvio_val = scores_val.mean(axis=1), scores_val.std(axis=1)
mejor = int(np.argmax(media_val))
mejor_profundidad = profundidades[mejor]

fig, ejes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)

# Izquierda: la de 5.8, con una sola división
ejes[0].plot(curva_simple["max_depth"], curva_simple["entrenamiento"], "o-",
             color="#d62728", linewidth=2, label="entrenamiento")
ejes[0].plot(curva_simple["max_depth"], curva_simple["prueba"], "s-",
             color="#1f77b4", linewidth=2, label="prueba")
ejes[0].axvline(mejor_simple, color="green", linestyle="--", linewidth=1.5,
                label=f"máximo = {mejor_simple}")
ejes[0].set_title("Con UNA división 80/20  (34 casos)")
ejes[0].set_xlabel("max_depth"); ejes[0].set_ylabel("F1 macro")
ejes[0].set_xticks(profundidades)
ejes[0].legend(fontsize=9)

# Derecha: con validación cruzada
ejes[1].plot(profundidades, media_tr, "o-", color="#d62728", linewidth=2,
             label="entrenamiento")
ejes[1].fill_between(profundidades, media_tr - desvio_tr, media_tr + desvio_tr,
                     color="#d62728", alpha=0.15)
ejes[1].plot(profundidades, media_val, "s-", color="#1f77b4", linewidth=2,
             label="validación cruzada")
ejes[1].fill_between(profundidades, media_val - desvio_val, media_val + desvio_val,
                     color="#1f77b4", alpha=0.15, label="± 1 desvío entre pliegues")
ejes[1].axvline(mejor_profundidad, color="green", linestyle="--", linewidth=1.6,
                label=f"mejor profundidad = {mejor_profundidad}")
ejes[1].set_title(f"Con validación cruzada de {N_PLIEGUES} pliegues  (171 casos)")
ejes[1].set_xlabel("max_depth")
ejes[1].set_xticks(profundidades)
ejes[1].legend(fontsize=9)

fig.suptitle("La misma curva, medida de dos maneras", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("=" * 78)
print("  LECTURA DE LA CURVA CON VALIDACIÓN CRUZADA")
print("=" * 78)
print(f"  Zona de SUBAJUSTE   : max_depth de 1 a {max(1, mejor_profundidad - 1)}")
print(f"  Mejor complejidad   : max_depth = {mejor_profundidad}  "
      f"(F1 macro = {media_val[mejor]:.4f} ± {desvio_val[mejor]:.4f})")
print(f"  Zona de SOBREAJUSTE : max_depth mayor a {mejor_profundidad}")
print()
print(f"  Brecha en el óptimo : {media_tr[mejor] - media_val[mejor]:.4f}")
print(f"  Brecha en el extremo: {media_tr[-1] - media_val[-1]:.4f}  (max_depth = {profundidades[-1]})")
print()
print(f"  La curva de una sola división daba como óptimo max_depth = {mejor_simple};")
print(f"  la validación cruzada da {mejor_profundidad}. La segunda merece mucha más confianza:")
print(f"  está promediando {N_PLIEGUES} mediciones en lugar de reportar una.")
print()
print("  Y la banda sombreada agrega algo que la curva de la izquierda no tenía:")
print("  una medida de cuánta confianza merece cada punto.")

---
# 7. La arquitectura del cuaderno

Vamos a evaluar cuatro modelos con tres escalados cada uno. Doce combinaciones, cada una con una
docena de métricas y media docena de gráficos.

Escribir eso a mano doce veces sería insostenible: cualquier ajuste habría que replicarlo en todos
lados, y a la primera distracción dos modelos quedarían evaluados con criterios distintos —con lo
cual la comparación final no valdría nada—.

La solución es separar **qué modelos se evalúan** de **cómo se los evalúa**:

```
CATALOGO_MODELOS  ──►  evaluar_modelo()  ──►  RESULTADOS  ──►  comparativa (punto 12)
  (una entrada          (el mismo             (tabla que        (lee de RESULTADOS,
   por modelo)          procedimiento          se va             sin saber cuántos
                        para todos)            llenando)          modelos hay)
```

Las tres consecuencias que importan:

1. **Todos los modelos se miden con exactamente el mismo código.** No hay forma de que uno quede
   evaluado con un criterio distinto.
2. **Agregar un modelo cuesta una entrada en el catálogo.** Nada más.
3. **La comparativa final lo toma solo.** No hay ninguna lista de modelos escrita a mano en el
   punto 12.

Las próximas cuatro celdas construyen esa maquinaria. Son la parte más densa del cuaderno; a partir
del punto 8 todo se vuelve repetitivo a propósito.

<a id="sec-7-1"></a>
## 7.1 Los escaladores y el esquema de validación

Definimos los tres tratamientos de escala que vamos a comparar. El esquema de validación cruzada
`cv` ya quedó definido en 6.3, y lo reutilizamos.

Que sea **el mismo objeto `cv`** para todos no es un detalle: garantiza que los pliegues sean
idénticos en todas las evaluaciones, y por lo tanto que las diferencias entre modelos se deban a
los modelos y no a que a uno le tocó una partición más fácil.

In [ ]:
# Los tres tratamientos de escala a comparar
ESCALADORES = {
    "Sin escalar": None,
    "Min-Max":     MinMaxScaler(),
    "Z-score":     StandardScaler(),
}

# Métricas que se calculan en cada pliegue, para el diagnóstico de ajuste
METRICAS_CV = {
    "accuracy": "accuracy",
    "f1_macro": "f1_macro",
    "roc_auc": "roc_auc",
}

print("Escaladores a comparar :", list(ESCALADORES.keys()))
print(f"Validación cruzada     : StratifiedKFold({N_PLIEGUES}, shuffle=True, "
      f"random_state={SEMILLA})")
print("Métricas por pliegue   :", list(METRICAS_CV.keys()))
print()
print("El mismo objeto cv para todos los modelos: los pliegues son idénticos en")
print("todas las evaluaciones, así que las diferencias son atribuibles al modelo.")

<a id="sec-7-2"></a>
## 7.2 El catálogo de modelos

El catálogo es un diccionario donde cada entrada describe un modelo. Los campos:

| Campo | Qué es |
|---|---|
| `estimador` | El objeto de scikit-learn, sin entrenar |
| `escalados` | Qué tratamientos de escala probarle. Los árboles no lo necesitan, pero se lo aplicamos igual para *verificar* que efectivamente no cambia nada |
| `color` | Un color fijo, para que el modelo se reconozca en todos los gráficos |
| `nota` | Una línea que recuerda qué representa ese modelo |
| `grafico_extra` | Opcional: una función que dibuja algo propio del modelo (coeficientes, importancias, el árbol). Si es `None`, no se dibuja nada |

`RESULTADOS` es la lista donde se acumulan las métricas de cada combinación, y `PREDICCIONES`
guarda las probabilidades predichas de cada una, que el punto 12 necesita para superponer las
curvas ROC.

`construir_pipeline()` es la función que arma el `Pipeline` del punto 4.6: escalador —si
corresponde— más modelo. **Ningún modelo de este cuaderno se evalúa fuera de un pipeline.**

In [ ]:
from sklearn.base import clone

CATALOGO_MODELOS = {}      # nombre -> configuración
RESULTADOS = []            # una fila por (modelo, escalado)
PREDICCIONES = {}          # (modelo, escalado) -> predicciones y detalle


def registrar_modelo(nombre, estimador, escalados=None, color="#4c72b0",
                     nota="", grafico_extra=None):
    CATALOGO_MODELOS[nombre] = {
        "estimador": estimador,
        "escalados": escalados if escalados is not None else list(ESCALADORES.keys()),
        "color": color,
        "nota": nota,
        "grafico_extra": grafico_extra,
    }
    print(f"Registrado: {nombre}")
    print(f"    escalados a probar : {CATALOGO_MODELOS[nombre]['escalados']}")
    if nota:
        print(f"    {nota}")


def construir_pipeline(nombre_modelo, nombre_escalado):
    # Arma el Pipeline: escalador (si corresponde) + modelo.
    # Se clona todo para que cada combinación entrene su propia copia limpia.
    pasos = []
    escalador = ESCALADORES[nombre_escalado]
    if escalador is not None:
        pasos.append(("escalador", clone(escalador)))
    pasos.append(("modelo", clone(CATALOGO_MODELOS[nombre_modelo]["estimador"])))
    return Pipeline(pasos)


print("Catálogo inicializado (vacío).")

<a id="sec-7-3"></a>
## 7.3 El cálculo de métricas

Acá se define, de una vez y para siempre, **qué se mide**. Estas dos funciones son las que hacen
que los cuatro modelos sean comparables.

**`calcular_metricas()`** recibe las etiquetas reales, las predichas y las probabilidades, y
devuelve un diccionario con todo. Agrupadas por lo que responden:

*Rendimiento general*
- **Accuracy** — proporción de aciertos sobre el total. Intuitiva, pero engañosa si las clases
  están desbalanceadas.
- **Balanced accuracy** — el promedio del recall de cada clase. Corrige ese sesgo.

*Rendimiento sobre la clase positiva*
- **Precisión** — de los que predije positivos, cuántos lo eran.
- **Recall** — de los que realmente eran positivos, cuántos detecté.
- **F1-score** — la media armónica de las dos anteriores. Si una es baja, el F1 es bajo.
- **F1 macro** — el promedio simple del F1 de cada clase; trata a las dos por igual.

*Calidad del ordenamiento (sin depender del umbral)*
- **ROC-AUC** — la probabilidad de que el modelo le asigne mayor puntaje a un caso positivo tomado
  al azar que a uno negativo. 0,5 es azar puro, 1 es perfecto.
- **Average precision** — el área bajo la curva precisión-recall. Más informativa que el AUC cuando
  la clase positiva es minoritaria.

*Acuerdo corregido por azar*
- **Coeficiente de Matthews (MCC)** — considera las cuatro celdas de la matriz de confusión. Va de
  -1 a +1 y es la métrica más honesta con clases desbalanceadas.
- **Kappa de Cohen** — mide el acuerdo descontando el que se lograría por azar.

*Calidad de las probabilidades*
- **Log loss** — castiga las predicciones seguras y equivocadas. Más bajo es mejor.
- **Brier score** — el error cuadrático medio de las probabilidades. Más bajo es mejor. Un modelo
  puede clasificar bien y estimar mal las probabilidades; estas dos lo detectan.

**`diagnosticar_ajuste()`** aplica lo del punto 5: compara el rendimiento en entrenamiento contra
el de validación, mira el desvío entre pliegues y emite el veredicto, usando los umbrales que
declaramos en 0.4.

In [ ]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    average_precision_score, precision_recall_curve,
    matthews_corrcoef, cohen_kappa_score,
    log_loss, brier_score_loss,
)


def calcular_metricas(y_real, y_pred, y_proba):
    return {
        # Rendimiento general
        "accuracy":          accuracy_score(y_real, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_real, y_pred),
        # Clase positiva
        "precision":         precision_score(y_real, y_pred, zero_division=0),
        "recall":            recall_score(y_real, y_pred, zero_division=0),
        "f1":                f1_score(y_real, y_pred, zero_division=0),
        # Promediadas entre clases
        "precision_macro":   precision_score(y_real, y_pred, average="macro", zero_division=0),
        "recall_macro":      recall_score(y_real, y_pred, average="macro", zero_division=0),
        "f1_macro":          f1_score(y_real, y_pred, average="macro", zero_division=0),
        "f1_weighted":       f1_score(y_real, y_pred, average="weighted", zero_division=0),
        # Independientes del umbral
        "roc_auc":           roc_auc_score(y_real, y_proba),
        "average_precision": average_precision_score(y_real, y_proba),
        # Corregidas por azar
        "mcc":               matthews_corrcoef(y_real, y_pred),
        "kappa":             cohen_kappa_score(y_real, y_pred),
        # Calidad de las probabilidades
        "log_loss":          log_loss(y_real, np.clip(y_proba, 1e-12, 1 - 1e-12)),
        "brier":             brier_score_loss(y_real, y_proba),
    }


def diagnosticar_ajuste(score_entrena, score_valida, desvio_valida):
    brecha = score_entrena - score_valida
    avisos = []

    if brecha > BRECHA_SOBREAJUSTE:
        avisos.append(
            f"SOBREAJUSTE: la brecha entre entrenamiento ({score_entrena:.4f}) y "
            f"validación ({score_valida:.4f}) es de {brecha:.4f}, por encima del umbral "
            f"de {BRECHA_SOBREAJUSTE:.2f}. El modelo memoriza más de lo que generaliza."
        )
    elif brecha > BRECHA_SOBREAJUSTE / 2:
        avisos.append(
            f"Sobreajuste leve: brecha de {brecha:.4f}. Todavía tolerable, pero conviene "
            f"no aumentar la complejidad del modelo."
        )

    if score_valida < SCORE_SUBAJUSTE:
        avisos.append(
            f"SUBAJUSTE: el rendimiento en validación ({score_valida:.4f}) está por debajo "
            f"de {SCORE_SUBAJUSTE:.2f}. El modelo puede ser demasiado simple para el problema."
        )

    if desvio_valida > DESVIO_INESTABLE:
        avisos.append(
            f"INESTABILIDAD: el desvío entre pliegues es {desvio_valida:.4f}, por encima de "
            f"{DESVIO_INESTABLE:.2f}. El rendimiento depende mucho de qué datos le toquen; "
            f"la media es poco confiable."
        )

    if not avisos:
        avisos.append(
            f"AJUSTE CORRECTO: brecha de {brecha:.4f} y desvío entre pliegues de "
            f"{desvio_valida:.4f}, los dos dentro de lo aceptable. El modelo generaliza bien."
        )

    return brecha, avisos


ejemplo_metricas = calcular_metricas(np.array([0, 1]), np.array([0, 1]), np.array([0.1, 0.9]))
print(f"Funciones definidas. calcular_metricas() devuelve {len(ejemplo_metricas)} métricas:")
for clave in ejemplo_metricas:
    print(f"    {clave}")

<a id="sec-7-4"></a>
## 7.4 El bloque de resultados

`evaluar_modelo()` es el corazón del cuaderno. Es **el mismo procedimiento para los cuatro
modelos**, y lo que hace, en orden:

1. Para cada escalado registrado, arma el `Pipeline` y lo evalúa con validación cruzada.
2. Obtiene las predicciones **fuera de pliegue** con `cross_val_predict`: cada caso se predice con
   un modelo que no lo vio durante su entrenamiento. Así las métricas y la matriz de confusión se
   calculan sobre las 171 filas, pero sin fuga.
3. Corre `cross_validate` con `return_train_score=True` para tener el rendimiento en entrenamiento
   y en validación pliegue por pliegue, que es lo que alimenta el diagnóstico del punto 5.
4. Guarda todo en `RESULTADOS` y `PREDICCIONES`.
5. Imprime la tabla comparativa de los escalados, elige el mejor por ROC-AUC y, sobre esa variante,
   despliega el bloque completo: métricas detalladas, diagnóstico de ajuste, reporte de
   clasificación y seis gráficos.
6. Si el modelo trae un `grafico_extra`, lo dibuja al final, claramente separado del bloque común.

**Sobre `cross_val_predict`.** Es lo que permite tener una matriz de confusión con los 171 casos en
lugar de con los 34 de una única partición de prueba. Cada predicción proviene de un modelo que no
vio ese caso, así que sigue siendo una estimación honesta.

In [ ]:
from sklearn.model_selection import cross_val_predict


def evaluar_modelo(nombre, mostrar_graficos=True):
    configuracion = CATALOGO_MODELOS[nombre]
    color_modelo = configuracion["color"]

    print("=" * 88)
    print(f"  {nombre.upper()}")
    if configuracion["nota"]:
        print(f"  {configuracion['nota']}")
    print("=" * 88)

    filas_modelo = []

    # ---------- 1. Evaluar cada escalado ----------
    for nombre_escalado in configuracion["escalados"]:
        pipeline = construir_pipeline(nombre, nombre_escalado)

        y_pred = cross_val_predict(pipeline, X, y, cv=cv, method="predict")
        y_proba = cross_val_predict(pipeline, X, y, cv=cv, method="predict_proba")[:, 1]

        metricas = calcular_metricas(y, y_pred, y_proba)

        detalle = cross_validate(pipeline, X, y, cv=cv, scoring=METRICAS_CV,
                                 return_train_score=True, n_jobs=1)

        for clave in METRICAS_CV:
            metricas[f"cv_{clave}_entrena"] = detalle[f"train_{clave}"].mean()
            metricas[f"cv_{clave}_media"] = detalle[f"test_{clave}"].mean()
            metricas[f"cv_{clave}_desvio"] = detalle[f"test_{clave}"].std()
            metricas[f"cv_{clave}_min"] = detalle[f"test_{clave}"].min()
            metricas[f"cv_{clave}_max"] = detalle[f"test_{clave}"].max()

        brecha, avisos = diagnosticar_ajuste(
            metricas["cv_f1_macro_entrena"],
            metricas["cv_f1_macro_media"],
            metricas["cv_f1_macro_desvio"],
        )
        metricas["brecha_ajuste"] = brecha
        metricas["diagnostico"] = avisos[0].split(":")[0]

        fila = {"modelo": nombre, "escalado": nombre_escalado, "color": color_modelo}
        fila.update(metricas)
        filas_modelo.append(fila)

        RESULTADOS.append(fila)
        PREDICCIONES[(nombre, nombre_escalado)] = {
            "y_pred": y_pred, "y_proba": y_proba,
            "por_pliegue": detalle, "avisos": avisos,
        }

    tabla_modelo = pd.DataFrame(filas_modelo)

    # ---------- 2. Comparación de escalados ----------
    print()
    print("  EFECTO DEL ESCALADO")
    print("  " + "-" * 84)
    resumen_escalado = tabla_modelo[[
        "escalado", "accuracy", "balanced_accuracy", "f1_macro",
        "roc_auc", "cv_f1_macro_desvio", "brecha_ajuste",
    ]].rename(columns={"cv_f1_macro_desvio": "desvio_pliegues", "brecha_ajuste": "brecha"})
    display(resumen_escalado.set_index("escalado"))

    rango_auc = tabla_modelo["roc_auc"].max() - tabla_modelo["roc_auc"].min()
    if rango_auc < 1e-9:
        print("  El escalado NO cambia absolutamente nada en este modelo.")
        print("  Es lo esperado en modelos basados en árboles: los cortes se hacen por")
        print("  umbrales, y Min-Max y Z-score son transformaciones monótonas que no")
        print("  alteran el orden de los valores, así que los cortes caen en el mismo lugar.")
    else:
        print(f"  Diferencia entre el mejor y el peor escalado (ROC-AUC): {rango_auc:.4f}")

    # ---------- 3. La mejor variante ----------
    mejor = tabla_modelo.sort_values("roc_auc", ascending=False).iloc[0]
    escalado_elegido = mejor["escalado"]
    guardado = PREDICCIONES[(nombre, escalado_elegido)]
    y_pred = guardado["y_pred"]
    y_proba = guardado["y_proba"]
    detalle = guardado["por_pliegue"]

    print()
    print(f"  MEJOR VARIANTE: {nombre} + {escalado_elegido}")
    print("  " + "=" * 84)

    # ---------- 4. Métricas detalladas ----------
    print()
    print("  MÉTRICAS  (predicciones fuera de pliegue sobre los 171 casos)")
    print("  " + "-" * 84)
    bloques = [
        ("Rendimiento general", [
            ("Accuracy", "accuracy", "proporción de aciertos sobre el total"),
            ("Balanced accuracy", "balanced_accuracy", "promedio del recall de cada clase"),
        ]),
        (f"Clase positiva ({CLASE_POSITIVA})", [
            ("Precisión", "precision", "de los predichos positivos, cuántos lo eran"),
            ("Recall", "recall", "de los positivos reales, cuántos detectó"),
            ("F1-score", "f1", "media armónica de precisión y recall"),
        ]),
        ("Promediadas entre clases", [
            ("Precisión macro", "precision_macro", "promedio simple entre clases"),
            ("Recall macro", "recall_macro", "promedio simple entre clases"),
            ("F1 macro", "f1_macro", "trata a las dos clases por igual"),
            ("F1 ponderado", "f1_weighted", "pondera por la frecuencia de cada clase"),
        ]),
        ("Independientes del umbral", [
            ("ROC-AUC", "roc_auc", "capacidad de ordenar; 0,5 = azar"),
            ("Average precision", "average_precision", "área bajo la curva precisión-recall"),
        ]),
        ("Corregidas por azar", [
            ("Matthews (MCC)", "mcc", "de -1 a +1; usa las cuatro celdas"),
            ("Kappa de Cohen", "kappa", "acuerdo descontando el azar"),
        ]),
        ("Calidad de las probabilidades", [
            ("Log loss", "log_loss", "más bajo es mejor"),
            ("Brier score", "brier", "más bajo es mejor"),
        ]),
    ]
    for titulo, items in bloques:
        print(f"    {titulo}")
        for etiqueta, clave, ayuda in items:
            print(f"      {etiqueta:<20} {mejor[clave]:>8.4f}    {ayuda}")

    # ---------- 5. Diagnóstico de ajuste ----------
    print()
    print("  DIAGNÓSTICO DE AJUSTE  (F1 macro por pliegue)")
    print("  " + "-" * 84)
    print(f"    {'Pliegue':>8} {'Entrenamiento':>15} {'Validación':>13} {'Brecha':>10}")
    for i, (tr, te) in enumerate(zip(detalle["train_f1_macro"], detalle["test_f1_macro"]), 1):
        print(f"    {i:>8} {tr:>15.4f} {te:>13.4f} {tr - te:>10.4f}")
    print(f"    {'MEDIA':>8} {detalle['train_f1_macro'].mean():>15.4f} "
          f"{detalle['test_f1_macro'].mean():>13.4f} "
          f"{detalle['train_f1_macro'].mean() - detalle['test_f1_macro'].mean():>10.4f}")
    print(f"    {'DESVÍO':>8} {detalle['train_f1_macro'].std():>15.4f} "
          f"{detalle['test_f1_macro'].std():>13.4f}")
    print()
    print(f"    Estimación final: F1 macro = {detalle['test_f1_macro'].mean():.4f} "
          f"± {detalle['test_f1_macro'].std():.4f}")
    print(f"    Rango entre pliegues: {detalle['test_f1_macro'].min():.4f} a "
          f"{detalle['test_f1_macro'].max():.4f}")
    print()
    for aviso in guardado["avisos"]:
        print(f"    >> {aviso}")

    # ---------- 6. Reporte de clasificación ----------
    print()
    print("  REPORTE DE CLASIFICACIÓN")
    print("  " + "-" * 84)
    print(classification_report(
        y, y_pred,
        target_names=[f"{clase_negativa} (0)", f"{CLASE_POSITIVA} (1)"],
        digits=4, zero_division=0,
    ))

    if not mostrar_graficos:
        return tabla_modelo

    # ---------- 7. Gráficos ----------
    fig = plt.figure(figsize=(16, 9.5))
    rejilla = fig.add_gridspec(2, 3, hspace=0.38, wspace=0.28)

    # (a) Matriz de confusión, conteos
    eje = fig.add_subplot(rejilla[0, 0])
    matriz = confusion_matrix(y, y_pred)
    sns.heatmap(matriz, annot=True, fmt="d", cmap="Blues", cbar=False, square=True,
                xticklabels=[clase_negativa, CLASE_POSITIVA],
                yticklabels=[clase_negativa, CLASE_POSITIVA], ax=eje,
                annot_kws={"size": 15, "weight": "bold"})
    eje.set_xlabel("predicho"); eje.set_ylabel("real")
    eje.set_title("Matriz de confusión (casos)")
    vn, fp, fn, vp = matriz.ravel()
    eje.text(0.5, -0.30, f"VN={vn}  FP={fp}  FN={fn}  VP={vp}",
             transform=eje.transAxes, ha="center", fontsize=9, style="italic")

    # (b) Matriz normalizada
    eje = fig.add_subplot(rejilla[0, 1])
    matriz_norm = confusion_matrix(y, y_pred, normalize="true")
    sns.heatmap(matriz_norm, annot=True, fmt=".2%", cmap="Greens", cbar=False, square=True,
                xticklabels=[clase_negativa, CLASE_POSITIVA],
                yticklabels=[clase_negativa, CLASE_POSITIVA], ax=eje,
                annot_kws={"size": 12, "weight": "bold"})
    eje.set_xlabel("predicho"); eje.set_ylabel("real")
    eje.set_title("Matriz normalizada por clase real")

    # (c) Curva ROC
    eje = fig.add_subplot(rejilla[0, 2])
    fpr, tpr, _ = roc_curve(y, y_proba)
    eje.plot(fpr, tpr, color=color_modelo, linewidth=2.4,
             label=f"AUC = {mejor['roc_auc']:.4f}")
    eje.fill_between(fpr, tpr, alpha=0.12, color=color_modelo)
    eje.plot([0, 1], [0, 1], "k--", linewidth=1, label="azar (AUC = 0,5)")
    eje.set_xlabel("tasa de falsos positivos")
    eje.set_ylabel("tasa de verdaderos positivos")
    eje.set_title("Curva ROC")
    eje.legend(loc="lower right", fontsize=9)

    # (d) Curva precisión-recall
    eje = fig.add_subplot(rejilla[1, 0])
    prec, rec, _ = precision_recall_curve(y, y_proba)
    eje.plot(rec, prec, color=color_modelo, linewidth=2.4,
             label=f"AP = {mejor['average_precision']:.4f}")
    eje.fill_between(rec, prec, alpha=0.12, color=color_modelo)
    eje.axhline(y.mean(), color="black", linestyle="--", linewidth=1,
                label=f"azar = {y.mean():.3f}")
    eje.set_xlabel("recall"); eje.set_ylabel("precisión")
    eje.set_title("Curva precisión-recall")
    eje.legend(loc="lower left", fontsize=9)

    # (e) Distribución de las probabilidades
    eje = fig.add_subplot(rejilla[1, 1])
    for valor, etiqueta in [(0, clase_negativa), (1, CLASE_POSITIVA)]:
        sns.histplot(y_proba[y == valor], bins=22, ax=eje, alpha=0.55,
                     label=f"real = {etiqueta}", color=COLOR_CLASE.get(etiqueta), kde=True)
    eje.axvline(0.5, color="black", linestyle="--", linewidth=1.4, label="umbral 0,5")
    eje.set_xlabel(f"probabilidad predicha de {CLASE_POSITIVA}")
    eje.set_ylabel("casos")
    eje.set_title("Separación de las probabilidades")
    eje.legend(fontsize=8)

    # (f) Rendimiento por pliegue
    eje = fig.add_subplot(rejilla[1, 2])
    pliegues = np.arange(1, N_PLIEGUES + 1)
    ancho = 0.38
    eje.bar(pliegues - ancho / 2, detalle["train_f1_macro"], ancho,
            label="entrenamiento", color="#d62728", alpha=0.75,
            edgecolor="black", linewidth=0.5)
    eje.bar(pliegues + ancho / 2, detalle["test_f1_macro"], ancho,
            label="validación", color=color_modelo, alpha=0.9,
            edgecolor="black", linewidth=0.5)
    eje.axhline(detalle["test_f1_macro"].mean(), color="black", linestyle="--", linewidth=1.2,
                label=f"media = {detalle['test_f1_macro'].mean():.3f}")
    eje.set_xlabel("pliegue"); eje.set_ylabel("F1 macro")
    eje.set_title("Entrenamiento vs. validación por pliegue")
    eje.set_xticks(pliegues)
    eje.set_ylim(0, 1.08)
    eje.legend(fontsize=8, loc="lower right")

    fig.suptitle(f"{nombre}  —  escalado: {escalado_elegido}",
                 fontsize=14, fontweight="bold", y=0.98)
    plt.show()

    # ---------- 8. Gráfico propio del modelo ----------
    if configuracion["grafico_extra"] is not None:
        print()
        print("  ANÁLISIS ESPECÍFICO DE ESTE MODELO")
        print("  " + "-" * 84)
        pipeline_final = construir_pipeline(nombre, escalado_elegido)
        pipeline_final.fit(X, y)
        configuracion["grafico_extra"](pipeline_final, color_modelo)

    return tabla_modelo


print("Función evaluar_modelo() definida.")
print("Es la misma para los cuatro modelos: ninguno puede quedar evaluado")
print("con un criterio distinto.")

<a id="sec-7-5"></a>
## 7.5 Cómo agregar un modelo nuevo

Esta es la parte práctica de toda la arquitectura. Para sumar un modelo al cuaderno:

**Paso 1 — Importarlo.**

```python
from sklearn.svm import SVC
```

**Paso 2 — Registrarlo en el catálogo.**

```python
registrar_modelo(
    "SVM (RBF)",
    SVC(kernel="rbf", probability=True, random_state=SEMILLA),
    color="#9467bd",
    nota="Frontera no lineal basada en márgenes; necesita escalado.",
)
```

**Paso 3 — Evaluarlo.**

```python
evaluar_modelo("SVM (RBF)")
```

Eso es todo. El modelo queda con exactamente el mismo bloque de métricas que los demás, y la
comparativa del punto 12 lo incluye automáticamente porque lee de `RESULTADOS`, sin ninguna lista
escrita a mano.

**Dos cosas a tener en cuenta:**

- El estimador tiene que exponer `predict_proba()`. La mayoría de los clasificadores de
  scikit-learn lo hace. `SVC` lo hace solo si se lo crea con `probability=True`.
- Si el modelo no necesita escalado, se le puede pasar `escalados=["Sin escalar"]` para no evaluar
  tres veces lo mismo. Nosotros igual se lo aplicamos a los árboles, porque verificarlo es parte de
  lo que queremos mostrar.

**Un ejemplo completo, listo para descomentar:**

In [ ]:
# ---------------------------------------------------------------------
# EJEMPLO: agregar un quinto modelo. Descomentá y corré.
# ---------------------------------------------------------------------

# from sklearn.naive_bayes import GaussianNB
#
# registrar_modelo(
#     "Naive Bayes",
#     GaussianNB(),
#     escalados=["Sin escalar", "Z-score"],
#     color="#8c564b",
#     nota="Supone independencia entre variables; muy rápido, buena línea de base.",
# )
# evaluar_modelo("Naive Bayes")
#
# Después de correr esto, volvé a ejecutar el punto 12:
# el modelo nuevo aparece en todas las tablas, gráficos y en el ranking.

print("El catálogo está listo para recibir modelos.")
print(f"Modelos registrados hasta ahora: {len(CATALOGO_MODELOS)}")

---
<a id="sec-8"></a>
# 8. Modelo 1 — Regresión Logística

**Cómo funciona.** La regresión logística calcula una combinación lineal de las variables —igual
que la regresión lineal— pero después pasa ese resultado por una función **sigmoide**, que aplasta
cualquier número real al intervalo entre 0 y 1:

$$P(y=1 \mid x) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + \dots + \beta_n x_n)}}$$

La salida no es una clase sino una **probabilidad**. Para convertirla en decisión se aplica un
**umbral de corte**, habitualmente 0,5: por encima se predice la clase positiva, por debajo la
negativa.

Esa probabilidad es más rica que una respuesta binaria. Una empresa no solo sabe *si* un cliente
va a cancelar: sabe *qué tan probable* es, y puede priorizar. Y si el costo de equivocarse no es
simétrico —perder un cliente cuesta más que llamarlo de más— el umbral se puede mover sin volver a
entrenar nada.

**Qué significan los coeficientes.** Cada β es el cambio en el **logaritmo de la razón de momios**
(*log-odds*) por unidad de la variable. En castellano: el signo dice hacia qué clase empuja la
variable, y la magnitud, con cuánta fuerza. `exp(β)` da el factor por el que se multiplican los
momios. Ninguno de los otros tres modelos de este cuaderno da algo tan directo de leer.

**Por qué es el primer modelo.** Tres razones:

1. **Es interpretable.** Cada variable tiene un coeficiente que dice cuánto pesa y en qué
   dirección.
2. **Es estable.** No tiene hiperparámetros críticos que ajustar y con pocos datos se comporta
   bien.
3. **Es la vara.** Cualquier modelo más complejo tiene que superarla para justificar su
   complejidad. Si un bosque de 400 árboles empata con una logística, el que conviene poner en
   producción es la logística.

### Los hiperparámetros que importan

| Parámetro | Qué controla | Efecto de subirlo |
|---|---|---|
| `C` | El inverso de la fuerza de la regularización | Menos regularización: coeficientes más grandes, más riesgo de sobreajuste |
| `penalty` | La forma de la penalización: `"l2"`, `"l1"`, `"elasticnet"` | L1 lleva coeficientes a cero exacto y selecciona variables sola |
| `solver` | El algoritmo de optimización | `lbfgs` (por defecto) sirve para L2; `liblinear` o `saga` hacen falta para L1 |
| `class_weight` | Cuánto pesa cada clase | `"balanced"` compensa el desbalance dándole más peso a la clase minoritaria |
| `max_iter` | Cuántas iteraciones puede usar el optimizador | Evita el aviso de no convergencia; subirlo no cambia el resultado, solo lo deja llegar |

### Ventajas y desventajas

| A favor | En contra |
|---|---|
| Interpretable coeficiente por coeficiente | Solo puede dibujar una frontera **lineal** |
| Devuelve probabilidades bien calibradas | Si la separación real es curva, subajusta y no hay parámetro que lo arregle |
| Rápida de entrenar y de predecir | Sensible a variables muy correlacionadas entre sí |
| Regularizada de fábrica: sobreajusta poco | Necesita escalado |
| Estable con pocos datos | No captura interacciones entre variables salvo que se las fabriquemos a mano |

### Dónde se usa en la práctica

Es probablemente el clasificador más usado del mundo fuera del ámbito académico, y no por inercia:
en dominios regulados **hay que poder explicar la decisión**. Scoring crediticio —el banco tiene
que justificar por qué rechazó un crédito—, epidemiología y ensayos clínicos, evaluación de riesgo
en seguros, modelos de propensión en marketing. En todos esos casos un modelo un poco menos preciso
pero auditable le gana a uno mejor que nadie puede explicar.

### Cómo se sobreajusta este modelo

Poco, y ese es su mérito. Sobreajusta cuando hay **más variables que filas** o cuando se sube `C`
a valores grandes desactivando la regularización. Con 7 variables y 171 filas estamos muy lejos de
esa zona.

### La configuración que usamos

- `max_iter=5000` — el optimizador por defecto a veces no converge en 100 iteraciones y emite un
  aviso. Subirlo es gratis.
- Regularización **L2** con `C=1.0` (el valor por defecto). Como vimos en 5.6, la regularización
  penaliza los coeficientes grandes, lo que previene el sobreajuste y —importante en este
  dataset— **estabiliza el modelo frente a la dependencia entre las variables**: recordemos que
  las siete columnas suman 1 en cada fila, lo que las vuelve linealmente dependientes. Sin
  regularización esa dependencia haría que los coeficientes fueran indeterminados. Con ella, el
  modelo elige una solución concreta y reproducible.
- **Necesita escalado**: la penalización L2 castiga todos los coeficientes por igual, así que con
  variables en escalas muy distintas la penalización sería injusta con las de números chicos.

El gráfico propio de este modelo son sus **coeficientes**.

In [ ]:
# LogisticRegression ya se importó en el punto 3.3 para la demostración.
# Lo dejamos explícito acá para que esta sección se pueda leer sola.
from sklearn.linear_model import LogisticRegression


def grafico_coeficientes(pipeline_ajustado, color):
    modelo = pipeline_ajustado.named_steps["modelo"]
    coeficientes = pd.Series(modelo.coef_[0], index=predictoras).sort_values()

    fig, ejes = plt.subplots(1, 2, figsize=(14, 4.4))

    colores = [COLOR_CLASE[CLASE_POSITIVA] if v > 0 else COLOR_CLASE[clase_negativa]
               for v in coeficientes.values]
    ejes[0].barh(coeficientes.index, coeficientes.values, color=colores,
                 edgecolor="black", linewidth=0.6)
    ejes[0].axvline(0, color="black", linewidth=1)
    ejes[0].set_xlabel("coeficiente")
    ejes[0].set_title(f"Coeficientes  (positivo empuja hacia {CLASE_POSITIVA})")

    magnitud = coeficientes.abs().sort_values(ascending=True)
    ejes[1].barh(magnitud.index, magnitud.values, color=color,
                 edgecolor="black", linewidth=0.6)
    ejes[1].set_xlabel("|coeficiente|")
    ejes[1].set_title("Importancia por magnitud")

    plt.tight_layout()
    plt.show()

    print("  Lectura de los coeficientes:")
    for variable, valor in coeficientes.sort_values(key=abs, ascending=False).items():
        direccion = CLASE_POSITIVA if valor > 0 else clase_negativa
        print(f"    {variable:<12} {valor:>+8.4f}   empuja hacia {direccion}")
    print()
    print(f"  Intercepto: {modelo.intercept_[0]:+.4f}")
    print()
    print("  Recordatorio: como las siete columnas suman 1, los coeficientes son")
    print("  relativos entre sí. La regularización L2 elige una solución concreta y")
    print("  reproducible, pero la magnitud individual hay que leerla con esa cautela.")


registrar_modelo(
    "Regresión Logística",
    LogisticRegression(max_iter=5000, random_state=SEMILLA),
    color="#1f77b4",
    nota="Frontera lineal; devuelve probabilidades; interpretable. Es la vara de comparación.",
    grafico_extra=grafico_coeficientes,
)

Y ahora el bloque de resultados. Es exactamente el mismo que van a recibir los otros tres modelos.

In [ ]:
tabla_logistica = evaluar_modelo("Regresión Logística")

---
<a id="sec-9"></a>
# 9. Modelo 2 — K-Nearest Neighbors

**Cómo funciona.** KNN no ajusta ninguna fórmula. Guarda los datos de entrenamiento y, cuando
tiene que clasificar un caso nuevo, calcula la **distancia** entre ese caso y todos los demás,
toma los *k* más cercanos y **vota por mayoría**.

Si *k*=5 y de los cinco vecinos más cercanos tres son de clase 1 y dos de clase 0, predice
clase 1.

**Es un modelo perezoso** (*lazy learner*), y eso lo hace distinto de los otros tres. No hay
entrenamiento propiamente dicho: `fit()` se limita a guardar los datos. Todo el trabajo ocurre en
`predict()`, cuando hay que medir distancias contra el dataset entero. La consecuencia práctica es
que entrena instantáneamente y predice lento, justo al revés que los demás.

Y no tiene un modelo del mundo: no hay coeficientes, no hay reglas, no hay nada que uno pueda leer
para entender qué aprendió. Lo único que hay es el dataset. Por eso se dice que es un método
**basado en instancias**.

**La distancia es el corazón del modelo.** La habitual es la **euclidiana** —la distancia en línea
recta—, que es la que usa scikit-learn por defecto. También existen la de **Manhattan** (como
moverse por una cuadrícula, solo en ángulos rectos) y la de **Minkowski**, que generaliza a las
dos con un parámetro `p`: con `p=1` es Manhattan, con `p=2` es euclidiana.

**El escalado no es opcional acá, es imprescindible.** Si una variable va de 0 a 1 y otra de 0 a
100.000, la segunda domina por completo el cálculo de la distancia: no porque sea más importante
sino porque sus números son más grandes. Es un sesgo puramente artificial. En nuestro dataset las
siete variables ya están en el mismo rango, así que el efecto va a ser chico —pero lo vamos a
medir igual, que es distinto de suponerlo—.

### Elegir *k* es el compromiso central del modelo

Y es un ejemplo perfecto de lo que vimos en el punto 5:

- ***k* muy bajo** (por ejemplo 1): el modelo decide en base a un único vecino, que puede ser
  atípico o estar mal etiquetado. Es muy sensible al ruido y tiende al **sobreajuste**. Con *k*=1
  el vecino más cercano de cada caso de entrenamiento es él mismo, así que el rendimiento en
  entrenamiento es perfecto por construcción: el caso más puro de memorización que existe.
- ***k* muy alto** (por ejemplo 50): gana estabilidad pero pierde capacidad de captar patrones
  locales. En el extremo, con *k* igual al tamaño del dataset, predice siempre la clase
  mayoritaria. Tiende al **subajuste**.

Dicho en los términos del 5.2: *k* es el mando del compromiso sesgo–varianza de este modelo.

No hay un valor universal. En general entre 3 y 15 funciona bien, pero lo correcto es probarlo y
medirlo. Conviene además usar valores **impares** en clasificación binaria, para que no haya
empates.

### Los hiperparámetros que importan

| Parámetro | Qué controla | Efecto de subirlo |
|---|---|---|
| `n_neighbors` (*k*) | Cuántos vecinos votan | Más estabilidad, menos detalle local. Es el mando de la complejidad |
| `weights` | Si todos los vecinos pesan igual (`"uniform"`) o según su distancia (`"distance"`) | `"distance"` le da más voz al vecino más cercano; suele ayudar con *k* grande |
| `metric` / `p` | Cómo se mide la distancia | `p=1` Manhattan, `p=2` euclidiana |
| `algorithm` | La estructura de datos para buscar vecinos (`ball_tree`, `kd_tree`, `brute`) | No cambia el resultado, solo la velocidad |

### Ventajas y desventajas

| A favor | En contra |
|---|---|
| No supone ninguna forma para la frontera: la dibuja como los datos lo pidan | Predecir es caro: hay que medir contra todo el dataset |
| Entrena instantáneamente | Hay que guardar el dataset entero para poder predecir |
| Fácil de explicar: "se parece a estos cinco casos" | No produce ningún modelo legible ni importancia de variables |
| Funciona en problemas multiclase sin cambios | **Imprescindible** escalar |
| Un solo hiperparámetro importante | Sufre la maldición de la dimensionalidad |

Sobre esa última fila: con muchas variables, todos los puntos terminan más o menos igual de lejos
entre sí y la noción de "vecino cercano" pierde sentido. KNN funciona bien con pocas variables
—como acá, que son siete— y se degrada rápido cuando son decenas.

### Dónde se usa en la práctica

Sistemas de recomendación sencillos ("usuarios parecidos al tuyo eligieron esto"), imputación de
valores faltantes (`KNNImputer` completa un dato con el promedio de sus vecinos), detección de
anomalías —un caso cuyo vecino más cercano está muy lejos es raro por definición—, y
reconocimiento de patrones con pocas variables bien elegidas.

### Cómo se sobreajusta este modelo

Con *k* chico, y de la manera más literal posible. La celda siguiente lo mide: buscamos el mejor
*k* con validación cruzada entre 1 y 31, y graficamos la brecha para todos ellos. Es la **forma 2**
de detección del punto 5.4, aplicada a un hiperparámetro que no es la profundidad de un árbol.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

valores_k = list(range(1, 32, 2))     # solo impares, para evitar empates
resultados_k = []

for k in valores_k:
    pipeline_k = Pipeline([
        ("escalador", StandardScaler()),
        ("modelo", KNeighborsClassifier(n_neighbors=k)),
    ])
    detalle_k = cross_validate(pipeline_k, X, y, cv=cv,
                               scoring=["f1_macro", "roc_auc"],
                               return_train_score=True, n_jobs=1)
    resultados_k.append({
        "k": k,
        "f1_entrena": detalle_k["train_f1_macro"].mean(),
        "f1_validacion": detalle_k["test_f1_macro"].mean(),
        "f1_desvio": detalle_k["test_f1_macro"].std(),
        "roc_auc": detalle_k["test_roc_auc"].mean(),
    })

resultados_k = pd.DataFrame(resultados_k)
mejor_k = int(resultados_k.loc[resultados_k["f1_validacion"].idxmax(), "k"])

print("=" * 78)
print("  BÚSQUEDA DEL MEJOR k  (validación cruzada de 5 pliegues)")
print("=" * 78)
display(resultados_k)
print(f"  Mejor k por F1 macro en validación : {mejor_k}")
print(f"  F1 macro alcanzado                 : {resultados_k['f1_validacion'].max():.4f}")

fig, ejes = plt.subplots(1, 2, figsize=(14, 4.4))

ejes[0].plot(resultados_k["k"], resultados_k["f1_entrena"], "o-", color="#d62728",
             linewidth=2, label="entrenamiento")
ejes[0].plot(resultados_k["k"], resultados_k["f1_validacion"], "s-", color="#ff7f0e",
             linewidth=2, label="validación cruzada")
ejes[0].fill_between(resultados_k["k"],
                     resultados_k["f1_validacion"] - resultados_k["f1_desvio"],
                     resultados_k["f1_validacion"] + resultados_k["f1_desvio"],
                     color="#ff7f0e", alpha=0.15)
ejes[0].axvline(mejor_k, color="green", linestyle="--", linewidth=1.6,
                label=f"mejor k = {mejor_k}")
ejes[0].set_xlabel("k (cantidad de vecinos)")
ejes[0].set_ylabel("F1 macro")
ejes[0].set_title("Elegir k: sobreajuste a la izquierda, subajuste a la derecha")
ejes[0].legend(fontsize=9)

brecha_k = resultados_k["f1_entrena"] - resultados_k["f1_validacion"]
ejes[1].plot(resultados_k["k"], brecha_k, "o-", color="#9467bd", linewidth=2)
ejes[1].axhline(BRECHA_SOBREAJUSTE, color="#d62728", linestyle="--", linewidth=1.4,
                label=f"umbral de sobreajuste = {BRECHA_SOBREAJUSTE}")
ejes[1].axvline(mejor_k, color="green", linestyle="--", linewidth=1.6)
ejes[1].set_xlabel("k (cantidad de vecinos)")
ejes[1].set_ylabel("brecha entrenamiento - validación")
ejes[1].set_title("La brecha se cierra al aumentar k")
ejes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print()
print(f"  Con k = 1 la brecha es de {brecha_k.iloc[0]:.4f}: el modelo acierta casi todo")
print("  en entrenamiento porque el vecino más cercano de cada caso es él mismo.")
print("  Es el ejemplo más puro de sobreajuste que se puede construir.")
print("  Al aumentar k la brecha se cierra y el modelo empieza a generalizar.")

Con *k* elegido por validación cruzada, registramos el modelo. Su gráfico propio es justamente esa
curva, que deja registrado de dónde salió el valor.

In [ ]:
def grafico_curva_k(pipeline_ajustado, color):
    k_usado = pipeline_ajustado.named_steps["modelo"].n_neighbors

    fig, eje = plt.subplots(figsize=(10, 4.2))
    eje.plot(resultados_k["k"], resultados_k["f1_validacion"], "o-", color=color,
             linewidth=2.2, label="F1 macro en validación cruzada")
    eje.fill_between(resultados_k["k"],
                     resultados_k["f1_validacion"] - resultados_k["f1_desvio"],
                     resultados_k["f1_validacion"] + resultados_k["f1_desvio"],
                     color=color, alpha=0.18, label="± 1 desvío entre pliegues")
    eje.axvline(k_usado, color="green", linestyle="--", linewidth=1.8,
                label=f"k elegido = {k_usado}")
    eje.set_xlabel("k (cantidad de vecinos)")
    eje.set_ylabel("F1 macro")
    eje.set_title("De dónde salió el k de este modelo")
    eje.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

    print(f"  k = {k_usado}, elegido por validación cruzada entre "
          f"{min(valores_k)} y {max(valores_k)}.")
    print("  La banda sombreada muestra que varios valores de k vecinos entre sí dan")
    print("  resultados que se solapan: la elección exacta no es crítica mientras se")
    print("  esté en la zona buena de la curva.")


registrar_modelo(
    f"KNN (k={mejor_k})",
    KNeighborsClassifier(n_neighbors=mejor_k),
    color="#ff7f0e",
    nota="Decide por proximidad; frontera flexible; el escalado es imprescindible.",
    grafico_extra=grafico_curva_k,
)

nombre_knn = f"KNN (k={mejor_k})"
tabla_knn = evaluar_modelo(nombre_knn)

---
<a id="sec-10"></a>
# 10. Modelo 3 — Árbol de Decisión

**Cómo funciona.** Un árbol de decisión aprende una secuencia de preguntas del tipo *"¿esta
variable es mayor que tal valor?"*. Cada pregunta parte los datos en dos, y el proceso se repite
sobre cada mitad hasta llegar a grupos lo más homogéneos posible en cuanto a la clase.

El resultado es literalmente un árbol: una raíz, nodos internos que preguntan, y **hojas** que
responden. Para clasificar un caso nuevo se lo hace caer desde la raíz respondiendo cada pregunta,
y la hoja donde termina da la predicción.

### Cómo elige cada corte

En cada nodo el algoritmo prueba **todas las variables y todos los umbrales posibles**, y se queda
con el corte que más "ordena" los datos. Ordenar se mide con una función de **impureza**:

| Criterio | Qué mide | Fórmula | Vale 0 cuando |
|---|---|---|---|
| **Gini** (por defecto) | La probabilidad de clasificar mal un caso tomado al azar | $1 - \sum p_i^2$ | El nodo tiene una sola clase |
| **Entropía** | La incertidumbre del nodo, en bits | $-\sum p_i \log_2 p_i$ | El nodo tiene una sola clase |

Los dos dan resultados casi idénticos en la práctica; Gini es un poco más rápido porque no calcula
logaritmos. El corte elegido es el que produce la mayor **reducción de impureza**, pesada por
cuántos casos van a cada lado. Ese mismo número, acumulado a lo largo de todo el árbol, es lo que
después se reporta como **importancia de la variable**.

Es un algoritmo **voraz**: elige el mejor corte en cada paso sin mirar a futuro. No garantiza el
mejor árbol posible, pero encontrar ese árbol óptimo es computacionalmente inviable.

### Qué aporta que los otros no tienen

- **Captura relaciones no lineales e interacciones** entre variables sin que haya que decírselo.
  Si la clase depende de "el tópico 3 es alto **y además** el tópico 5 es bajo", el árbol lo
  encuentra solo; la regresión logística necesitaría que le fabricáramos esa variable.
- **Es completamente legible.** Se puede dibujar el árbol entero y seguir el camino de cualquier
  caso hasta su predicción. No es una caja negra, y por eso lo vamos a graficar.
- **No necesita escalado**, y eso nos sirve como control: si al aplicarle Min-Max o Z-score el
  resultado cambiara aunque sea un poco, habría un error en nuestra maquinaria de evaluación. Un
  árbol corta por umbrales, y las dos transformaciones son monótonas —no cambian el orden de los
  valores—, así que los cortes deben caer exactamente en los mismos lugares.
- **Tolera variables irrelevantes**: simplemente no las usa, y se nota en la importancia.

### Su gran debilidad: la inestabilidad

Un árbol crecido sin restricciones memoriza el entrenamiento, como vamos a ver en 5.7. Y es muy
sensible: cambiando unos pocos casos, el árbol resultante puede ser **completamente distinto** —no
un poco distinto: otro árbol, con otra variable en la raíz—. En los términos del punto 5.2, es un
modelo de **varianza muy alta**.

Eso tiene una consecuencia incómoda para la interpretabilidad: el árbol es legible, sí, pero lo
que se lee puede ser un accidente de la muestra. Un árbol entrenado con otras 171 filas contaría
otra historia igual de convincente.

Por eso hay que **podarlo**, limitando su complejidad. Y por eso existe el Random Forest del punto
siguiente, que es la respuesta directa a este problema.

### Los hiperparámetros que importan

Todos son formas de podar, es decir, de frenar el crecimiento:

| Parámetro | Qué controla | Por defecto |
|---|---|---|
| `max_depth` | Cuántas preguntas encadenadas como máximo | `None`: **sin límite** |
| `min_samples_leaf` | Cuántos casos tiene que tener una hoja como mínimo | 1 |
| `min_samples_split` | Cuántos casos hacen falta para partir un nodo | 2 |
| `max_features` | Cuántas variables se consideran en cada corte | todas |
| `criterion` | Gini o entropía | `"gini"` |
| `ccp_alpha` | Poda posterior por costo-complejidad | 0: sin poda |
| `class_weight` | Peso de cada clase | ninguno |

**Los valores por defecto sobreajustan.** Un `DecisionTreeClassifier()` recién creado crece hasta
que cada hoja sea pura, aunque eso signifique una hoja por caso. Es el único de nuestros cuatro
modelos que viene de fábrica en el extremo de la complejidad, y por eso es el único al que le
buscamos los hiperparámetros.

### Ventajas y desventajas

| A favor | En contra |
|---|---|
| Legible de punta a punta | Muy inestable: chico cambio en los datos, otro árbol |
| Capta no linealidades e interacciones | Sobreajusta por defecto |
| Indiferente a la escala | Fronteras siempre perpendiculares a los ejes: una diagonal la aproxima con escalones |
| Maneja variables numéricas y categóricas | Tiende a favorecer variables con muchos valores distintos |
| Rápido de entrenar y de predecir | Rara vez es el mejor modelo por sí solo |

### Dónde se usa en la práctica

Como modelo final, sobre todo donde la decisión tiene que poder explicarse en una reunión:
protocolos de triage médico, reglas de negocio, criterios de elegibilidad. Y como **componente**,
que es su uso más frecuente hoy: los árboles son los ladrillos del Random Forest, del Gradient
Boosting y de XGBoost, que son los modelos que suelen ganar en datos tabulares.

### Cómo buscamos la poda

Buscamos los hiperparámetros con `GridSearchCV`, que prueba todas las combinaciones de una grilla
evaluando cada una con validación cruzada.

**Una advertencia honesta.** Esto es exactamente la forma 4 de fuga del punto 4.3: estamos
eligiendo hiperparámetros con la misma validación cruzada con la que después vamos a evaluar el
modelo. Lo hacemos igual, porque la alternativa rigurosa —validación cruzada anidada— duplicaría
la complejidad del cuaderno. Pero queda anotado, y vuelve a aparecer en las limitaciones del punto
12.8: las métricas del árbol y de KNN son algo más optimistas que las de la logística y el bosque,
que no se ajustaron.

In [ ]:
from sklearn.tree import plot_tree
from sklearn.model_selection import GridSearchCV
# DecisionTreeClassifier ya se importó en el punto 5.7

grilla_arbol = {
    "max_depth": [2, 3, 4, 5, 6, 8, None],
    "min_samples_leaf": [1, 3, 5, 8, 12],
    "criterion": ["gini", "entropy"],
}

busqueda = GridSearchCV(
    DecisionTreeClassifier(random_state=SEMILLA),
    grilla_arbol,
    scoring="f1_macro",
    cv=cv,
    n_jobs=1,
    return_train_score=True,
)
busqueda.fit(X, y)

print("=" * 78)
print("  BÚSQUEDA DE HIPERPARÁMETROS DEL ÁRBOL")
print("=" * 78)
print(f"  Combinaciones evaluadas : {len(busqueda.cv_results_['params'])}")
print(f"  Ajustes realizados      : {len(busqueda.cv_results_['params']) * N_PLIEGUES}")
print(f"  Mejores parámetros      : {busqueda.best_params_}")
print(f"  F1 macro en validación  : {busqueda.best_score_:.4f}")

resultados_grilla = pd.DataFrame(busqueda.cv_results_)
resultados_grilla["brecha"] = (resultados_grilla["mean_train_score"] -
                               resultados_grilla["mean_test_score"])
mejores = resultados_grilla.nlargest(8, "mean_test_score")[
    ["param_max_depth", "param_min_samples_leaf", "param_criterion",
     "mean_train_score", "mean_test_score", "std_test_score", "brecha"]
]
print()
print("  Las ocho mejores combinaciones:")
display(mejores.reset_index(drop=True))

peores = resultados_grilla.nlargest(5, "brecha")[
    ["param_max_depth", "param_min_samples_leaf", "mean_train_score",
     "mean_test_score", "brecha"]
]
print("  Las cinco combinaciones con MÁS sobreajuste:")
display(peores.reset_index(drop=True))
print("  Son las más complejas: max_depth alto y min_samples_leaf bajo.")
print("  Mejor score de entrenamiento, peor de validación. Es el punto 5 en una tabla.")

Registramos el árbol con los parámetros encontrados. Su gráfico propio es el árbol dibujado más la
importancia de cada variable.

La **importancia** de una variable en un árbol es cuánto contribuyó a reducir la impureza a lo
largo de todos los cortes en los que participó. Es una medida distinta del coeficiente de la
logística —no tiene signo, no dice hacia qué clase empuja— pero responde la misma pregunta de
fondo: qué variables está usando el modelo.

In [ ]:
def grafico_arbol(pipeline_ajustado, color):
    modelo = pipeline_ajustado.named_steps["modelo"]

    importancias = pd.Series(modelo.feature_importances_, index=predictoras)
    importancias = importancias.sort_values(ascending=True)

    fig, eje = plt.subplots(figsize=(9, 4))
    eje.barh(importancias.index, importancias.values, color=color,
             edgecolor="black", linewidth=0.6)
    eje.set_xlabel("importancia (reducción de impureza)")
    eje.set_title("Qué variables usa el árbol")
    plt.tight_layout()
    plt.show()

    usadas = int((importancias > 0).sum())
    print(f"  El árbol usa {usadas} de las {len(predictoras)} variables disponibles.")
    print("  Variables ordenadas por importancia:")
    for variable, valor in importancias.sort_values(ascending=False).items():
        marca = "" if valor > 0 else "   (no usada)"
        print(f"    {variable:<12} {valor:>7.4f}{marca}")

    fig, eje = plt.subplots(figsize=(16, 7))
    plot_tree(modelo, feature_names=predictoras,
              class_names=[str(clase_negativa), str(CLASE_POSITIVA)],
              filled=True, rounded=True, fontsize=8, proportion=True, ax=eje)
    eje.set_title("El árbol completo  (cada caja: condición, impureza, casos, clase)",
                  fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()

    print(f"  Profundidad del árbol : {modelo.get_depth()}")
    print(f"  Hojas terminales      : {modelo.get_n_leaves()}")
    print()
    print("  Cómo leer cada caja: arriba la condición que se evalúa (si es verdadera se")
    print("  va a la izquierda), después la impureza, la proporción de casos que llegan")
    print("  a ese nodo, el reparto entre clases y la clase mayoritaria. El color indica")
    print("  qué clase domina y su intensidad, cuán pura es la hoja.")


registrar_modelo(
    "Árbol de Decisión",
    DecisionTreeClassifier(random_state=SEMILLA, **busqueda.best_params_),
    color="#2ca02c",
    nota="Reglas explícitas; capta no linealidades; indiferente a la escala.",
    grafico_extra=grafico_arbol,
)

tabla_arbol = evaluar_modelo("Árbol de Decisión")

---
<a id="sec-11"></a>
# 11. Modelo 4 — Random Forest

Este modelo entra por dos motivos a la vez: uno técnico y uno demostrativo.

## El motivo técnico

Un **Random Forest** (bosque aleatorio) es un **conjunto** (*ensemble*) de muchos árboles de
decisión que votan. Es la respuesta directa al problema que dejó planteado el punto 10: el árbol
capta bien las relaciones no lineales, pero es tan inestable que no se puede confiar en ninguno en
particular.

La idea de fondo es de sentido común: si un experto es bueno pero errático, no se le pregunta a
uno solo. Se le pregunta a cuatrocientos y se promedia. Los errores erráticos —que por definición
van para cualquier lado— se cancelan entre sí; el acierto sistemático, que va siempre para el
mismo lado, sobrevive al promedio.

### Las dos dosis de azar

Para que los árboles cometan errores **distintos** hay que hacerlos distintos, y el bosque usa dos
mecanismos:

**1. *Bagging* (muestreo bootstrap).** Cada árbol se entrena sobre una muestra distinta, tomada
del dataset **con reposición** y del mismo tamaño que el original. Como se saca con reposición,
algunas filas aparecen repetidas y otras no aparecen: en promedio, cada árbol ve cerca del 63 % de
las filas distintas. Las que quedan afuera se llaman ***out-of-bag*** y sirven para estimar el
rendimiento gratis, sin conjunto de prueba —es el parámetro `oob_score`—.

**2. Submuestreo de variables.** En **cada corte** de cada árbol, el algoritmo no puede elegir
entre todas las variables sino solo entre un subconjunto al azar. Esto parece un desperdicio, y es
lo que hace que el bosque funcione: sin esto, si hay una variable claramente mejor que las demás,
todos los árboles la elegirían en la raíz y terminarían pareciéndose mucho. Promediar modelos
parecidos no cancela nada. Forzar la variedad es lo que vuelve útil el promedio.

En los términos del punto 5.2: el bosque **conserva el bajo sesgo del árbol y le cancela la
varianza**. Es el único de los cuatro que mejora en las dos columnas a la vez.

### Por qué casi no sobreajusta

Es una propiedad que sorprende: **agregar árboles nunca empeora el modelo**. Más árboles es
siempre mejor o igual, y solo cuesta tiempo de cómputo. La razón es que los árboles no se suman,
se **promedian**: agregar uno más reduce la varianza del promedio, no aumenta la complejidad de la
función aprendida.

El "casi" existe igual. Un bosque puede sobreajustar si los datos tienen mucho ruido y los árboles
crecen sin límite, y si el dataset es muy chico el promedio de árboles que vieron casi las mismas
filas no cancela tanto como uno esperaría. Con 171 filas conviene no dar por sentado que es
inmune, y por eso lo pasamos por el mismo diagnóstico que a los demás.

Lo que sí pierde, y no es poco, es la legibilidad: 400 árboles no se leen. Lo único que queda es
la **importancia de las variables**, y ni siquiera eso es del todo confiable cuando hay variables
correlacionadas entre sí, porque la importancia se reparte entre ellas.

### Los hiperparámetros que importan

| Parámetro | Qué controla | Comentario |
|---|---|---|
| `n_estimators` | Cuántos árboles | Más es mejor o igual; solo cuesta tiempo. 100 a 500 es lo habitual |
| `max_features` | Cuántas variables se consideran en cada corte | Es **el** parámetro del bosque: controla cuán distintos son los árboles entre sí |
| `max_depth`, `min_samples_leaf` | La poda de cada árbol individual | Suele dejarse sin límite: el promedio ya controla la varianza |
| `bootstrap` | Si se muestrea con reposición | `True` por defecto; ponerlo en `False` desactiva el *bagging* |
| `oob_score` | Estimar el rendimiento con las filas que quedaron afuera | Una validación gratis, aunque optimista con datasets chicos |
| `n_jobs` | Cuántos núcleos usar | Los árboles son independientes: paraleliza perfecto |
| `class_weight` | Peso de cada clase | `"balanced"` para clases desbalanceadas |

### Ventajas y desventajas

| A favor | En contra |
|---|---|
| Suele ser el mejor de los cuatro sin ajustar nada | Deja de ser interpretable |
| Muy robusto al sobreajuste | Más lento de entrenar y de predecir |
| Indiferente a la escala | Ocupa mucha más memoria: hay que guardar todos los árboles |
| Da importancia de variables | La importancia se reparte mal entre variables correlacionadas |
| Tolera outliers y variables irrelevantes | No extrapola: nunca predice fuera del rango que vio |
| Paraleliza sin esfuerzo | Sus buenos resultados por defecto invitan a no pensar |

### Dónde se usa en la práctica

Es el caballo de batalla de los datos tabulares: detección de fraude, scoring de riesgo, predicción
de abandono de clientes, bioinformática, teledetección. En competencias y en producción suele ser
la primera línea de base seria que se prueba después de la logística, y en muchos casos es también
la última: la diferencia contra modelos más sofisticados no siempre justifica el trabajo extra.

## El motivo demostrativo

Mirá lo que cuesta agregar un modelo a este cuaderno:

```python
registrar_modelo("Random Forest", RandomForestClassifier(...), color="#9467bd")
evaluar_modelo("Random Forest")
```

Dos líneas, y el modelo queda con el mismo bloque de métricas que los otros tres y entra
automáticamente en toda la comparativa del punto 12. Eso es lo que compra la arquitectura del
punto 7.

In [ ]:
from sklearn.ensemble import RandomForestClassifier


def grafico_bosque(pipeline_ajustado, color):
    modelo = pipeline_ajustado.named_steps["modelo"]

    importancias = pd.DataFrame({
        "variable": predictoras,
        "importancia": modelo.feature_importances_,
        "desvio": np.std([arbol.feature_importances_ for arbol in modelo.estimators_], axis=0),
    }).sort_values("importancia", ascending=True)

    fig, eje = plt.subplots(figsize=(9, 4))
    eje.barh(importancias["variable"], importancias["importancia"],
             xerr=importancias["desvio"], color=color,
             edgecolor="black", linewidth=0.6, capsize=3, error_kw={"linewidth": 1})
    eje.set_xlabel("importancia media entre los árboles")
    eje.set_title("Importancia de las variables  (la barra de error es el desvío entre árboles)")
    plt.tight_layout()
    plt.show()

    print(f"  El bosque tiene {len(modelo.estimators_)} árboles.")
    print("  Variables ordenadas por importancia:")
    for _, fila in importancias.sort_values("importancia", ascending=False).iterrows():
        print(f"    {fila['variable']:<12} {fila['importancia']:>7.4f}  ± {fila['desvio']:.4f}")
    print()
    print("  Comparado con el árbol único del punto 10, el bosque reparte la")
    print("  importancia entre más variables: cada árbol ve solo un subconjunto en cada")
    print("  corte, así que las variables secundarias tienen oportunidad de mostrar lo")
    print("  que aportan.")


registrar_modelo(
    "Random Forest",
    RandomForestClassifier(n_estimators=400, random_state=SEMILLA, n_jobs=1),
    color="#9467bd",
    nota="Conjunto de árboles; corrige la inestabilidad del árbol único; no es legible.",
    grafico_extra=grafico_bosque,
)

tabla_bosque = evaluar_modelo("Random Forest")

---
<a id="sec-12"></a>
# 12. Comparativa exhaustiva y ranking

Todo lo evaluado hasta acá se fue acumulando en `RESULTADOS`. Esta sección lo analiza entero.

Un detalle de diseño que vale la pena señalar: **en ninguna celda del punto 12 hay una lista de
modelos escrita a mano**. Todo se arma a partir de `RESULTADOS`. Si agregás un quinto modelo y
volvés a correr esta sección, aparece solo en todas las tablas, en todos los gráficos y en el
ranking.

<a id="sec-12-1"></a>
## 12.1 Tabla maestra

Todas las combinaciones de modelo y escalado, con todas las métricas. Es la tabla de la que sale
todo lo demás.

In [ ]:
maestra = pd.DataFrame(RESULTADOS)

print("=" * 88)
print("  TABLA MAESTRA")
print("=" * 88)
print(f"  Modelos evaluados     : {maestra['modelo'].nunique()}")
print(f"  Escalados comparados  : {maestra['escalado'].nunique()}")
print(f"  Combinaciones totales : {len(maestra)}")
columnas_metricas = [c for c in maestra.columns
                     if c not in ("modelo", "escalado", "color", "diagnostico")]
print(f"  Métricas por fila     : {len(columnas_metricas)}")
print()

columnas_vista = ["modelo", "escalado", "accuracy", "balanced_accuracy", "precision",
                  "recall", "f1", "f1_macro", "roc_auc", "average_precision",
                  "mcc", "kappa", "log_loss", "brier",
                  "cv_f1_macro_desvio", "brecha_ajuste", "diagnostico"]

vista = maestra[columnas_vista].rename(columns={
    "cv_f1_macro_desvio": "desvio_pliegues", "brecha_ajuste": "brecha",
})
display(vista.sort_values("roc_auc", ascending=False).reset_index(drop=True))

<a id="sec-12-2"></a>
## 12.2 La mejor variante de cada modelo

De las tres variantes de escalado de cada modelo nos quedamos con la mejor por ROC-AUC. Esa es la
que representa al modelo en el ranking final.

Usamos ROC-AUC como criterio de selección porque **no depende del umbral de corte**: mide la
capacidad del modelo de ordenar los casos, con independencia de dónde se ponga la línea entre una
clase y la otra. Eso lo vuelve la métrica más estable para comparar familias de modelos distintas.

In [ ]:
mejores_por_modelo = (maestra.sort_values("roc_auc", ascending=False)
                             .groupby("modelo", as_index=False)
                             .first())
mejores_por_modelo = (mejores_por_modelo.sort_values("roc_auc", ascending=False)
                                        .reset_index(drop=True))

print("=" * 88)
print("  LA MEJOR VARIANTE DE CADA MODELO")
print("=" * 88)
display(mejores_por_modelo[[
    "modelo", "escalado", "accuracy", "balanced_accuracy", "f1_macro",
    "roc_auc", "mcc", "cv_f1_macro_media", "cv_f1_macro_desvio", "brecha_ajuste",
]].rename(columns={"cv_f1_macro_media": "f1_cv_media",
                   "cv_f1_macro_desvio": "f1_cv_desvio",
                   "brecha_ajuste": "brecha"}))

print("  Línea de base (responder siempre la clase mayoritaria):")
print(f"    accuracy = {linea_base:.4f},  ROC-AUC = 0.5000")
print()
for _, fila in mejores_por_modelo.iterrows():
    mejora = fila["accuracy"] - linea_base
    print(f"    {fila['modelo']:<24} accuracy {fila['accuracy']:.4f}  "
          f"({mejora:+.4f} sobre la línea de base)")

<a id="sec-12-3"></a>
## 12.3 Curvas ROC superpuestas

La **curva ROC** se construye barriendo todos los umbrales de corte posibles. Para cada uno se
calcula la tasa de verdaderos positivos —qué proporción de los positivos reales detecta— contra la
tasa de falsos positivos —qué proporción de los negativos clasifica mal—. El **área bajo la curva**
(AUC) resume todo ese barrido en un número.

Su interpretación más intuitiva: el AUC es la probabilidad de que, tomando al azar un caso positivo
y uno negativo, el modelo le asigne mayor puntaje al positivo. 0,5 es azar puro; 1 es perfecto.

Superponerlas permite ver **dónde** un modelo es mejor que otro. Dos modelos pueden tener un AUC
casi idéntico y curvas de forma muy distinta: uno mejor en la zona de umbrales exigentes y otro en
la de umbrales permisivos.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(14, 5.6))

for _, fila in mejores_por_modelo.iterrows():
    clave = (fila["modelo"], fila["escalado"])
    proba = PREDICCIONES[clave]["y_proba"]

    fpr, tpr, _ = roc_curve(y, proba)
    ejes[0].plot(fpr, tpr, linewidth=2.3, color=fila["color"],
                 label=f"{fila['modelo']} — AUC {fila['roc_auc']:.4f}")

    prec, rec, _ = precision_recall_curve(y, proba)
    ejes[1].plot(rec, prec, linewidth=2.3, color=fila["color"],
                 label=f"{fila['modelo']} — AP {fila['average_precision']:.4f}")

ejes[0].plot([0, 1], [0, 1], "k--", linewidth=1.2, label="azar (AUC = 0,500)")
ejes[0].set_xlabel("tasa de falsos positivos")
ejes[0].set_ylabel("tasa de verdaderos positivos")
ejes[0].set_title("Curvas ROC")
ejes[0].legend(loc="lower right", fontsize=9)

ejes[1].axhline(y.mean(), color="black", linestyle="--", linewidth=1.2,
                label=f"azar ({y.mean():.3f})")
ejes[1].set_xlabel("recall")
ejes[1].set_ylabel("precisión")
ejes[1].set_title("Curvas precisión-recall")
ejes[1].legend(loc="lower left", fontsize=9)

plt.tight_layout()
plt.show()

<a id="sec-12-4"></a>
## 12.4 El efecto de la normalización

La pregunta que respondemos acá: **¿cuál de los dos escalados conviene, y a qué modelos les
importa?**

Lo esperable, según lo explicado en 2.3:

- La **regresión logística** y **KNN** deberían mostrar alguna diferencia entre escalados.
- El **árbol** y el **bosque** deberían ser exactamente indiferentes. Si alguno de los dos mostrara
  diferencia, habría un error en la maquinaria.

In [ ]:
pivote_auc = maestra.pivot_table(index="modelo", columns="escalado", values="roc_auc")
pivote_f1 = maestra.pivot_table(index="modelo", columns="escalado", values="f1_macro")
orden_escalados = [e for e in ESCALADORES if e in pivote_auc.columns]
pivote_auc = pivote_auc[orden_escalados]
pivote_f1 = pivote_f1[orden_escalados]

print("=" * 88)
print("  ROC-AUC POR MODELO Y ESCALADO")
print("=" * 88)
display(pivote_auc)

print("=" * 88)
print("  F1 MACRO POR MODELO Y ESCALADO")
print("=" * 88)
display(pivote_f1)

fig, ejes = plt.subplots(1, 2, figsize=(15, 5))

pivote_auc.plot(kind="bar", ax=ejes[0], edgecolor="black", linewidth=0.6, width=0.78)
ejes[0].set_ylabel("ROC-AUC")
ejes[0].set_title("ROC-AUC por modelo y escalado")
ejes[0].set_ylim(max(0, pivote_auc.to_numpy().min() - 0.06), 1.0)
ejes[0].legend(title="escalado", fontsize=9)
ejes[0].tick_params(axis="x", rotation=18)
ejes[0].set_xlabel("")

sensibilidad = (pivote_auc.max(axis=1) - pivote_auc.min(axis=1)).sort_values(ascending=True)
colores_sens = [maestra.loc[maestra["modelo"] == m, "color"].iloc[0] for m in sensibilidad.index]
ejes[1].barh(sensibilidad.index, sensibilidad.values, color=colores_sens,
             edgecolor="black", linewidth=0.6)
ejes[1].set_xlabel("diferencia de ROC-AUC entre el mejor y el peor escalado")
ejes[1].set_title("Cuánto le importa el escalado a cada modelo")

plt.tight_layout()
plt.show()

print("=" * 88)
print("  LECTURA")
print("=" * 88)
for modelo in sensibilidad.sort_values(ascending=False).index:
    rango = sensibilidad[modelo]
    mejor_esc = pivote_auc.loc[modelo].idxmax()
    if rango < 1e-9:
        print(f"    {modelo:<24} INDIFERENTE al escalado (diferencia exacta de 0).")
    else:
        print(f"    {modelo:<24} diferencia de {rango:.4f}; el mejor es {mejor_esc}.")

print()
modelos_sensibles = sensibilidad[sensibilidad > 1e-9].index.tolist()
if modelos_sensibles:
    solo_sensibles = maestra[maestra["modelo"].isin(modelos_sensibles)]
    ganadores = solo_sensibles.loc[solo_sensibles.groupby("modelo")["roc_auc"].idxmax(),
                                   "escalado"]
    print("  Escalado preferido, contando solo los modelos a los que les importa:")
    for escalado, cuenta in ganadores.value_counts().items():
        print(f"    {escalado:<14} {cuenta} de {len(modelos_sensibles)} modelos sensibles")
    print()
    print("  ('Sin escalar' tiene ventaja acá porque las siete variables ya están en")
    print("   el mismo rango: todas son proporciones entre 0 y 1. Por eso el escalado")
    print("   tiene poco margen de mejora. En un dataset con variables en escalas muy")
    print("   distintas, el resultado sería muy diferente.)")
else:
    print("  Ningún modelo resultó sensible al escalado.")

print()
print("  Conviene no sobreinterpretar: en los modelos donde la diferencia es")
print("  menor que el desvío entre pliegues, la elección del escalado no está")
print("  demostrada por los datos. Lo que sí está demostrado es cuándo NO importa.")

<a id="sec-12-5"></a>
## 12.5 Estabilidad entre pliegues

Una media sola no alcanza para elegir un modelo. Dos modelos pueden tener el mismo F1 promedio y
comportarse de manera muy distinta: uno consistente en los cinco pliegues, otro oscilando entre
resultados muy buenos y muy malos.

El **desvío entre pliegues** mide esa consistencia. Es la forma 3 de detección del punto 5.4, y es
la información que permite decir si la diferencia entre dos modelos es real o cabe dentro del ruido
de la medición. Si dos modelos difieren en 0,01 de F1 y el desvío de cada uno es 0,05, esa
diferencia no significa nada.

In [ ]:
estabilidad = mejores_por_modelo[[
    "modelo", "escalado", "color",
    "cv_f1_macro_media", "cv_f1_macro_desvio", "cv_f1_macro_min", "cv_f1_macro_max",
    "cv_roc_auc_media", "cv_roc_auc_desvio",
]].copy()
estabilidad["amplitud"] = estabilidad["cv_f1_macro_max"] - estabilidad["cv_f1_macro_min"]
estabilidad = estabilidad.sort_values("cv_f1_macro_media", ascending=False)

print("=" * 88)
print("  ESTABILIDAD ENTRE PLIEGUES")
print("=" * 88)
display(estabilidad.drop(columns="color").rename(columns={
    "cv_f1_macro_media": "f1_media", "cv_f1_macro_desvio": "f1_desvio",
    "cv_f1_macro_min": "f1_min", "cv_f1_macro_max": "f1_max",
    "cv_roc_auc_media": "auc_media", "cv_roc_auc_desvio": "auc_desvio",
}).reset_index(drop=True))

fig, ejes = plt.subplots(1, 2, figsize=(15, 5))

posiciones = np.arange(len(estabilidad))
ejes[0].barh(posiciones, estabilidad["cv_f1_macro_media"],
             xerr=estabilidad["cv_f1_macro_desvio"],
             color=estabilidad["color"], edgecolor="black", linewidth=0.6,
             capsize=5, error_kw={"linewidth": 1.4, "ecolor": "black"})
ejes[0].set_yticks(posiciones)
ejes[0].set_yticklabels(estabilidad["modelo"])
ejes[0].set_xlabel("F1 macro (media ± desvío entre pliegues)")
ejes[0].set_title("Rendimiento con su margen de error")
ejes[0].set_xlim(max(0, (estabilidad["cv_f1_macro_media"] -
                         estabilidad["cv_f1_macro_desvio"]).min() - 0.08), 1.02)
ejes[0].invert_yaxis()

datos_caja, etiquetas, colores_caja = [], [], []
for _, fila in estabilidad.iterrows():
    clave = (fila["modelo"], fila["escalado"])
    datos_caja.append(PREDICCIONES[clave]["por_pliegue"]["test_f1_macro"])
    etiquetas.append(fila["modelo"])
    colores_caja.append(fila["color"])

caja = ejes[1].boxplot(datos_caja, tick_labels=etiquetas, patch_artist=True, widths=0.55)
for parche, color in zip(caja["boxes"], colores_caja):
    parche.set_facecolor(color)
    parche.set_alpha(0.65)
for mediana in caja["medians"]:
    mediana.set_color("black")
    mediana.set_linewidth(1.8)
ejes[1].set_ylabel("F1 macro por pliegue")
ejes[1].set_title("Distribución del rendimiento entre los 5 pliegues")
ejes[1].tick_params(axis="x", rotation=18)

plt.tight_layout()
plt.show()

mas_estable = estabilidad.loc[estabilidad["cv_f1_macro_desvio"].idxmin()]
menos_estable = estabilidad.loc[estabilidad["cv_f1_macro_desvio"].idxmax()]
print(f"  Más estable  : {mas_estable['modelo']} "
      f"(desvío {mas_estable['cv_f1_macro_desvio']:.4f})")
print(f"  Menos estable: {menos_estable['modelo']} "
      f"(desvío {menos_estable['cv_f1_macro_desvio']:.4f})")
print()
desvio_tipico = estabilidad["cv_f1_macro_desvio"].median()
print(f"  Desvío típico entre pliegues: {desvio_tipico:.4f}")
print(f"  REGLA PRÁCTICA: dos modelos que difieran en menos de {desvio_tipico:.4f} de F1")
print("  macro no se pueden distinguir con estos datos. La diferencia entra dentro")
print("  del ruido de la medición.")

<a id="sec-12-6"></a>
## 12.6 Diagnóstico de ajuste comparado

La **brecha** entre el rendimiento en entrenamiento y el de validación, para los cuatro modelos a
la vez. Es el resumen de todo lo que se discutió en el punto 5, aplicado a los modelos reales.

Un modelo con brecha grande puede tener buen rendimiento en validación y ser igualmente una mala
elección: significa que está memorizando y que con datos nuevos, distintos de estos, va a caer más
que uno con brecha chica.

In [ ]:
ajuste = mejores_por_modelo[[
    "modelo", "escalado", "color",
    "cv_f1_macro_entrena", "cv_f1_macro_media", "brecha_ajuste", "cv_f1_macro_desvio",
    "diagnostico",
]].copy().sort_values("brecha_ajuste", ascending=True)

print("=" * 88)
print("  DIAGNÓSTICO DE AJUSTE")
print("=" * 88)
display(ajuste.drop(columns="color").rename(columns={
    "cv_f1_macro_entrena": "f1_entrenamiento",
    "cv_f1_macro_media": "f1_validacion",
    "brecha_ajuste": "brecha",
    "cv_f1_macro_desvio": "desvio",
}).reset_index(drop=True))

fig, ejes = plt.subplots(1, 2, figsize=(15, 5))

posiciones = np.arange(len(ajuste))
ancho = 0.38
ejes[0].bar(posiciones - ancho/2, ajuste["cv_f1_macro_entrena"], ancho,
            label="entrenamiento", color="#d62728", alpha=0.8,
            edgecolor="black", linewidth=0.6)
ejes[0].bar(posiciones + ancho/2, ajuste["cv_f1_macro_media"], ancho,
            label="validación", color="#1f77b4", alpha=0.9,
            edgecolor="black", linewidth=0.6)
ejes[0].set_xticks(posiciones)
ejes[0].set_xticklabels(ajuste["modelo"], rotation=18, ha="right")
ejes[0].set_ylabel("F1 macro")
ejes[0].set_title("Entrenamiento vs. validación")
ejes[0].set_ylim(0, 1.1)
ejes[0].legend(fontsize=9)

colores_brecha = ["#2ca02c" if b <= BRECHA_SOBREAJUSTE / 2
                  else "#ff7f0e" if b <= BRECHA_SOBREAJUSTE
                  else "#d62728" for b in ajuste["brecha_ajuste"]]
ejes[1].barh(ajuste["modelo"], ajuste["brecha_ajuste"], color=colores_brecha,
             edgecolor="black", linewidth=0.6)
ejes[1].axvline(BRECHA_SOBREAJUSTE, color="#d62728", linestyle="--", linewidth=1.6,
                label=f"umbral de sobreajuste ({BRECHA_SOBREAJUSTE})")
ejes[1].axvline(BRECHA_SOBREAJUSTE / 2, color="#ff7f0e", linestyle=":", linewidth=1.4,
                label=f"aviso leve ({BRECHA_SOBREAJUSTE/2})")
ejes[1].set_xlabel("brecha (entrenamiento - validación)")
ejes[1].set_title("Cuánto memoriza cada modelo")
ejes[1].legend(fontsize=9)
ejes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print("=" * 88)
print("  VEREDICTO POR MODELO")
print("=" * 88)
for _, fila in ajuste.iterrows():
    clave = (fila["modelo"], fila["escalado"])
    print(f"  {fila['modelo']}  ({fila['escalado']})")
    for aviso in PREDICCIONES[clave]["avisos"]:
        print(f"      {aviso}")
    print()

<a id="sec-12-7"></a>
## 12.7 Ranking final

Para ordenar los modelos hace falta un criterio, y el criterio hay que declararlo **antes** de
mirar los resultados —si no, es muy fácil elegir el que favorece al modelo que uno ya prefería—.

**El criterio: un puntaje compuesto de cuatro métricas, con pesos.**

| Componente | Peso | Por qué |
|---|---|---|
| **ROC-AUC** | 35% | Mide la capacidad de ordenar los casos sin depender del umbral. Es la métrica más estable para comparar familias distintas de modelos |
| **F1 macro** | 30% | Captura el desempeño efectivo con el umbral de 0,5, tratando a las dos clases por igual |
| **MCC** | 20% | Usa las cuatro celdas de la matriz de confusión y es la más honesta cuando las clases están desbalanceadas |
| **Estabilidad** | 15% | `1 - desvío entre pliegues`. Penaliza los modelos cuyo rendimiento depende de qué datos les toquen |

Los tres primeros componentes miden **cuán bien predice**; el cuarto, **cuánto se puede confiar en
esa medición**. Un modelo apenas mejor pero mucho más inestable no es una mejor elección.

Los pesos son una decisión de diseño, no una verdad. Están escritos en la celda siguiente y se
pueden cambiar: el ranking se recalcula solo.

In [ ]:
PESOS_RANKING = {
    "roc_auc": 0.35,
    "f1_macro": 0.30,
    "mcc": 0.20,
    "estabilidad": 0.15,
}

ranking = mejores_por_modelo.copy()
ranking["estabilidad"] = 1 - ranking["cv_f1_macro_desvio"]
# El MCC va de -1 a +1: lo llevamos a 0-1 para que sea comparable con las demás
ranking["mcc_normalizado"] = (ranking["mcc"] + 1) / 2

ranking["puntaje"] = (
    PESOS_RANKING["roc_auc"] * ranking["roc_auc"] +
    PESOS_RANKING["f1_macro"] * ranking["f1_macro"] +
    PESOS_RANKING["mcc"] * ranking["mcc_normalizado"] +
    PESOS_RANKING["estabilidad"] * ranking["estabilidad"]
)

ranking = ranking.sort_values("puntaje", ascending=False).reset_index(drop=True)
ranking.insert(0, "puesto", range(1, len(ranking) + 1))

print("=" * 96)
print("  RANKING FINAL DE MODELOS")
print("=" * 96)
display(ranking[[
    "puesto", "modelo", "escalado", "puntaje",
    "roc_auc", "f1_macro", "mcc", "cv_f1_macro_desvio",
    "accuracy", "balanced_accuracy", "brecha_ajuste",
]].rename(columns={"cv_f1_macro_desvio": "desvio", "brecha_ajuste": "brecha"}))

print()
for _, fila in ranking.iterrows():
    medalla = {1: "1ro", 2: "2do", 3: "3ro"}.get(fila["puesto"], f"{fila['puesto']}to")
    print(f"  {medalla:>4}  {fila['modelo']:<24} puntaje {fila['puntaje']:.4f}   "
          f"(AUC {fila['roc_auc']:.4f} | F1 {fila['f1_macro']:.4f} | "
          f"MCC {fila['mcc']:.4f} | desvío {fila['cv_f1_macro_desvio']:.4f})")

print()
lider = ranking.iloc[0]
print("  Diferencia del primero contra los demás:")
for _, fila in ranking.iloc[1:].iterrows():
    dif_auc = lider["roc_auc"] - fila["roc_auc"]
    dif_f1 = lider["f1_macro"] - fila["f1_macro"]
    ruido = max(lider["cv_f1_macro_desvio"], fila["cv_f1_macro_desvio"])
    veredicto = "diferencia DEMOSTRADA" if dif_f1 > ruido else "diferencia dentro del ruido"
    print(f"    vs. {fila['modelo']:<24} AUC {dif_auc:+.4f}  F1 {dif_f1:+.4f}   -> {veredicto}")

Y el ranking visto de una sola mirada: el puntaje compuesto, el desglose de sus componentes, y un
mapa de calor con todas las métricas de todos los modelos.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(15, 5.2))

posiciones = np.arange(len(ranking))
ejes[0].barh(posiciones, ranking["puntaje"], color=ranking["color"],
             edgecolor="black", linewidth=0.8)
for i, puntaje in enumerate(ranking["puntaje"]):
    ejes[0].text(puntaje + 0.004, i, f"{puntaje:.4f}", va="center", fontsize=10,
                 fontweight="bold")
ejes[0].set_yticks(posiciones)
ejes[0].set_yticklabels([f"{p}. {m}" for p, m in zip(ranking["puesto"], ranking["modelo"])])
ejes[0].set_xlabel("puntaje compuesto")
ejes[0].set_title("Ranking final")
ejes[0].set_xlim(max(0, ranking["puntaje"].min() - 0.06),
                 min(1.02, ranking["puntaje"].max() + 0.06))
ejes[0].invert_yaxis()

componentes = ranking.set_index("modelo")[["roc_auc", "f1_macro", "mcc_normalizado",
                                           "estabilidad"]]
componentes.columns = ["ROC-AUC", "F1 macro", "MCC (0-1)", "Estabilidad"]
componentes.plot(kind="bar", ax=ejes[1], edgecolor="black", linewidth=0.5, width=0.8)
ejes[1].set_ylabel("valor")
ejes[1].set_title("Los cuatro componentes del puntaje")
ejes[1].set_ylim(0, 1.05)
ejes[1].legend(fontsize=9, ncol=2)
ejes[1].tick_params(axis="x", rotation=18)
ejes[1].set_xlabel("")

plt.tight_layout()
plt.show()

metricas_mapa = ["accuracy", "balanced_accuracy", "precision", "recall", "f1",
                 "f1_macro", "roc_auc", "average_precision", "mcc", "kappa"]
mapa = ranking.set_index("modelo")[metricas_mapa]

fig, eje = plt.subplots(figsize=(12, 3.6))
sns.heatmap(mapa, annot=True, fmt=".4f", cmap="RdYlGn", linewidths=0.6,
            cbar_kws={"label": "valor de la métrica"}, ax=eje)
eje.set_title("Todas las métricas, todos los modelos  (verde = mejor)")
eje.set_ylabel("")
plt.tight_layout()
plt.show()

<a id="sec-12-8"></a>
## 12.8 Veredicto y limitaciones

La celda siguiente redacta la conclusión leyendo el ranking, de manera que siga siendo válida si
agregás modelos y volvés a correr todo.

In [ ]:
lider = ranking.iloc[0]
segundo = ranking.iloc[1] if len(ranking) > 1 else None
mas_simple = (ranking.loc[ranking["modelo"].str.contains("Logística")].iloc[0]
              if ranking["modelo"].str.contains("Logística").any() else None)

print("=" * 88)
print("  VEREDICTO")
print("=" * 88)
print()
print(f"  1. GANADOR: {lider['modelo']}  (escalado: {lider['escalado']})")
print(f"     ROC-AUC {lider['roc_auc']:.4f} | F1 macro {lider['f1_macro']:.4f} | "
      f"accuracy {lider['accuracy']:.4f} | MCC {lider['mcc']:.4f}")
print(f"     Estimación con su margen: F1 macro = {lider['cv_f1_macro_media']:.4f} "
      f"± {lider['cv_f1_macro_desvio']:.4f}")
print()

if segundo is not None:
    dif = lider["f1_macro"] - segundo["f1_macro"]
    ruido = max(lider["cv_f1_macro_desvio"], segundo["cv_f1_macro_desvio"])
    print("  2. ¿LA VENTAJA ES REAL?")
    print(f"     Diferencia con {segundo['modelo']}: {dif:.4f} de F1 macro.")
    print(f"     Desvío entre pliegues del par: {ruido:.4f}.")
    if dif > ruido:
        print("     La diferencia SUPERA el ruido de la medición: la ventaja está demostrada.")
    else:
        print("     La diferencia NO supera el ruido de la medición. Con estos datos los")
        print("     dos modelos son indistinguibles, y conviene elegir por otro criterio:")
        print("     simplicidad, velocidad o interpretabilidad.")
    print()

if mas_simple is not None and mas_simple["modelo"] != lider["modelo"]:
    dif_simple = lider["f1_macro"] - mas_simple["f1_macro"]
    ruido_simple = max(lider["cv_f1_macro_desvio"], mas_simple["cv_f1_macro_desvio"])
    print("  3. ¿VALE LA PENA LA COMPLEJIDAD?")
    print(f"     El modelo más simple e interpretable es la regresión logística")
    print(f"     (puesto {int(mas_simple['puesto'])}, F1 macro {mas_simple['f1_macro']:.4f}).")
    print(f"     El ganador la supera por {dif_simple:.4f}, con un ruido de {ruido_simple:.4f}.")
    if dif_simple > ruido_simple:
        print("     La complejidad extra se justifica.")
    else:
        print("     La complejidad extra NO se justifica con estos datos: para producción")
        print("     conviene la logística, que además se puede explicar coeficiente por")
        print("     coeficiente.")
    print()
elif mas_simple is not None:
    print("  3. El modelo más simple es también el mejor. No hay complejidad que justificar.")
    print()

print("  4. TODOS SUPERAN LA LÍNEA DE BASE")
print(f"     Responder siempre la clase mayoritaria da {linea_base:.4f} de accuracy.")
peores = ranking.loc[ranking["accuracy"] <= linea_base, "modelo"].tolist()
if peores:
    print(f"     ATENCIÓN: estos modelos NO la superan: {peores}")
else:
    print(f"     Los {len(ranking)} modelos la superan. El mínimo es "
          f"{ranking['accuracy'].min():.4f}, el máximo {ranking['accuracy'].max():.4f}.")
print()

print("  5. EL ESCALADO")
sensibles = sensibilidad[sensibilidad > 1e-9]
indiferentes = sensibilidad[sensibilidad <= 1e-9]
if len(indiferentes):
    print(f"     Indiferentes al escalado: {list(indiferentes.index)}")
    print("     Es lo esperado: son modelos basados en árboles.")
if len(sensibles):
    print(f"     Sensibles al escalado: {list(sensibles.index)}")
    print(f"     La mayor diferencia observada fue {sensibles.max():.4f} de ROC-AUC.")
print()

print("=" * 88)
print("  LIMITACIONES")
print("=" * 88)
print()
print(f"  a. EL DATASET ES CHICO. Con {len(X)} filas, cada pliegue de validación tiene")
print(f"     unos {len(X)//N_PLIEGUES} casos. Las diferencias de menos de "
      f"{estabilidad['cv_f1_macro_desvio'].median():.3f} entre modelos")
print("     no se pueden demostrar con estos datos.")
print()
print("  b. LOS HIPERPARÁMETROS SE ELIGIERON SOBRE LOS MISMOS DATOS. El k de KNN y")
print("     la poda del árbol se seleccionaron con la misma validación cruzada con la")
print("     que después se los evaluó. Es la forma 4 de fuga del punto 4.3, y hace que")
print("     sus métricas sean algo optimistas comparadas con las de la logística y el")
print("     bosque, que no se ajustaron. La solución rigurosa es la validación cruzada")
print("     anidada: un bucle interno para elegir hiperparámetros y uno externo para")
print("     evaluar.")
print()
print("  c. LAS VARIABLES SON DEPENDIENTES ENTRE SÍ. Las siete columnas suman 1 en")
print("     cada fila, así que no son independientes. Eso no afecta la capacidad")
print("     predictiva, pero sí vuelve delicada la lectura de los coeficientes de la")
print("     regresión logística: hay que leerlos como pesos relativos, no absolutos.")
print()
print("  d. NO HAY CONJUNTO DE PRUEBA VIRGEN. Todas las decisiones de este cuaderno")
print("     se tomaron mirando las mismas 171 filas. Para una estimación realmente")
print("     independiente habría que apartar datos al principio y no tocarlos hasta")
print("     el final, o conseguir datos nuevos.")
print()
print("=" * 88)
detalle_escalado = ("sin escalado previo" if lider["escalado"] == "Sin escalar"
                    else f"con escalado {lider['escalado']}")
print(f"  RECOMENDACIÓN: {lider['modelo']}, {detalle_escalado}")
if mas_simple is not None and mas_simple["modelo"] != lider["modelo"]:
    dif_simple = lider["f1_macro"] - mas_simple["f1_macro"]
    ruido_simple = max(lider["cv_f1_macro_desvio"], mas_simple["cv_f1_macro_desvio"])
    if dif_simple <= ruido_simple:
        print(f"                  o bien {mas_simple['modelo']}, si se prioriza poder")
        print("                  explicar las decisiones del modelo.")
print("=" * 88)

---
<a id="sec-cierre"></a>
# Cierre — Resumen de lo hecho

**El recorrido de las dos partes**

| Etapa | Qué se hizo | Dónde |
|---|---|---|
| Carga | Formas de llevar el archivo a Colab, con GitHub por defecto | P1 y P2, 0.2 |
| EDA | Objetivo, distribuciones, densidad por clase, correlaciones, interactivo | P1, 2 |
| Limpieza | Cinco controles y exportación del dataset limpio | P1, 3 |
| Verificación | Revalidación de la entrada | P2, 0.3 |
| Marco | Supervisado vs. no supervisado; regresión vs. clasificación | P2, 1 |
| Transformación | Feature engineering evaluado y descartado, codificación, escaladores | P2, 2 |
| División | 80/20 estratificada y demostración de su fragilidad | P2, 3 |
| Fuga de información | Siete formas, dos demostraciones y el pipeline como antídoto | P2, 4 |
| Overfitting | Sesgo y varianza, cuatro formas de detectarlo, cómo se mide, qué hacer | P2, 5 |
| Validación cruzada | K-Fold estratificado y la comparación con una sola división | P2, 6 |
| Arquitectura | Catálogo de modelos y bloque de métricas único | P2, 7 |
| Modelos | Regresión Logística, KNN, Árbol de Decisión, Random Forest | P2, 8 a 11 |
| Comparativa | Tabla maestra, ROC, escalado, estabilidad, ajuste, ranking | P2, 12 |

**Las decisiones que se tomaron, y por qué**

1. **No se eliminó ningún dato.** Ni outliers ni casos raros: la auditoría mostró que todos los
   valores son válidos, y con 171 filas cada caso pesa demasiado como para descartarlo sin una
   razón fuerte.
2. **No se agregaron variables derivadas.** Se construyeron, se midieron y se descartaron: no
   traían información que no estuviera ya en las siete originales.
3. **Todo se evaluó con validación cruzada.** Una sola división 80/20 sobre este dataset da
   resultados que varían varios puntos según el azar de la partición.
4. **Se reporta media y desvío, nunca la media sola.** Sin el desvío no se puede saber si una
   diferencia entre modelos es real.
5. **Todo pasó por un `Pipeline`.** No por elegancia: porque es la única forma de garantizar que no
   haya fuga de información.
6. **Se comparó cada modelo con y sin escalado.** Confirma la teoría donde corresponde y cuantifica
   el efecto donde importa.

**Cómo seguir**

- **Agregar modelos**: punto 7.5. Tres líneas, y vuelven a correrse los puntos 8 a 12.
- **Cambiar el criterio del ranking**: los pesos están en la primera celda de 12.7.
- **Probar otros escaladores**: agregar una entrada al diccionario `ESCALADORES` en 7.1; todos los
  modelos lo toman automáticamente.
- **Ajustar hiperparámetros con rigor**: reemplazar la búsqueda del árbol por una validación
  cruzada anidada, que evita el optimismo señalado en la limitación (b).
- **Usar otro dataset**: correr la Parte 1 sobre el archivo nuevo y cambiar `RUTA_DATOS` acá.
  Mientras la primera columna sea la variable objetivo con dos clases y el resto sean numéricas,
  los dos cuadernos funcionan sin más cambios.

In [ ]:
print("=" * 88)
print("  PARTE 2 COMPLETADA")
print("=" * 88)
print(f"  Casos analizados        : {len(X)}")
print(f"  Variables predictoras   : {len(predictoras)}")
print(f"  Modelos evaluados       : {maestra['modelo'].nunique()}")
print(f"  Escalados comparados    : {maestra['escalado'].nunique()}")
print(f"  Combinaciones evaluadas : {len(maestra)}")
print(f"  Ajustes de modelo       : aproximadamente {len(maestra) * N_PLIEGUES * 2}")
print()
print(f"  Mejor modelo            : {ranking.iloc[0]['modelo']} "
      f"({ranking.iloc[0]['escalado']})")
print(f"  ROC-AUC                 : {ranking.iloc[0]['roc_auc']:.4f}")
print(f"  F1 macro                : {ranking.iloc[0]['cv_f1_macro_media']:.4f} "
      f"± {ranking.iloc[0]['cv_f1_macro_desvio']:.4f}")
print("=" * 88)